# *C5orf67* / *MAP3K1*: Borzoi and AlphaGenome in one notebook

Both sequence-model analyses of the four linked *C5orf67* variants, run in one kernel so their
results can be compared directly, plus a third section that measures the two things that most
plausibly explain their disagreement: **where the variants sit in the prediction window** and
**how much of *MAP3K1* is quantified**.

### Variants analysed

*C5orf67* carries two largely independent signals, and both are analysed here. The **core
3-prime block** is the fine-mapped signal: rs459193, rs256903, rs173964 and rs256904, in
near-complete linkage at the 3-prime end of the gene, together with the further variants the
fine mapping places in the same credible sets. The **rs3843467 block** is the *SHC2*
trans-eQTL variant that led to this region, with its proxies; it shares an r-squared of 0.03
with the core block and appears in none of its credible sets.

The variant list is not typed in. It is derived from the CARMA and SharePro outputs at run
time, and every coordinate, allele and orientation is read from a file those pipelines wrote
— no step queries a variant database. Those files live in this repository, under
`scripts/2_colocalization_and_finemapping/`, and are found automatically; point `DATA_DIR`
somewhere else and re-run to use a different set.

| Section | What it does |
|---|---|
| **A. Borzoi** | Supplementary Figures 8–10, corrected Figure 3b,c, the four effect measures, hypergeometric enrichment |
| **B. AlphaGenome** | Locus overview, variant scoring, ISM, background comparison, per-variant and haplotype sequence editing, the same four measures |
| **C. Cross-model** | Window-position sweep, cropping diagnostic, ceiling-corrected enrichment, and a manifest of the figures for the manuscript |

**The two models disagree, and Section C is about why.** Borzoi predicts *MAP3K1* down in
alternate-allele carriers and finds adipose tracks enriched among the most affected; AlphaGenome
predicts up, with adipose tracks not leading the raw ranking. Three candidate explanations are
testable here:

1. **Different haplotypes.** The alternate allele of rs173964 is recorded as `A>C` in one place
   and `A>G` in another. Both sections check every allele against Ensembl before predicting
   (A3 and B10a) and will say so if they disagree. *If they do, stop and resolve it — nothing
   below means anything until both models see the same sequence.*
2. **Window position.** Borzoi's variants sit ~100 bp from the edge of a 524 kb input.
   Section C1 sweeps that offset and shows how much of the effect, and of the track ranking,
   depends on it. It also establishes a hard geometric constraint: with Borzoi's 196,608 bp
   output crop, the variants and the *MAP3K1* promoter **cannot both** be inside the predicted
   span, so the edge placement is forced rather than chosen.
3. **Region quantified.** Borzoi covers the 5′ portion of *MAP3K1*; AlphaGenome the whole gene
   body. C2 (Borzoi) and Appendix A (AlphaGenome) quantify what that changes.

A fourth possibility — that adipose enrichment in AlphaGenome is masked by highly expressing
contexts, since absolute effect measures scale with baseline expression — is tested in C4 by
re-running the enrichment on baseline-residualised effects. That test can fail, and if it does
the masking explanation should be dropped rather than argued.

**Direction of effect is not settled by either model.** Both are trained on reference genomes
and their sign is known to be unreliable (Huang et al. 2023; Sasse et al. 2023) — their
disagreement here is a demonstration of that. The measured data (AMSC genotypes, GTEx,
village eQTLs) is what establishes direction; the models speak to cellular context and
regulatory logic.

**Running this.** Run the sections in order. Section A needs the Borzoi weights (downloaded
from HuggingFace on first use) and Section B an AlphaGenome API key — see *Running locally*
below, or, in Colab, a secret named `ALPHA_GENOME_API_KEY`. Section C reuses both sections'
results from memory, so run it in the same session; it also makes its own predictions for the
sweep. A GPU is worth having: on the CPU a single Borzoi forward pass takes minutes, and
Section C makes one per window offset.

---

## Running locally

The notebook was written in Colab and runs unchanged there, but nothing in it needs Colab any
more: the data comes from this repository, the API key from the environment, and the outputs
go to a folder beside the notebook.

**Kernel.** Select any Python 3.10+ interpreter as the kernel (in VS Code: *Select Kernel* at
the top right of the notebook). The install cell below installs what is missing into whichever
interpreter is running the kernel, and skips the install entirely when everything imports —
so an environment you have already prepared is left alone.

**API key.** Section B calls the AlphaGenome API, which needs a key from
<https://deepmind.google.com/science/alphagenome>. The setup cell in Section B takes it from
the first of these that has one, so pick whichever suits you:

| Where | How |
|---|---|
| `.env` at the repository root | A line reading `ALPHA_GENOME_API_KEY=your-key-here`. VS Code loads this file into the kernel's environment automatically (`python.envFile`, which defaults to `${workspaceFolder}/.env`), and the setup cell also reads it directly, so a key added after the kernel started still works without a restart. `.env` is git-ignored. |
| Environment variable | `export ALPHA_GENOME_API_KEY=...` before launching VS Code or `jupyter lab`. A variable exported in a terminal *after* the editor started is not visible to the kernel. |
| `~/.alphagenome_api_key` | A file in your home folder containing just the key. Useful if you work in several checkouts. |
| Typed in | If none of the above is found, the cell prompts for it with `getpass` and keeps it in memory for the session only. |

The key is never printed, only where it was found.

**Data.** The fine-mapping and colocalization outputs the variant definition is built from are
already in this repository, in three folders under `scripts/2_colocalization_and_finemapping/`:

| Folder | Files |
|---|---|
| `dat/` | `variant_order.tsv`, `rs3843467_signed.ld`, `*_signed.z` |
| `2b_finemapping/results/` | `carma_results_*_signed.csv`, `carma_seed_stability_*.csv` |
| `2a_colocalization/results/` | `res_signed.sharepro.txt`, `res_signed_2h.sharepro.txt` |

The next cell finds them by walking up from the notebook to the repository root, so nothing
has to be copied or configured. Set `DATA_DIR` to read them from somewhere else.

**Outputs.** Figures and tables go to `results/` beside this notebook, in
`scripts/3_sequence_to_function_modeling/AlphaGenome_Borzoi/`, unless `RESULTS_DIR` says
otherwise. In Colab they go next to the data on Drive instead, because the runtime disk is
wiped on disconnect.

In [ ]:
# @title Install both toolchains { display-mode: "form" }
# @markdown Colab imports numpy, pandas and pyarrow as compiled extensions at startup. If pip
# @markdown upgrades one of them now, the files on disk change while the interpreter keeps the
# @markdown old binary in memory, and every later import fails with errors like
# @markdown `cannot import name '_center' from 'numpy._core.umath'` — numpy failing to import
# @markdown part of itself. No amount of further installing fixes that; only a restart does.
# @markdown
# @markdown So this pins those three to the versions already loaded and installs everything in
# @markdown one command, letting the resolver see all constraints at once. Installing both
# @markdown toolchains together also matters: doing them in separate cells is what usually
# @markdown triggers the upgrade, because the second resolve does not see the first's result.
# @markdown
# @markdown Locally the same reasoning applies to whichever interpreter is running the kernel,
# @markdown which is printed below — but if everything already imports, nothing is installed
# @markdown and a prepared environment is left exactly as it is.
AUTO_RESTART = True  # @param {type:"boolean"}
# @markdown Install even when every package already imports. Leave off unless you want the
# @markdown versions refreshed.
FORCE_INSTALL = False  # @param {type:"boolean"}

import importlib.metadata as importlib_metadata
import importlib.util
import os
import sys

from IPython.display import clear_output

PACKAGES = ['alphagenome', 'borzoi-pytorch', 'enformer-pytorch', 'pyfaidx']
# Distribution name -> the module it provides, which is what "is it there" really asks.
MODULES = {'alphagenome': 'alphagenome', 'borzoi-pytorch': 'borzoi_pytorch',
           'enformer-pytorch': 'enformer_pytorch', 'pyfaidx': 'pyfaidx',
           'torch': 'torch'}
PIN_PACKAGES = ['numpy', 'pandas', 'pyarrow']

IN_COLAB = 'google.colab' in sys.modules

print(f'interpreter: {sys.executable}')
print(f'python     : {sys.version.split()[0]}\n')

missing = [name for name, module in MODULES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('not importable here: ' + ', '.join(missing))

if not (missing or FORCE_INSTALL):
    print('Everything imports already — nothing installed. Set FORCE_INSTALL to override.')
else:
    import subprocess

    def _session_started_at() -> float:
        """When this Python process started. 0.0 when it cannot be determined."""
        try:
            import psutil

            return psutil.Process(os.getpid()).create_time()
        except Exception:  # noqa: BLE001
            pass
        try:                                  # Linux without psutil, e.g. a bare Colab VM
            with open('/proc/stat') as handle:
                boot = next(int(line.split()[1])
                            for line in handle if line.startswith('btime'))
            with open(f'/proc/{os.getpid()}/stat') as handle:
                ticks = int(handle.read().rsplit(')', 1)[1].split()[19])
            return boot + ticks / os.sysconf('SC_CLK_TCK')
        except Exception:  # noqa: BLE001
            return 0.0

    pins = []
    for package in PIN_PACKAGES:
        try:
            pins.append(f'{package}=={importlib_metadata.version(package)}')
        except importlib_metadata.PackageNotFoundError:
            pass

    def _pip_install(arguments):
        return subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', *arguments],
            capture_output=True, text=True,
        )

    to_install = PACKAGES if FORCE_INSTALL else [p for p in PACKAGES if p in missing]
    if 'torch' in missing:
        # borzoi-pytorch pulls torch in, but say so: on a Mac that is a large download and
        # the wheel is CPU/MPS only.
        print('torch is not installed; it will come in as a dependency.')

    result = _pip_install([*pins, *to_install])
    pinned = result.returncode == 0
    if not pinned:
        print('Pinned install failed; retrying without the pins. A restart will very likely '
              'be needed afterwards.\n')
        print(result.stderr[-1500:])
        result = _pip_install(to_install)
        if result.returncode != 0:
            raise RuntimeError(result.stderr[-3000:])

    clear_output()

    # Which already-imported modules pip has just replaced on disk. This is not only about
    # the three pinned ones: installing alphagenome upgrades typing_extensions, typeguard
    # and jaxtyping, and if this session imported an older typing_extensions at startup the
    # import fails later with `cannot import name 'NoExtraItems' from 'typing_extensions'`
    # — the new typeguard reading the old module the interpreter still holds in memory.
    # Only a restart fixes that, so every top-level module is checked, not a fixed list.
    session_start = _session_started_at()
    stale = []
    for name, module in list(sys.modules.items()):
        if '.' in name or not session_start:
            continue
        module_file = getattr(module, '__file__', None)
        try:
            if module_file and os.path.getmtime(module_file) > session_start:
                stale.append(name)
        except OSError:
            continue

    print('Installed: ' + (', '.join(to_install) or 'nothing was missing'))
    print('Pinned    : ' + (', '.join(pins) if pinned else 'not applied (see above)'))
    if not session_start:
        print('\nCould not tell when this process started, so the staleness check below was '
              'skipped.\n  Restart the kernel now if anything was installed.')
    elif stale:
        print(f'\nReplaced on disk after this session started: {", ".join(sorted(stale))}.')
        print('This session is still holding the old copies, so restart the kernel before '
              'running\n  anything else — in VS Code, the Restart button above the '
              'notebook; in Colab,\n  Runtime -> Restart session. Then run from the top.')
        if AUTO_RESTART and IN_COLAB:
            print('\nAUTO_RESTART is on — restarting now. Colab will say the session '
                  'crashed; that is expected. Re-run from the top afterwards.')
            os.kill(os.getpid(), 9)
    else:
        print('\nNothing this session had already imported was replaced — carry on. If a '
              'later import\n  still fails with "cannot import name ...", restart the '
              'kernel anyway: that error means\n  a module on disk is newer than the copy '
              'in memory.')

In [ ]:
# @title Environment check — run this before anything else
# @markdown Confirms every dependency of both sections imports cleanly. The failure worth
# @markdown catching here is a numpy that cannot import part of itself, which means pip
# @markdown replaced it mid-session and only a runtime restart will help.
import importlib.metadata as importlib_metadata

problems = []
for package in ['numpy', 'pandas', 'scipy', 'matplotlib', 'torch', 'borzoi_pytorch',
                'alphagenome']:
    name = package.replace('_', '-')
    try:
        module = __import__(package)
        version = getattr(module, '__version__', None) or importlib_metadata.version(name)
        print(f'{name:16s} {version}')
    except ImportError as error:
        print(f'{name:16s} FAILED — {error}')
        problems.append((name, str(error)))

if problems:
    stale = any('cannot import name' in message for _, message in problems)
    print('\n' + '=' * 72)
    if stale:
        print('Stale-compiled-module failure: pip replaced numpy (or pandas or pyarrow)\n'
              'after the session started. Installing more cannot fix it.\n\n'
              'Runtime -> Restart session, then run from the top.')
    else:
        print('Something failed to import. Re-run the install cell; if it persists,\n'
              'restart the runtime and run from the top.')
    print('=' * 72)
    raise SystemExit('Stopping here — see above.')

print('\nAll imports clean.')

---

## Memory

A Colab runtime has around 12 GB of RAM and no swap, so exhausting it kills the kernel and
loses everything held in memory. The cell below installs a few helpers the expensive sections
use to report and release memory as they go; run it before anything else.

In [ ]:
# @title Memory budget and helpers { display-mode: "form" }
# A Colab runtime has roughly 12 GB of RAM and no swap: when it is exhausted the kernel is
# killed and everything in memory is lost. A laptop has swap, so the same situation becomes
# slow rather than fatal — but a Borzoi prediction is large enough that swapping one is worse
# than not making it, so the accounting below is worth watching either way. A single Borzoi prediction is a float32 array of
# 7,611 tracks x 6,144 bins, which is 187 MB, so a handful of them held at once is enough to
# end the session. The cells below therefore reduce each prediction to a per-track summary as
# soon as it is made and free the array, keeping only the few full arrays the figures draw.
#
# These helpers make that visible and repeatable.
import gc
import os

# @markdown Print RAM in use after each expensive step.
REPORT_MEMORY = True  # @param {type:"boolean"}

try:
    import psutil

    _PROCESS = psutil.Process(os.getpid())
except ImportError:                                   # not installed: degrade to no numbers
    psutil = None
    _PROCESS = None


def memory_used_gb() -> float:
    """Resident memory of this process, in GB. NaN when psutil is unavailable."""
    if _PROCESS is None:
        return float('nan')
    return _PROCESS.memory_info().rss / 1024 ** 3


def memory_total_gb() -> float:
    if psutil is None:
        return float('nan')
    return psutil.virtual_memory().total / 1024 ** 3


def report_memory(label: str = '') -> None:
    """One line of RAM accounting, printed only when REPORT_MEMORY is on."""
    if not REPORT_MEMORY:
        return
    used, total = memory_used_gb(), memory_total_gb()
    if used != used:                                  # NaN
        print(f'    [memory] {label}: psutil unavailable')
        return
    bar_width = 24
    filled = int(bar_width * min(used / total, 1.0)) if total == total else 0
    bar = '#' * filled + '.' * (bar_width - filled)
    print(f'    [memory] {used:5.2f} / {total:.1f} GB  [{bar}]  {label}')


def release(*names: str) -> None:
    """Drop the named globals and collect. Missing names are ignored."""
    for name in names:
        globals().pop(name, None)
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def array_gb(*arrays) -> float:
    return sum(getattr(a, 'nbytes', 0) for a in arrays) / 1024 ** 3


report_memory('at start')
if psutil is not None and memory_total_gb() < 20:
    print(f'\n{memory_total_gb():.1f} GB of RAM. Borzoi predictions are 187 MB each, so keep '
          'SCORE_SINGLE_VARIANTS\noff or leave the reduce-and-free path enabled in the '
          'prediction cell — both are the default.')


---

## Locating the fine-mapping data

Everything the variant definition is built from was written by the pipelines in
`scripts/2_colocalization_and_finemapping/`, and lives in three folders there: `dat/` (the
variant index and the signed LD matrix), `2b_finemapping/results/` (CARMA) and
`2a_colocalization/results/` (SharePro). The cell below walks up from the notebook to the
repository root, finds those folders, and binds `DATA_DIR` and `DATA_DIRS`, which every later
cell resolves its paths against. Nothing has to be copied into place.

Set `DATA_DIR` to read the files from somewhere else — a flat folder holding all of them
works as well as the repository layout, and in Colab a Drive path does. If the folder is on
Drive and Drive is not mounted yet, the cell mounts it first.

Figures and tables are written to `results/` beside this notebook. In Colab they go beside
the data on Drive instead, since the runtime's own disk is wiped on disconnect.

In [ ]:
# @title Locate the fine-mapping data { display-mode: "form" }
# The variant definition is built from files the CARMA and SharePro pipelines wrote. In this
# repository they sit in three folders under scripts/2_colocalization_and_finemapping, so the
# cell walks up from the notebook to the repository root and looks there. It binds DATA_DIR
# (the folder those three sit under) and DATA_DIRS (the folders themselves), which the
# variant cell resolves its paths against.
#
# In Colab the same cell mounts Drive and searches it, so the notebook runs there unchanged.
import os
import sys
from pathlib import Path

# @markdown Folder holding the fine-mapping outputs. Leave blank to find them in this
# @markdown repository. A flat folder holding all the files works too, as does a Drive path.
DATA_DIR = ""  # @param {type:"string"}
# @markdown Where figures and tables go. Blank writes to `results` beside this notebook, or,
# @markdown in Colab, beside the data on Drive.
RESULTS_DIR = ""  # @param {type:"string"}
# @markdown Mount Google Drive before searching. No effect outside Colab.
MOUNT_DRIVE = True  # @param {type:"boolean"}
# @markdown How deep below `MyDrive` to search when nothing is found locally.
SEARCH_DEPTH = 6  # @param {type:"slider", min:2, max:10, step:1}

IN_COLAB = 'google.colab' in sys.modules
MOUNT_POINT = Path('/content/drive')

# Where the analysis outputs sit relative to the folder DATA_DIR names. The empty string is
# the folder itself, which covers a flat copy of everything.
FINEMAP_SUBDIRS = ('', 'dat', '2b_finemapping/results', '2a_colocalization/results')

# The file the search identifies a candidate by, and what the notebook reads.
MARKER = 'variant_order.tsv'
EXPECTED = [
    'variant_order.tsv',
    'rs3843467_signed.ld',
    'carma_results_T2D_signed.csv',
    'carma_results_fasting_insulin_signed.csv',
    'carma_seed_stability_T2D.csv',
    'carma_seed_stability_fasting_insulin.csv',
]
OPTIONAL = ['res_signed.sharepro.txt', 'res_signed_2h.sharepro.txt',
            'T2D_signed.z', 'fasting_insulin_signed.z']

# --- where this notebook is ---------------------------------------------------------------
# VS Code binds __vsc_ipynb_file__ to the notebook's path; Jupyter does not, and there the
# working directory is wherever the kernel was started. Either way this ends up pointing at
# the notebook's folder, falling back below to its known place in the repository.
NOTEBOOK_NAME = 'C5orf67_MAP3K1_Borzoi_and_AlphaGenome.ipynb'
_self = Path(globals().get('__vsc_ipynb_file__') or Path.cwd()).expanduser().resolve()
NOTEBOOK_DIR = _self.parent if _self.is_file() else _self


def _repository_root(start: Path):
    """The nearest ancestor of `start` that looks like this repository checkout."""
    for folder in [start, *start.parents]:
        if (folder / 'scripts' / '2_colocalization_and_finemapping').is_dir():
            return folder
        if (folder / '.git').exists() and (folder / 'scripts').is_dir():
            return folder
    return None


REPO_ROOT = _repository_root(NOTEBOOK_DIR)

# A kernel started from the repository root, as `jupyter lab` does, would otherwise write the
# outputs there. If the notebook is not in the folder we think we are in but its own folder
# exists in this checkout, that folder is the better answer.
_canonical = (REPO_ROOT / 'scripts' / '3_sequence_to_function_modeling' / 'AlphaGenome_Borzoi'
              if REPO_ROOT else None)
if _canonical is not None and _canonical.is_dir() \
        and not (NOTEBOOK_DIR / NOTEBOOK_NAME).is_file():
    NOTEBOOK_DIR = _canonical


def _resolve(root: Path, filename: str):
    """The file `filename` under `root`, looked for in each of the layouts we accept."""
    for subdir in FINEMAP_SUBDIRS:
        candidate = root / subdir / filename if subdir else root / filename
        if candidate.is_file():
            return candidate
    # One wildcard level, so a folder of per-trait subfolders is still found.
    hits = sorted(root.glob(f'*/{filename}'))
    return hits[0] if hits else None


def _looks_right(root) -> bool:
    return root is not None and root.is_dir() and _resolve(root, MARKER) is not None


# --- pick a root --------------------------------------------------------------------------
_explicit = Path(DATA_DIR.strip()).expanduser() if DATA_DIR.strip() else None

if _explicit is not None and str(_explicit).startswith('/content/drive') and IN_COLAB \
        and MOUNT_DRIVE and not (MOUNT_POINT / 'MyDrive').is_dir():
    from google.colab import drive

    drive.mount(str(MOUNT_POINT))

if _explicit is not None:
    if not _looks_right(_explicit):
        raise FileNotFoundError(
            f'{_explicit} does not contain {MARKER}, in the folder itself or in any of '
            f'{[s for s in FINEMAP_SUBDIRS if s]}.\n'
            f'  Contents: '
            f'{sorted(p.name for p in _explicit.iterdir())[:12] if _explicit.is_dir() else "not a folder"}')
    DATA_DIR = _explicit
else:
    candidates = [c for c in [
        REPO_ROOT / 'scripts' / '2_colocalization_and_finemapping' if REPO_ROOT else None,
        NOTEBOOK_DIR / 'data',
        NOTEBOOK_DIR,
        Path('/content/data'), Path('/content'),
    ] if _looks_right(c)]

    if not candidates and IN_COLAB:
        # Nothing local: fall back to the Colab behaviour of searching Drive for the marker.
        if MOUNT_DRIVE and not (MOUNT_POINT / 'MyDrive').is_dir():
            from google.colab import drive

            drive.mount(str(MOUNT_POINT))
        print(f'searching Drive for {MARKER} (depth {SEARCH_DEPTH})...')
        for _root in (MOUNT_POINT / 'MyDrive', MOUNT_POINT / 'Shareddrives',
                      MOUNT_POINT / 'Shared drives'):
            if not _root.is_dir():
                continue
            _base = len(_root.parts)
            for _current, _subdirs, _files in os.walk(_root):
                _here = Path(_current)
                if len(_here.parts) - _base >= SEARCH_DEPTH:
                    _subdirs.clear()
                    continue
                _subdirs[:] = [d for d in _subdirs if not d.startswith('.') and d not in
                               {'__pycache__', 'node_modules', '.git', '.ipynb_checkpoints'}]
                if MARKER in _files:
                    candidates.append(_here)
        candidates.sort(key=lambda p: len(p.parts))

    if not candidates:
        raise FileNotFoundError(
            f'{MARKER} was not found.\n'
            f'  Notebook folder : {NOTEBOOK_DIR}\n'
            f'  Repository root : {REPO_ROOT or "not identified — is this a checkout?"}\n'
            f'  Expected at     : <repo>/scripts/2_colocalization_and_finemapping/dat/'
            f'{MARKER}\n\n'
            '  Either the fine-mapping outputs are not in this checkout, or the notebook was '
            'opened\n  outside it. Set DATA_DIR above to the folder holding them.')
    DATA_DIR = candidates[0]
    if len(candidates) > 1:
        print(f'{len(candidates)} folders contain {MARKER}. Using the first:')
        for _c in candidates[:5]:
            print(f'   {"->" if _c == DATA_DIR else "  "} {_c}')

# --- report what was found ----------------------------------------------------------------
FINEMAP_FILES = {name: _resolve(DATA_DIR, name) for name in EXPECTED + OPTIONAL}
FINEMAP_FILES = {k: v for k, v in FINEMAP_FILES.items() if v is not None}
# The folders the files actually live in, which the variant cell searches directly.
DATA_DIRS = list(dict.fromkeys(p.parent for p in FINEMAP_FILES.values()))

print(f'DATA_DIR = {DATA_DIR}')
for _folder in DATA_DIRS:
    try:
        _shown = _folder.relative_to(DATA_DIR)
    except ValueError:
        _shown = _folder
    print(f'  {str(_shown) or "."}/')
    for _name, _path in FINEMAP_FILES.items():
        if _path.parent == _folder:
            print(f'    {_name:44s} {_path.stat().st_size / 1024:9.1f} KB')

_missing = [f for f in EXPECTED if f not in FINEMAP_FILES]
if _missing:
    print(f'\nmissing: {_missing}')
    print('  The variant cell will say which of these it actually needs. '
          'carma_seed_stability_*\n  is optional; without it, selection falls back to the '
          'credible sets and effect groups.')
if not any(f.endswith('sharepro.txt') for f in FINEMAP_FILES):
    print('\nno SharePro output found. Without it the effect groups cannot be read, and '
          'selection\n  rests on the CARMA credible sets alone.')

# --- where the outputs go -----------------------------------------------------------------
if RESULTS_DIR.strip():
    RESULTS_ROOT = Path(RESULTS_DIR.strip()).expanduser()
elif IN_COLAB and str(DATA_DIR).startswith(str(MOUNT_POINT)):
    # Colab wipes the runtime disk on disconnect, so outputs go to Drive beside the data.
    RESULTS_ROOT = DATA_DIR / 'results'
else:
    RESULTS_ROOT = NOTEBOOK_DIR / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'\noutputs -> {RESULTS_ROOT}')

---

# Section A — Borzoi

Everything below reproduces the Borzoi panels of the manuscript: Supplementary Figures 8–10, the
corrected Figure 3b,c (antisense tracks dropped), the four effect measures, and the
hypergeometric enrichment of adipose tracks. Section A writes to `borzoi_figures/`.

Section C reuses these results, so run A to completion before B.

In [ ]:
# @title Imports, device and output folder { display-mode: "form" }
# @markdown Use Apple Silicon's GPU (Metal) for Borzoi when there is no CUDA device. It is
# @markdown far faster than the CPU, but a few operations are still unimplemented in the MPS
# @markdown backend and raise rather than falling back, so the cell tries one convolution
# @markdown first and stays on the CPU if it fails.
USE_MPS = True  # @param {type:"boolean"}

from pathlib import Path

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from scipy import stats

# RESULTS_ROOT comes from the data cell. Falling back to the working directory keeps this
# cell runnable on its own.
OUTPUT_DIR = Path(globals().get('RESULTS_ROOT', Path.cwd())) / 'borzoi_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif USE_MPS and getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    try:
        torch.nn.Conv1d(4, 4, 3).to('mps')(torch.zeros(1, 4, 16, device='mps'))
        DEVICE = 'mps'
    except Exception as error:  # noqa: BLE001
        print(f'MPS rejected a test convolution ({type(error).__name__}); using the CPU.')
        DEVICE = 'cpu'
else:
    DEVICE = 'cpu'

print(f'torch {torch.__version__} on {DEVICE}')
if DEVICE == 'cpu':
    print('No GPU: each prediction takes a few minutes. In Colab, Runtime -> Change runtime '
          'type -> GPU;\n  on a Mac with Apple Silicon, leave USE_MPS on.')
elif DEVICE == 'mps':
    print('Using the Metal backend. If a later cell fails with an unimplemented MPS '
          'operator,\n  set USE_MPS to False and re-run from here.')

# The published figures use this font; falls back silently if it is not installed.
mpl.rcParams['font.family'] = ['Latin Modern Roman', 'DejaVu Serif']

### Reproducibility

Two runs of this notebook can disagree, and it is worth being precise about why, because the
usual remedy does not apply. **Nothing in the analysis samples.** Borzoi and AlphaGenome are
deterministic given their weights and an input sequence, both are used in `eval` mode, and the
only calls to a random generator anywhere below are the jitter in the scatter plots and the
resampling intervals added here — all seeded. Setting a seed therefore does not change a single
prediction, and a run that differs from the last one does not differ because of one.

What does change an answer, in rough order of how much:

| Source | Effect | Handled by |
|---|---|---|
| A different variant set from an upstream re-run | Different sequence goes into the model | CARMA fixes its seed; the variant table is written to the manifest |
| A different Borzoi replicate (`FOLD`) | The four are trained independently and disagree | `FOLD` recorded in the manifest |
| A different device or precision (CUDA / MPS / CPU, TF32) | Small numeric differences, ~1e-6 relative | Deterministic kernels below; device recorded |
| `targets_human.txt` re-fetched from a moving branch | Track annotation could change under you | Pinned to a commit and checksummed |
| Live Ensembl or GENCODE annotation | Exon boundaries shift between releases | Release recorded in the manifest |

None of those are large. What makes them *look* large is the last step: a statistic that counts
how many adipose tracks land in the top twenty turns a 1% numeric difference into a different
integer, and the integer is what the manuscript quotes. That is fixed further down, in the
enrichment cell, by reporting statistics that do not cut the ranking anywhere.

The cell below fixes the seeds anyway — they cost nothing and they do make the resampling
intervals reproducible — asks torch for deterministic kernels, and starts a manifest that every
later cell adds to. Re-running the notebook and diffing two manifests answers "what changed?"
in one command.

In [ ]:
# @title Reproducibility: seeds, deterministic kernels and a run manifest { display-mode: "form" }
# Seeding is insurance rather than a fix: no prediction below depends on a random number.
# It does fix the two things that do sample — plot jitter and the bootstrap intervals in the
# enrichment cells — so those are stable from run to run.
#
# The manifest is the part that actually helps. Every cell that reads something mutable, or
# settles a choice that changes the numbers, records it here, and the file is written beside
# the figures. Two runs that disagree can then be diffed instead of argued about.
SEED = 0  # @param {type:"integer"}
# @markdown Ask torch for deterministic kernels. Slightly slower, and on CUDA it also turns off
# @markdown TF32, whose reduced mantissa is one of the few ways two GPUs disagree on the same
# @markdown input.
DETERMINISTIC_KERNELS = True  # @param {type:"boolean"}

import hashlib
import os
import platform
import random
import sys
from datetime import datetime, timezone

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# The generator the resampling statistics draw from. Passing it explicitly beats relying on
# global state, which any later cell could reset without saying so.
RNG = np.random.default_rng(SEED)

if DETERMINISTIC_KERNELS:
    # cudnn picks convolution algorithms by benchmarking unless told not to, and the fastest
    # algorithm is not always the same one twice. TF32 is a 10-bit mantissa on Ampere and
    # later: fast, and enough to move a track across a rank boundary.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if hasattr(torch.backends.cuda, 'matmul'):
        torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends.cudnn, 'allow_tf32'):
        torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as error:  # noqa: BLE001  (older torch, or an op with no determin. impl)
        print(f'Deterministic algorithms unavailable ({type(error).__name__}); continuing.')

# --- the manifest -------------------------------------------------------------------------
PROVENANCE = {}


def note(**entries) -> None:
    """Record facts about this run. Later calls overwrite earlier ones for the same key."""
    PROVENANCE.update(entries)


def file_digest(path, length: int = 12) -> str:
    """Short sha256 of a file, or a marker if it is not there."""
    path = Path(path)
    if not path.is_file():
        return 'absent'
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    return f'sha256:{digest[:length]}'


def write_provenance(filename: str = 'run_provenance.csv'):
    """Write the manifest beside the figures, one key per row, and return the path."""
    destination = Path(globals().get('RESULTS_ROOT', Path.cwd())) / filename
    frame = pd.DataFrame(sorted(PROVENANCE.items()), columns=['key', 'value'])
    frame.to_csv(destination, index=False)
    print(f'run manifest ({len(frame)} entries) -> {destination}')
    return destination


note(
    run_started=datetime.now(timezone.utc).isoformat(timespec='seconds'),
    seed=SEED,
    deterministic_kernels=DETERMINISTIC_KERNELS,
    device=DEVICE,
    platform=f'{platform.system()} {platform.machine()}',
    python=sys.version.split()[0],
    torch=torch.__version__,
    numpy=np.__version__,
    pandas=pd.__version__,
)
if DEVICE == 'cuda':
    note(gpu=torch.cuda.get_device_name(0))

print(f'seed {SEED}, deterministic kernels '
      f'{"on" if DETERMINISTIC_KERNELS else "off"}, device {DEVICE}')
print('Seeds do not change any prediction here — nothing in the analysis samples. They fix '
      'the\n  resampling intervals and the plot jitter. The manifest is what makes two runs '
      'comparable.')

### Figure fonts

Colab does not ship Times New Roman, which is licensed and not redistributable with Linux. Two
ways to get it. The cell below installs the Microsoft core fonts on a Debian runtime that can
reach the package mirror, and otherwise falls back to Liberation Serif, which is
metric-compatible with
Times New Roman, meaning the glyph widths are identical and a figure laid out in one will lay out
identically in the other. For a manuscript figure the fallback is normally indistinguishable.

macOS ships Times New Roman, so locally on a Mac the cell finds it and does nothing else.

Run this once, before any plotting cell. Every figure in the notebook inherits it.


In [ ]:
# @title Figure fonts: Times New Roman, with a metric-compatible fallback { display-mode: "form" }
# Set INSTALL_MSTTCOREFONTS to False to skip the download and go straight to Liberation Serif.
INSTALL_MSTTCOREFONTS = True

import shutil
import subprocess
import sys
from matplotlib import font_manager

# apt-get exists only on the Debian-based runtimes Colab uses; macOS already has the font.
_needs_font = 'Times New Roman' not in {f.name for f in font_manager.fontManager.ttflist}
if INSTALL_MSTTCOREFONTS and _needs_font and sys.platform.startswith('linux'):
    try:
        # The package prompts for a licence acceptance, so preseed the answer.
        subprocess.run(
            'echo "ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula '
            'select true" | debconf-set-selections && '
            'apt-get -qq install -y ttf-mscorefonts-installer > /dev/null 2>&1',
            shell=True, check=False, timeout=300,
        )
        # matplotlib caches the font list, so it has to be rebuilt after an install.
        cache = Path(mpl.get_cachedir())
        for stale in cache.glob('fontlist*.json'):
            stale.unlink()
        font_manager._load_fontmanager(try_read_cache=False)
    except Exception as error:
        print(f'Could not install msttcorefonts ({error}); using the fallback.')

available = {f.name for f in font_manager.fontManager.ttflist}
SERIF = next((name for name in ('Times New Roman', 'Liberation Serif', 'Nimbus Roman',
                                'DejaVu Serif') if name in available), 'serif')

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': [SERIF, 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
    # stix is the serif maths face that pairs with Times, so exponents and
    # superscripts in axis labels match the surrounding text.
    'mathtext.fontset': 'stix',
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 13,
    'pdf.fonttype': 42,   # embed as TrueType so the text stays editable in Illustrator
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

print(f'Serif family in use: {SERIF}')
if SERIF != 'Times New Roman':
    print('Liberation Serif is metric-compatible with Times New Roman, so glyph widths '
          'and therefore the layout are identical.')


In [ ]:
# @title Connectivity check — which hosts can this runtime reach? { display-mode: "form" }
# @markdown `Temporary failure in name resolution` means DNS is not resolving: the runtime has
# @markdown no outbound network, or the host is blocked. This cell tests each host the notebook
# @markdown needs and says what is still possible without it.
import socket

HOSTS = {
    'huggingface.co': 'Borzoi model weights. Without this the model cannot be loaded at all.',
    'api.genome.ucsc.edu': 'Reference sequence (first choice).',
    'rest.ensembl.org': 'Reference sequence (fallback), exon annotation, allele checks.',
    'raw.githubusercontent.com': 'Borzoi track annotation (targets_human.txt).',
}

reachable = {}
for host, purpose in HOSTS.items():
    try:
        socket.getaddrinfo(host, 443, proto=socket.IPPROTO_TCP)
        with socket.create_connection((host, 443), timeout=15):
            pass
        reachable[host] = True
        status = 'reachable'
    except Exception as error:  # noqa: BLE001
        reachable[host] = False
        status = f'UNREACHABLE ({type(error).__name__})'
    print(f'{host:28s} {status}\n{"":28s} {purpose}')

offline = not any(reachable.values())
print('\n' + '=' * 78)
if offline:
    print('No host is reachable — this runtime has no outbound network.\n\n'
          'Borzoi weights are downloaded from HuggingFace at load time, so the notebook\n'
          'cannot run here no matter which sequence source is chosen. Options:\n'
          '  - Colab: Runtime -> Disconnect and delete runtime, then reconnect. A fresh VM\n'
          '    usually resolves DNS again; this failure is often transient.\n'
          '  - If you are on a managed/enterprise runtime with egress restrictions, the\n'
          '    notebook needs to run somewhere with access to the four hosts above.\n'
          '  - If you already have the weights and a local hg38 FASTA, set SEQUENCE_SOURCE\n'
          '    to "local_fasta" below and point the model load at your local checkpoint.')
elif not reachable['api.genome.ucsc.edu']:
    print('UCSC is unreachable but other hosts are not — the sequence cell will fall back to\n'
          'Ensembl automatically. Nothing to do.')
else:
    print('All hosts reachable.')
print('=' * 78)

---

## Variant definition

Everything the three sections predict on is defined here, and derived from the fine mapping
rather than maintained by hand. Re-running CARMA or SharePro and re-running this cell is the
whole of the update procedure.

### Where each value comes from

No step queries a variant database. Each is read from a file the fine-mapping pipelines
wrote:

| Needed | Taken from | File |
|---|---|---|
| Which variants | credible sets, seed stability, effect groups | `carma_results_*.csv`, `carma_seed_stability_*.csv`, `res*sharepro.txt` |
| GRCh37 position, both alleles | the LD matrix's own row index | `variant_order.tsv` |
| Which allele raises T2D risk | sign of the harmonised beta, propagated by the sign of *r* | `carma_results_T2D_signed.csv`, `rs3843467_signed.ld` |
| GRCh38 position | GRCh37 plus one offset, calibrated on the hand-checked variants | — |

The last row is the only inference. Within a locus this small the two builds differ by a
single constant, so the cell calculates that constant from the five variants whose GRCh38
coordinates were checked by hand, requires all five to agree, and stops if they do not.

### Two blocks, defined by their anchors

A variant joins a block by linkage disequilibrium with that block's anchor — rs256903 for the
core 3-prime signal, rs3843467 for the independent one — so membership is a property of the
data rather than a list to maintain. A selected variant correlated with neither anchor above
`MIN_R_FOR_BLOCK` belongs to some further signal in the region; it is named in the output and
left out, since the analyses here concern these two.

### Orientation

`ALT` is the T2D risk-raising allele throughout. Two routes to it, both from pipeline files:

- **LD sign** (primary). Every sign in the matrix and every harmonised beta refers to PLINK's
  A1 allele, so `sign(r)` between a variant and its anchor says whether their A1 alleles sit
  on the same haplotype. This does not weaken when a variant's own association does.
- **Beta sign** (cross-check). `beta > 0` means A1 raises the trait — decisive at large |Z|,
  uninformative at small |Z|.

Where both are available and disagree, the cell reports it and uses the LD sign.

### What is checked against what

Three literals in the cell exist to be tested against rather than read from:
`VERIFIED_VARIANTS`, five hand-checked GRCh38 coordinates and alleles; `PHASED_RISK_ALLELE`,
seven risk alleles read from the LDlink `Correlated_Alleles` column of Supplementary Table 6;
and the GRCh38 reference base itself, checked in the Borzoi sequence cell against the base
actually found at each coordinate. An error in the build offset, in the A1/A2 to REF/ALT
mapping, or in the sign convention fails at least one of the three.

### Scope

Every analysis is computed once, over all the credible-set variants of the two blocks, and
then reported at two scopes. **Main** is the four fine-mapped variants plus rs3843467 — what
the main-text figures show. **Supplementary** is all of them. Because the variant scorers work
one variant at a time, the main-text numbers are a strict subset of the same run rather than a
second one: nothing is scored twice and the two cannot disagree.

Outputs are written to a subfolder per scope and carry the scope in their filename, so the two
versions of a figure never overwrite one another.


In [ ]:
# @title Variant definition — built from the CARMA and SharePro outputs
# The variant list is derived at run time from the fine mapping rather than typed in, so
# re-running CARMA or SharePro and re-running this cell is the whole of the update procedure.
#
# Nothing here queries a variant database. Every value comes from a file the fine-mapping
# pipelines wrote, in the folder named by FINEMAP_DIR:
#
#   which variants        carma_results_*.csv, carma_seed_stability_*.csv, res*sharepro.txt
#   GRCh37 position       variant_order.tsv (or a *_signed.z file)
#   both alleles          variant_order.tsv
#   which allele is risk  sign of the harmonised beta, propagated by the sign of r
#                         (rs3843467_signed.ld)
#   GRCh38 position       GRCh37 plus one offset, calibrated on the hand-checked variants
#
# Each of those steps is checked against something established independently: five
# hand-checked hg38 coordinates, a seven-entry phase table read from LDlink, and — in the
# sequence cell further down — the hg38 reference base actually found at each position.
import glob as _glob
from pathlib import Path as _Path

import numpy as _np
import pandas as _pd

# --------------------------------------------------------------------------------------
# 0. Inputs
# --------------------------------------------------------------------------------------
# @markdown Overrides `DATA_DIR` from the cell above. Leave blank to use it.
FINEMAP_DIR = ""  # @param {type:"string"}
# @markdown Which colocalization run defines the effect groups. 2-hour glucose contributes
# @markdown 63,396 samples against 151,013-200,622 for the other traits, and including it
# @markdown lowers the shared probability of the fine-mapped effect group from 1.00 to 0.18.
SHAREPRO_RUN = "3-trait"  # @param ["3-trait", "4-trait"]
# @markdown Every analysis is computed once, over every credible-set variant, and then
# @markdown reported at two scopes: `main` is the four fine-mapped variants plus rs3843467,
# @markdown the set the main-text figures show; `supplementary` is all of them. Scoring is
# @markdown per variant, so the main-text numbers are a subset of the same run rather than a
# @markdown second one — no analysis is repeated to produce both.
MAIN_SCOPE_NAME = "main_5variants"  # @param {type:"string"}
SUPP_SCOPE_NAME = "supp_25variants"  # @param {type:"string"}

# DATA_DIRS is the list of folders the cell above found the files in; DATA_DIR is what
# they sit under. Both are searched, with the repository's layout appended to each root so
# that a folder named directly still resolves the CARMA and SharePro outputs.
_ROOTS = [p for p in [
    _Path(FINEMAP_DIR) if FINEMAP_DIR.strip() else None,
    globals().get('DATA_DIR'),
    *globals().get('DATA_DIRS', []),
    _Path.cwd() / 'data', _Path.cwd(), _Path('/content'), _Path('/content/data'),
] if p is not None]
_ROOTS += [r / s for r in list(_ROOTS) for s in
           ('dat', 'results', '2a_colocalization/results', '2b_finemapping/results')]


def _find(*patterns, required=True, what=''):
    for root in _ROOTS:
        if not root.is_dir():
            continue
        for pattern in patterns:
            hits = sorted(_glob.glob(str(root / pattern))) or \
                   sorted(_glob.glob(str(root / '*' / pattern)))
            if hits:
                return _Path(hits[0])
    if required:
        raise FileNotFoundError(
            f'could not find {what or patterns[0]}. Looked for {list(patterns)} under:\n    '
            + '\n    '.join(str(r) for r in _ROOTS if r.is_dir())
            + '\n\n  Run the data cell above, which locates the fine-mapping outputs and '
              'binds DATA_DIR,\n  or set FINEMAP_DIR here.')
    return None


# --------------------------------------------------------------------------------------
# 1. The two signals, and the facts that are checked rather than read
# --------------------------------------------------------------------------------------
# C5orf67 carries two largely independent signals, and each analysis block is defined by its
# anchor. Variants join a block by linkage disequilibrium with that anchor, so membership is
# a property of the data rather than a list to maintain.
BLOCK_ANCHOR = {
    'core_3prime': 'rs256903',    # the fine-mapped signal at the 3-prime end of C5orf67
    'rs3843467_block': 'rs3843467',  # the SHC2 trans-eQTL variant and its proxies
}
# @markdown Minimum |r| with a block anchor for a variant to join that block. Variants
# @markdown correlated with neither anchor above this are reported and left out.
MIN_R_FOR_BLOCK = 0.5  # @param {type:"slider", min:0.2, max:0.95, step:0.05}

# The four fine-mapped variants plus rs3843467: the set the manuscript figures report.
MANUSCRIPT_RSIDS = ['rs459193', 'rs256903', 'rs173964', 'rs256904', 'rs3843467']
HAPLOTYPE_RSIDS_CANONICAL = ['rs459193', 'rs256903', 'rs173964', 'rs256904']

# hg38, 1-based, ALT is the T2D risk-raising allele. Checked by hand against dbSNP.
VERIFIED_VARIANTS = {
    'rs459193':  ('chr5', 56510924, 'A', 'G'),
    'rs256903':  ('chr5', 56513311, 'C', 'A'),
    'rs173964':  ('chr5', 56513638, 'A', 'G'),
    'rs256904':  ('chr5', 56514478, 'A', 'T'),
    'rs3843467': ('chr5', 56560548, 'G', 'T'),
}

# Which allele travels with the rs256903 A allele, from the LDlink Correlated_Alleles column
# of Supplementary Table 6. The orientation rule has to reproduce every entry.
PHASED_RISK_ALLELE = {
    'rs256903': 'A', 'rs459193': 'G', 'rs173964': 'G', 'rs256904': 'T',
    'rs465002': 'T', 'rs464605': 'T', 'rs458741': 'C', 'rs3843467': 'T',
}

# @markdown A variant enters through seed stability if it sat in a credible set in at least
# @markdown this many of the 20 CARMA seeds, in either fine-mapped trait.
MIN_SEEDS_IN_CS = 5  # @param {type:"slider", min:1, max:20, step:1}
# @markdown |Z| at which a variant's own beta sign is trusted to orient it. Below this the
# @markdown orientation is propagated from the block anchor through the sign of r.
Z_TRUST = 3.0  # @param {type:"slider", min:1.0, max:6.0, step:0.5}

# --------------------------------------------------------------------------------------
# 2. Load the fine mapping
# --------------------------------------------------------------------------------------
carma = {
    'T2D': _pd.read_csv(_find('carma_results_T2D_signed.csv',
                              what='the CARMA T2D results')).set_index('SNP'),
    'fasting_insulin': _pd.read_csv(_find('carma_results_fasting_insulin_signed.csv',
                                          what='the CARMA fasting insulin results')
                                    ).set_index('SNP'),
}
print(f'CARMA: {len(carma["T2D"])} variants, traits {list(carma)}')

stability = {}
for _trait in ('T2D', 'fasting_insulin'):
    _path = _find(f'carma_seed_stability_{_trait}.csv', required=False)
    if _path is not None:
        stability[_trait] = _pd.read_csv(_path).set_index('SNP')
print(f'seed stability: {list(stability) or "not found, selecting on credible sets only"}')

_SHAREPRO_FILES = {
    '3-trait': ('res_signed.sharepro.txt', 'res_signed_sharepro.txt'),
    '4-trait': ('res_signed_2h.sharepro.txt', 'res_signed_2h_sharepro.txt'),
}
sharepro_all = {}
for _label, _patterns in _SHAREPRO_FILES.items():
    _path = _find(*_patterns, required=False)
    if _path is None:
        continue
    _table = _pd.read_csv(_path, sep='\t')
    sharepro_all[_label] = [{'members': r.cs.split('/'), 'share': float(r.share)}
                            for r in _table.itertuples()]
if SHAREPRO_RUN not in sharepro_all:
    raise FileNotFoundError(f'the {SHAREPRO_RUN} SharePro output was not found. '
                            f'Available: {list(sharepro_all) or "none"}')
sharepro = sharepro_all[SHAREPRO_RUN]
print(f'SharePro: using the {SHAREPRO_RUN} run, {len(sharepro)} effect groups '
      f'(read {list(sharepro_all)})')

# --------------------------------------------------------------------------------------
# 3. Select
# --------------------------------------------------------------------------------------
evidence = {}


def _note(rsid, reason):
    evidence.setdefault(rsid, set()).add(reason)


for _i, _group in enumerate(sharepro):
    for _rsid in _group['members']:
        _note(_rsid, f'SharePro EG{_i} (shared p = {_group["share"]:.4f})')
for _trait, _table in carma.items():
    for _rsid in _table.index[_table['CS'] > 0]:
        _note(_rsid, f'CARMA {_trait} CS{int(_table.loc[_rsid, "CS"])}')
for _trait, _table in stability.items():
    for _rsid in _table.index[_table['n_in_CS'] >= MIN_SEEDS_IN_CS]:
        _note(_rsid, f'CARMA {_trait} in a credible set in '
                     f'{int(_table.loc[_rsid, "n_in_CS"])}/'
                     f'{int(_table.loc[_rsid, "n_seeds"])} seeds')
# The five the manuscript reports are carried whatever the current fine mapping says, so the
# published panels stay reproducible. rs3843467 is in none of the credible sets by design —
# it is the independent signal — and rs173964 enters an effect group only when 2-hour
# glucose is included.
for _rsid in MANUSCRIPT_RSIDS:
    _note(_rsid, 'reported in the manuscript')

SELECTED = sorted(evidence)
print(f'\n{len(SELECTED)} variants selected by the fine mapping')

# --------------------------------------------------------------------------------------
# 4. Positions and alleles, from the LD matrix's own variant index
# --------------------------------------------------------------------------------------
# variant_order.tsv is the row index of the signed LD matrix. `reference_allele` is PLINK's
# A1 — the allele every sign in the matrix and every harmonised beta refers to — and
# `other_allele` is A2. PLINK read the VCF with --keep-allele-order, so A2 is the reference
# base and A1 the alternate. That mapping is asserted against the hand-checked variants
# below, and again against the fetched sequence in the Borzoi cell.
_order_path = _find('variant_order.tsv', required=False)
if _order_path is not None:
    order = _pd.read_csv(_order_path, sep='\t').rename(
        columns={'reference_allele': 'A1', 'other_allele': 'A2'})
else:
    _z_path = _find('*_signed.z', what='variant_order.tsv or a *_signed.z file')
    order = _pd.read_csv(_z_path, sep='\t').rename(
        columns={'rsid': 'SNP', 'allele1': 'A1', 'allele2': 'A2'})
    order['row'] = range(len(order))
    _order_path = _z_path
for _needed in ('SNP', 'position', 'A1', 'A2'):
    if _needed not in order.columns:
        raise ValueError(f'{_order_path} has no {_needed!r} column. '
                         f'Columns present: {list(order.columns)}')
order['A1'] = order['A1'].astype(str).str.upper()
order['A2'] = order['A2'].astype(str).str.upper()
ORDER = order.set_index('SNP')
ROW_OF = {rsid: i for i, rsid in enumerate(ORDER.index)}
print(f'variant index: {_order_path.name}, {len(ORDER)} rows')

_absent = [r for r in SELECTED if r not in ORDER.index]
if _absent:
    raise ValueError(f'{len(_absent)} selected variant(s) are missing from the variant '
                     f'index: {_absent}\n  The fine-mapping results and the variant index '
                     'are from different runs.')

# GRCh37 to GRCh38. Within a locus this small the two builds differ by one constant, so the
# shift is calibrated from the hand-checked variants rather than a chain file, and unanimity
# is required — a single disagreement means the assumption does not hold here.
_offsets = {r: v[1] - int(ORDER.loc[r, 'position'])
            for r, v in VERIFIED_VARIANTS.items() if r in ORDER.index}
if len(set(_offsets.values())) != 1:
    raise ValueError('the GRCh37 to GRCh38 shift is not constant across the hand-checked '
                     'variants:\n' + '\n'.join(f'  {k}: {v:+,}' for k, v in _offsets.items())
                     + '\n\nAn indel in the chain falls inside this locus; lift the '
                       'positions over with pyliftover instead.')
BUILD_OFFSET = next(iter(_offsets.values()))
print(f'GRCh37 to GRCh38 offset {BUILD_OFFSET:+,} bp, unanimous across '
      f'{len(_offsets)} hand-checked variants')

_ld_path = _find('*_signed.ld', what='the signed LD matrix')
LD = _np.loadtxt(_ld_path)
if LD.shape != (len(ORDER), len(ORDER)):
    raise ValueError(f'{_ld_path.name} is {LD.shape} but the variant index has '
                     f'{len(ORDER)} rows; they are from different runs.')
print(f'signed LD: {_ld_path.name}, {LD.shape[0]}x{LD.shape[1]}')


def _r(a, b):
    return float(LD[ROW_OF[a], ROW_OF[b]]) if a in ROW_OF and b in ROW_OF else _np.nan


def _best_z(rsid):
    zs = {t: (float(carma[t].loc[rsid, 'Z']) if rsid in carma[t].index else _np.nan)
          for t in carma}
    trait = max(zs, key=lambda t: abs(zs[t]) if _np.isfinite(zs[t]) else -1)
    return trait, zs[trait]


# --------------------------------------------------------------------------------------
# 5. Assign each variant to a block
# --------------------------------------------------------------------------------------
# A variant is analysed if the fine mapping selected it and it is in strong linkage
# disequilibrium with one of the two anchors. Variants correlated with neither belong to
# some further signal in the region; they are named and left out, because the analyses below
# are about these two.
assignment, UNASSIGNED = {}, []
for _rsid in SELECTED:
    _correlations = {b: abs(_r(_rsid, a)) for b, a in BLOCK_ANCHOR.items()}
    _best = max(_correlations, key=lambda b: _correlations[b])
    if _correlations[_best] >= MIN_R_FOR_BLOCK:
        assignment[_rsid] = _best
    else:
        UNASSIGNED.append((_rsid, _correlations))

VARIANT_GROUPS = {b: sorted((r for r, g in assignment.items() if g == b),
                            key=lambda r: int(ORDER.loc[r, 'position']))
                  for b in BLOCK_ANCHOR}

if UNASSIGNED:
    print(f'\n{len(UNASSIGNED)} selected variant(s) belong to neither block and are not '
          'analysed:')
    for _rsid, _corr in UNASSIGNED:
        _detail = ', '.join(f'|r| {v:.2f} with {BLOCK_ANCHOR[b]}' for b, v in _corr.items())
        _trait, _z = _best_z(_rsid)
        print(f'  {_rsid:12s} {_detail}   (best |Z| {abs(_z):.2f} in {_trait})')
    print(f'  They form a separate, weakly supported signal. Raise MIN_R_FOR_BLOCK to '
          'exclude more, lower it to include them.')

# --------------------------------------------------------------------------------------
# 6. Orientation: which allele raises T2D risk
# --------------------------------------------------------------------------------------
# Two routes, both from pipeline files. The harmonised betas and every sign in the matrix
# refer to A1, so sign(beta) > 0 means A1 raises the trait — decisive at large |Z| and
# meaningless at small |Z|. sign(r) with the block anchor says whether two variants carry
# their A1 alleles on the same haplotype, and does not weaken when the association does.
# The LD route is primary and the beta route cross-checks it; disagreements are reported.
RISK_IS_REFERENCE, CANONICAL_VARIANTS, ORIENTATION = set(), {}, {}
_notes = []

_anchor_risk_is_A1 = {}
for _block, _anchor in BLOCK_ANCHOR.items():
    _known = PHASED_RISK_ALLELE[_anchor]
    _anchor_risk_is_A1[_block] = (ORDER.loc[_anchor, 'A1'] == _known)
    _trait, _z = _best_z(_anchor)
    _notes.append(f'  [{_block}] anchored on {_anchor}, risk allele {_known} from the phase '
                  f'table (Z = {_z:+.2f} in {_trait})')

for _block, _rsids in VARIANT_GROUPS.items():
    _anchor = BLOCK_ANCHOR[_block]
    for _rsid in _rsids:
        _a1, _a2 = ORDER.loc[_rsid, 'A1'], ORDER.loc[_rsid, 'A2']
        _r_anchor = 1.0 if _rsid == _anchor else _r(_rsid, _anchor)
        _trait, _z = _best_z(_rsid)

        _propagated = (_r_anchor > 0) == _anchor_risk_is_A1[_block]
        _own = (_z > 0) if abs(_z) >= Z_TRUST else None
        if _own is not None and _own != _propagated:
            _notes.append(f'  {_rsid}: LD sign and beta sign disagree (r with {_anchor} '
                          f'{_r_anchor:+.3f}, own Z {_z:+.2f} in {_trait}). Using the LD '
                          'sign.')

        _risk = _a1 if _propagated else _a2
        _reference, _alternate = _a2, _a1        # A2 is the reference base
        if _risk == _reference:
            # The risk allele is already in the genome, so writing it in would be a no-op.
            # The other allele is modelled and the effect direction reads inverted.
            RISK_IS_REFERENCE.add(_rsid)
        CANONICAL_VARIANTS[_rsid] = ('chr5', int(ORDER.loc[_rsid, 'position']) + BUILD_OFFSET,
                                     _reference, _alternate)
        ORIENTATION[_rsid] = {'risk_allele': _risk, 'r_with_anchor': _r_anchor,
                              'z': _z, 'trait': _trait}

# --------------------------------------------------------------------------------------
# 7. Validate against what was established independently
# --------------------------------------------------------------------------------------
_bad = [f'  {k}: derived {CANONICAL_VARIANTS.get(k)} | hand-checked {v}'
        for k, v in VERIFIED_VARIANTS.items()
        if k in CANONICAL_VARIANTS and tuple(CANONICAL_VARIANTS[k]) != tuple(v)]
if _bad:
    raise ValueError(
        'the derived table disagrees with the hand-checked variants:\n' + '\n'.join(_bad)
        + '\n\nOne of three things is wrong for every variant, not just these: the build '
          'offset, the A1/A2 to REF/ALT mapping, or the beta sign convention.')
print(f'\ncoordinates and alleles reproduce all '
      f'{sum(1 for k in VERIFIED_VARIANTS if k in CANONICAL_VARIANTS)} hand-checked variants')

_phase_bad = [f'  {r}: phase table says {k}, derived {ORIENTATION[r]["risk_allele"]}'
              for r, k in PHASED_RISK_ALLELE.items()
              if r in ORIENTATION and ORIENTATION[r]['risk_allele'] != k]
if _phase_bad:
    raise ValueError('the orientation contradicts the LDlink phase table:\n'
                     + '\n'.join(_phase_bad))
print(f'orientation reproduces all '
      f'{sum(1 for r in PHASED_RISK_ALLELE if r in ORIENTATION)} phase table entries')

# --------------------------------------------------------------------------------------
# 8. Scope, and the objects the rest of the notebook uses
# --------------------------------------------------------------------------------------
# The two reporting scopes. Both are subsets of the same CANONICAL_VARIANTS, so a figure
# drawn at either scope uses identical underlying scores.
MAIN_RSIDS = [r for r in CANONICAL_VARIANTS if r in MANUSCRIPT_RSIDS]
SUPP_RSIDS = list(CANONICAL_VARIANTS)
SCOPES = {MAIN_SCOPE_NAME: MAIN_RSIDS, SUPP_SCOPE_NAME: SUPP_RSIDS}

GROUP_RSIDS = {b: [r for r in rsids if r in CANONICAL_VARIANTS]
               for b, rsids in VARIANT_GROUPS.items()}
GROUP_RSIDS = {b: rsids for b, rsids in GROUP_RSIDS.items() if rsids}
GROUP_OF = {r: b for b, rsids in GROUP_RSIDS.items() for r in rsids}
GROUP_ANCHOR = {b: BLOCK_ANCHOR[b] for b in GROUP_RSIDS}

VARIANT_EVIDENCE = _pd.DataFrame([
    {'rsid': r, 'block': GROUP_OF[r], 'hg38_pos': CANONICAL_VARIANTS[r][1],
     'REF': CANONICAL_VARIANTS[r][2], 'ALT': CANONICAL_VARIANTS[r][3],
     'risk_allele': ORIENTATION[r]['risk_allele'], 'risk_is_REF': r in RISK_IS_REFERENCE,
     'r_with_anchor': round(ORIENTATION[r]['r_with_anchor'], 3),
     'Z_T2D': round(float(carma['T2D'].loc[r, 'Z']), 2) if r in carma['T2D'].index else _np.nan,
     'Z_FI': round(float(carma['fasting_insulin'].loc[r, 'Z']), 2)
     if r in carma['fasting_insulin'].index else _np.nan,
     'PIP_T2D': round(float(carma['T2D'].loc[r, 'PIP']), 3) if r in carma['T2D'].index else _np.nan,
     'PIP_FI': round(float(carma['fasting_insulin'].loc[r, 'PIP']), 3)
     if r in carma['fasting_insulin'].index else _np.nan,
     'in_main_scope': r in MANUSCRIPT_RSIDS,
     'evidence': '; '.join(sorted(evidence[r]))}
    for r in CANONICAL_VARIANTS
]).sort_values(['block', 'hg38_pos']).reset_index(drop=True)


def assert_canonical(table, source_name):
    """Raises if `table` disagrees with CANONICAL_VARIANTS on any shared rsID."""
    problems = []
    for rsid, entry in table.items():
        expected = CANONICAL_VARIANTS.get(rsid)
        if expected is None:
            problems.append(f'  {rsid}: not in CANONICAL_VARIANTS')
        elif tuple(entry) != tuple(expected):
            problems.append(
                f'  {rsid}: {source_name} has {entry[2]}>{entry[3]} at {entry[0]}:{entry[1]:,}'
                f' | canonical {expected[2]}>{expected[3]} at {expected[0]}:{expected[1]:,}')
    if problems:
        raise ValueError(f'{source_name} disagrees with CANONICAL_VARIANTS:\n'
                         + '\n'.join(problems))
    print(f'  {source_name}: {len(table)} variant(s) match CANONICAL_VARIANTS.')


for _n in _notes:
    print(_n)

print(f'\n{len(CANONICAL_VARIANTS)} variants (hg38, ALT is the T2D risk-raising allele). '
      f'Reported at two scopes:')
for _name, _rsids in SCOPES.items():
    print(f'  {_name:16s} {len(_rsids):2d} variants')
print()
for _block, _rsids in GROUP_RSIDS.items():
    print(f'  [{_block}]  anchor {GROUP_ANCHOR[_block]}')
    for _rsid in _rsids:
        _c, _p, _ref, _alt = CANONICAL_VARIANTS[_rsid]
        _star = ' *' if _rsid in MAIN_RSIDS else '  '
        _flag = '   risk allele is the reference, direction inverted' \
            if _rsid in RISK_IS_REFERENCE else ''
        print(f'   {_star}{_rsid:12s} {_c}:{_p:,} {_ref}>{_alt}  '
              f'r={ORIENTATION[_rsid]["r_with_anchor"]:+.2f}{_flag}')
print(f'   * in the {MAIN_SCOPE_NAME} scope, shown in the main-text figures')


### Fine-mapping evidence behind each variant

The table below is what the rest of the notebook is built on: which block each variant joins
and at what correlation with the anchor, which allele raises T2D risk, and how strong the
association actually is in each fine-mapped trait.

Two features of the fine mapping are worth carrying into the figure legends.

**Sharing is a property of the effect group, not of individual variants.** No variant reaches
a posterior inclusion probability above 0.5 in both T2D and fasting insulin — CARMA cannot
discriminate within a block in near-complete linkage, and spreads the probability across its
members. The cross-trait claim rests on SharePro's effect-group sharing instead, which is why
the two should not be quoted interchangeably.

**The colocalization is sensitive to which traits enter it.** On fasting insulin, T2D and
fasting glucose the fine-mapped effect group carries a shared probability of 1.00. Adding
2-hour glucose, which contributes 63,396 samples against 151,013 to 200,622 for the others,
lowers it to 0.18 and exchanges one member of the group for another in near-complete linkage
with it. The three-trait run is the default; `SHAREPRO_RUN` switches.

In [ ]:
# @title Fine-mapping evidence behind each variant { display-mode: "form" }
from pathlib import Path

import pandas as pd

with pd.option_context('display.width', 200, 'display.max_columns', 40,
                       'display.max_rows', 60):
    display(VARIANT_EVIDENCE)
_evidence_path = Path(globals().get('RESULTS_ROOT', Path.cwd())) / 'variant_evidence.csv'
_evidence_path.parent.mkdir(parents=True, exist_ok=True)
VARIANT_EVIDENCE.to_csv(_evidence_path, index=False)
print(f'written to {_evidence_path}')

# Posterior inclusion probability is spread across variants in near-complete linkage, so a
# high joint PIP is not the form the cross-trait evidence takes here. Stated explicitly
# because a reader may expect it to be.
_both = VARIANT_EVIDENCE[(VARIANT_EVIDENCE.PIP_T2D > 0.5) & (VARIANT_EVIDENCE.PIP_FI > 0.5)]
print(f'\nvariants with PIP > 0.5 in both fine-mapped traits: {len(_both)}'
      + ('' if len(_both) else '  (cross-trait support comes from SharePro effect-group '
                               'sharing, not from CARMA agreeing variant by variant)'))

# How much the colocalization depends on the fourth trait.
if len(sharepro_all) == 2:
    print('\nSharePro effect groups by run:')
    for _label, _groups in sharepro_all.items():
        for _i, _g in enumerate(_groups):
            print(f'  {_label:8s} EG{_i}  shared p = {_g["share"]:.4f}  '
                  f'{"/".join(_g["members"])}')


In [ ]:
# @title Configuration: variants, gene, model fold { display-mode: "form" }
# @markdown Borzoi replicate to load. The published figures used fold 0; the four replicates
# @markdown are independently trained and differ somewhat.
FOLD = 0  # @param [0, 1, 2, 3] {type:"raw"}
# @markdown Drop minus-strand RNA tracks from Figures 3b,c.
DROP_ANTISENSE = True  # @param {type:"boolean"}
# @markdown Tracks in the ranked box panel and coloured in the scatter.
N_TOP_BOX = 20  # @param {type:"slider", min:5, max:40, step:1}
N_TOP_SCATTER = 15  # @param {type:"slider", min:0, max:30, step:1}
# @markdown Also score every variant on its own, not just the allele sets. Adds one forward
# @markdown pass per variant on the upstream window.
SCORE_SINGLE_VARIANTS = True  # @param {type:"boolean"}

BORZOI_CONTEXT_LENGTH = 524_288   # model input
OUTPUT_CONTEXT_LENGTH = 196_608   # predicted (cropped) span
BIN_SIZE = 32                     # output resolution
CROP = (BORZOI_CONTEXT_LENGTH - OUTPUT_CONTEXT_LENGTH) // 2

# Every variant Section A knows about, both credible sets. Alleles come from
# CANONICAL_VARIANTS so Sections A and B cannot drift apart.
VARIANTS = dict(CANONICAL_VARIANTS)
assert_canonical(VARIANTS, 'Section A (Borzoi)')

GENE = 'MAP3K1'
GENE_COORDINATES = ('chr5', 56815549, 56896152)   # hg38, plus strand

# The window is anchored on the leftmost variant of the core block, which is where the
# published framing put it. That anchor depends on the variant set, so it is pinned: if a
# later fine mapping adds a variant further upstream the window would move and every MAP3K1
# number would move with it, silently. The check below makes that loud instead.
PUBLISHED_LEFTMOST = 56_498_805   # rs30351
ANCHOR_RSIDS = list(GROUP_RSIDS['core_3prime'])
LEFTMOST = min(CANONICAL_VARIANTS[r][1] for r in ANCHOR_RSIDS)
if LEFTMOST != PUBLISHED_LEFTMOST:
    print(f'NOTE: the window anchor is {LEFTMOST:,}, not the {PUBLISHED_LEFTMOST:,} the '
          'published\n      framing used. Every Borzoi number below is on a different input '
          'window.')

WINDOWS = {
    'centred':  LEFTMOST - BORZOI_CONTEXT_LENGTH // 2,
    'upstream': LEFTMOST - 100,
}
print('Input windows (start coordinate, hg38):')
for name, start in WINDOWS.items():
    output_start = start + CROP
    print(f'  {name:9s} input {start:,}-{start + BORZOI_CONTEXT_LENGTH:,}  '
          f'output {output_start:,}-{output_start + OUTPUT_CONTEXT_LENGTH:,}')

# Every variant has to sit inside the input window of every framing it is predicted on, or
# the allele would not be written at all. Checked rather than assumed.
def _inside(rsid, window_name):
    start = WINDOWS[window_name]
    return start < VARIANTS[rsid][1] <= start + BORZOI_CONTEXT_LENGTH


OUTSIDE_WINDOW = {name: [r for r in VARIANTS if not _inside(r, name)] for name in WINDOWS}
_any_outside = sorted({r for rsids in OUTSIDE_WINDOW.values() for r in rsids})
if _any_outside:
    print(f'\n{len(_any_outside)} variant(s) fall outside a Borzoi input window and are '
          'excluded from the\nallele sets predicted on that window. The 1 Mb AlphaGenome '
          'windows still cover them:')
    for _rsid in _any_outside:
        _where = ', '.join(n for n in WINDOWS if _rsid in OUTSIDE_WINDOW[n])
        print(f'  {_rsid:12s} {VARIANTS[_rsid][1]:,}  outside: {_where}')
else:
    print(f'\nall {len(VARIANTS)} variants lie inside both input windows')

# --- allele sets Borzoi will predict on -------------------------------------------------
# One set per block, plus the four fine-mapped variants as their own set, which is the one
# the published panels report.
BORZOI_ALLELE_SETS = {'haplotype_4': HAPLOTYPE_RSIDS_CANONICAL}
BORZOI_ALLELE_SETS.update({name: rsids for name, rsids in GROUP_RSIDS.items() if rsids})
if SCORE_SINGLE_VARIANTS:
    for _rsid in VARIANTS:
        BORZOI_ALLELE_SETS[f'single_{_rsid}'] = [_rsid]

# Both windows for the published haplotype, the upstream framing only for the rest, since
# that is the framing every MAP3K1 number in the manuscript uses.
SET_WINDOWS = {name: (list(WINDOWS) if name == 'haplotype_4' else ['upstream'])
               for name in BORZOI_ALLELE_SETS}

# Drop from each set any variant that does not fit inside all of that set's windows, so no
# figure legend can claim a set contains variants it does not.
for _name, _rsids in list(BORZOI_ALLELE_SETS.items()):
    _kept = [r for r in _rsids if all(_inside(r, w) for w in SET_WINDOWS[_name])]
    if _kept:
        if len(_kept) != len(_rsids):
            print(f'  {_name}: predicting on {len(_kept)} of {len(_rsids)}, outside: '
                  f'{sorted(set(_rsids) - set(_kept))}')
        BORZOI_ALLELE_SETS[_name] = _kept
    else:
        del BORZOI_ALLELE_SETS[_name], SET_WINDOWS[_name]

def bin_of(position: int, window_start: int) -> int:
    """Output bin containing a 1-based genomic position, or -1 if outside the window."""
    output_start = window_start + CROP
    index = (position - 1 - output_start) // BIN_SIZE
    return int(index) if 0 <= index < OUTPUT_CONTEXT_LENGTH // BIN_SIZE else -1


_n_passes = 1 + len(WINDOWS) + sum(len(SET_WINDOWS[n]) for n in BORZOI_ALLELE_SETS)
print(f'{len(BORZOI_ALLELE_SETS)} allele sets, about {_n_passes} forward passes')


In [ ]:
# @title Track annotation from the Borzoi repository { display-mode: "form" }
# Which track is which comes from a file in someone else's repository, and `main` is a moving
# target: if that file changes, every track index in this notebook silently means something
# else. It is therefore pinned to the commit that last touched it, and the copy actually used
# is checksummed into the run manifest.
#
# f47fe8a is the commit that last modified examples/targets_human.txt (2023-08-17). Set
# TARGETS_REF to "main" to follow upstream again, and expect the manifest checksum to change.
TARGETS_REF = "f47fe8a7dee9e515485a74006f492571b4c92610"  # @param {type:"string"}
# @markdown Local copy, used if GitHub is unreachable. Download `targets_human.txt` from the
# @markdown Borzoi repository, put it beside the notebook, and give the filename here.
LOCAL_TARGETS_PATH = ""  # @param {type:"string"}

TARGETS_URL = (f'https://raw.githubusercontent.com/calico/borzoi/{TARGETS_REF}/examples/'
               'targets_human.txt')
targets_cache = Path('targets_human.txt')

if LOCAL_TARGETS_PATH.strip():
    targets_path = Path(LOCAL_TARGETS_PATH.strip())
    targets = pd.read_csv(targets_path, sep='\t', index_col=0)
elif targets_cache.exists():
    targets_path = targets_cache
    targets = pd.read_csv(targets_cache, sep='\t', index_col=0)
    print(f'Using cached {targets_cache}')
else:
    try:
        targets = pd.read_csv(TARGETS_URL, sep='\t', index_col=0)
        targets.to_csv(targets_cache, sep='\t')
        targets_path = targets_cache
    except Exception as error:  # noqa: BLE001
        raise RuntimeError(
            f'Could not download the track annotation ({type(error).__name__}). '
            'Download targets_human.txt from the Borzoi repository, put it beside this '
            'notebook, and set LOCAL_TARGETS_PATH.'
        ) from error

# The identifier suffix encodes strand: '+' and '-' for the stranded assays, neither for
# the unstranded ones (the GTEx RNA tracks among them).
targets['strand'] = np.where(
    targets['identifier'].str.endswith('+'), '+',
    np.where(targets['identifier'].str.endswith('-'), '-', '.')
)
targets['assay'] = targets['description'].str.split(':').str[0]

rna_targets = targets[targets['description'].str.startswith('RNA:')]
RNA_FIRST_INDEX = int(rna_targets.index.min())
ADIPOSE_PATTERN = 'fat|adipose|adipocyte'
adipose_rna = rna_targets[
    rna_targets['description'].str.contains(ADIPOSE_PATTERN, case=False)
]

# The checksum, not the URL, is what proves two runs read the same annotation. The pinned
# copy is cb0c1d4baf514f44 — a different value in the manifest means the tracks moved. The
# track counts go in too: they are what every index below is relative to.
note(targets_ref=TARGETS_REF, targets_file=file_digest(targets_path),
     targets_n_tracks=len(targets), targets_n_rna=len(rna_targets),
     targets_n_adipose_rna=len(adipose_rna), adipose_pattern=ADIPOSE_PATTERN)

print(f'{len(targets):,} tracks total, {file_digest(targets_path)}; '
      f'{len(rna_targets):,} RNA tracks '
      f'(indices {RNA_FIRST_INDEX}-{int(rna_targets.index.max())}).')
print(f'{len(adipose_rna)} RNA tracks match "{ADIPOSE_PATTERN}" — these are the 33 tracks '
      'the manuscript\'s hypergeometric test counts as adipose.')
print(rna_targets['strand'].value_counts().to_string())

In [ ]:
# @title Fetch the reference sequence and write in the four alternate alleles { display-mode: "form" }
# @markdown Where the reference sequence comes from. "auto" tries UCSC, then Ensembl, then a
# @markdown local FASTA if one is given — the first that works wins, so a single host being
# @markdown down or blocked is not fatal.
SEQUENCE_SOURCE = "auto"  # @param ["auto", "ucsc", "ensembl", "local_fasta"]
# @markdown Path to an hg38 FASTA (indexed .fa/.fa.gz, or a Drive path). Only needed for the
# @markdown local source; requires pysam or pyfaidx.
LOCAL_FASTA_PATH = ""  # @param {type:"string"}
# @markdown Cache fetched sequences to disk. A re-run then costs nothing and works offline;
# @markdown point this at a Drive folder to survive the runtime being recycled.
CACHE_DIR = "sequence_cache"  # @param {type:"string"}
# @markdown Retries per chunk before falling through to the next source.
N_RETRIES = 3  # @param {type:"slider", min:1, max:6, step:1}

import gzip
import time

# A relative CACHE_DIR resolves against RESULTS_ROOT, so the cache lands on Drive and
# survives the runtime being recycled. An absolute path is taken as given.
cache_directory = Path(CACHE_DIR)
if not cache_directory.is_absolute():
    cache_directory = Path(globals().get('RESULTS_ROOT', Path.cwd())) / cache_directory
cache_directory.mkdir(parents=True, exist_ok=True)


def _with_retries(call, description: str, attempts: int = None):
    """Runs `call`, retrying transient network failures with a widening pause."""
    attempts = attempts or N_RETRIES
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return call()
        except Exception as error:  # noqa: BLE001
            last_error = error
            if attempt < attempts:
                pause = 2 ** attempt
                print(f'    {description}: attempt {attempt} failed '
                      f'({type(error).__name__}); retrying in {pause}s')
                time.sleep(pause)
    raise last_error


def _fetch_ucsc(chromosome: str, start: int, end: int, chunk: int = 100_000) -> str:
    """UCSC REST API. Coordinates are 0-based, half-open — the same convention as here."""
    parts = []
    for chunk_start in range(start, end, chunk):
        chunk_end = min(chunk_start + chunk, end)

        def get_chunk(a=chunk_start, b=chunk_end):
            response = requests.get(
                'https://api.genome.ucsc.edu/getData/sequence',
                params={'genome': 'hg38', 'chrom': chromosome, 'start': a, 'end': b},
                timeout=180,
            )
            response.raise_for_status()
            payload = response.json()
            if 'dna' not in payload:
                raise RuntimeError(f'UCSC returned no sequence: {payload}')
            return payload['dna']

        parts.append(_with_retries(get_chunk, 'UCSC'))
    return ''.join(parts)


def _fetch_ensembl(chromosome: str, start: int, end: int, chunk: int = 1_000_000) -> str:
    """Ensembl REST, pinned to GRCh38. Its coordinates are 1-based and inclusive."""
    parts = []
    name = chromosome.removeprefix('chr')
    for chunk_start in range(start, end, chunk):
        chunk_end = min(chunk_start + chunk, end)

        def get_chunk(a=chunk_start, b=chunk_end):
            response = requests.get(
                f'https://rest.ensembl.org/sequence/region/human/{name}:{a + 1}..{b}',
                params={'coord_system_version': 'GRCh38'},
                headers={'Content-Type': 'text/plain'},
                timeout=180,
            )
            response.raise_for_status()
            return response.text.strip()

        parts.append(_with_retries(get_chunk, 'Ensembl'))
    return ''.join(parts)


def _fetch_local(chromosome: str, start: int, end: int) -> str:
    """Local indexed FASTA, via pysam or pyfaidx. Both take 0-based, half-open coordinates."""
    if not LOCAL_FASTA_PATH.strip():
        raise RuntimeError('LOCAL_FASTA_PATH is empty.')
    path = LOCAL_FASTA_PATH.strip()
    try:
        import pysam

        with pysam.FastaFile(path) as handle:
            names = set(handle.references)
            key = chromosome if chromosome in names else chromosome.removeprefix('chr')
            return handle.fetch(key, start, end)
    except ImportError:
        from pyfaidx import Fasta

        genome = Fasta(path)
        key = chromosome if chromosome in genome else chromosome.removeprefix('chr')
        return str(genome[key][start:end])


SOURCES = {'ucsc': _fetch_ucsc, 'ensembl': _fetch_ensembl, 'local_fasta': _fetch_local}
SOURCE_ORDER = (['ucsc', 'ensembl', 'local_fasta'] if SEQUENCE_SOURCE == 'auto'
                else [SEQUENCE_SOURCE])


def fetch_reference(chromosome: str, start: int, end: int) -> str:
    """Reference sequence, from the first source that works, cached to disk.

    The cache is keyed on the exact interval, so a re-run — or a runtime with no network at
    all — reuses what was fetched before instead of failing.
    """
    cache_file = cache_directory / f'{chromosome}_{start}_{end}_hg38.txt.gz'
    if cache_file.exists():
        with gzip.open(cache_file, 'rt') as handle:
            sequence = handle.read().strip()
        print(f'  cache hit: {cache_file.name}')
        return sequence

    # Any cached interval that contains this one answers it by slicing. Sections A and B ask
    # for different windows over the same locus, so the wider of the two serves both.
    for candidate in sorted(cache_directory.glob(f'{chromosome}_*_hg38.txt.gz')):
        try:
            _, cached_start, cached_end, _ = candidate.stem.replace('.txt', '').split('_')
            cached_start, cached_end = int(cached_start), int(cached_end)
        except ValueError:
            continue
        if cached_start <= start and end <= cached_end:
            with gzip.open(candidate, 'rt') as handle:
                wider = handle.read().strip()
            if len(wider) == cached_end - cached_start:
                print(f'  cache hit: sliced out of {candidate.name}')
                return wider[start - cached_start:end - cached_start]

    errors = []
    for source in SOURCE_ORDER:
        try:
            print(f'  fetching {chromosome}:{start:,}-{end:,} from {source} ...')
            sequence = SOURCES[source](chromosome, start, end).upper()
            if len(sequence) != end - start:
                raise RuntimeError(
                    f'{source} returned {len(sequence):,} bp, expected {end - start:,}'
                )
            with gzip.open(cache_file, 'wt') as handle:
                handle.write(sequence)
            print(f'  got it from {source}; cached to {cache_file}')
            return sequence
        except Exception as error:  # noqa: BLE001
            errors.append(f'{source}: {type(error).__name__}: {error}')
            print(f'  {source} failed — {type(error).__name__}')

    raise RuntimeError(
        'No sequence source worked:\n  ' + '\n  '.join(errors)
        + '\n\nIf this is a name-resolution failure, the runtime has no outbound network — '
          'see the connectivity cell above. If only UCSC is blocked, set SEQUENCE_SOURCE to '
          '"ensembl"; if nothing is reachable, provide LOCAL_FASTA_PATH and set the source '
          'to "local_fasta".'
    )


BASE_INDEX = {'A': 0, 'C': 1, 'G': 2, 'T': 3}


def one_hot(sequence: str) -> torch.Tensor:
    """(1, 4, L) one-hot tensor; anything that is not ACGT becomes an all-zero column."""
    indices = np.array([BASE_INDEX.get(base, -1) for base in sequence])
    encoded = np.zeros((4, len(sequence)), dtype=np.float32)
    valid = indices >= 0
    encoded[indices[valid], np.flatnonzero(valid)] = 1.0
    return torch.from_numpy(encoded).unsqueeze(0)


def build_sequences(window_start: int, rsids=None) -> tuple:
    """Reference and perturbed sequences for one input window and one allele set.

    A 1-based genomic position P sits at offset P - 1 - window_start in a sequence fetched
    with a 0-based start, which is the one place an off-by-one would silently corrupt every
    downstream number — so each reference allele is checked against the fetched base and a
    mismatch raises rather than warns.
    """
    window_end = window_start + BORZOI_CONTEXT_LENGTH
    reference = fetch_reference('chr5', window_start, window_end)
    assert len(reference) == BORZOI_CONTEXT_LENGTH, len(reference)

    bases = list(reference)
    chosen = list(VARIANTS) if rsids is None else list(rsids)
    for rsid in chosen:
        _, position, ref, alt = VARIANTS[rsid]
        offset = position - 1 - window_start
        observed = reference[offset]
        if observed != ref:
            raise ValueError(
                f'{rsid}: expected reference {ref} at {position:,}, found {observed}. '
                'Coordinates or assembly are wrong.'
            )
        bases[offset] = alt
    perturbed = ''.join(bases)

    n_differences = sum(a != b for a, b in zip(reference, perturbed))
    assert n_differences == len(chosen), (n_differences, len(chosen))
    return reference, perturbed


# One reference per window, then one perturbed sequence per allele set per window.
reference_sequences = {name: build_sequences(start, [])[0]
                       for name, start in WINDOWS.items()}

sequences = {}                      # kept for the published haplotype_4 panels
allele_set_sequences = {}           # {set_name: {window: perturbed_sequence}}
for set_name, rsids in BORZOI_ALLELE_SETS.items():
    allele_set_sequences[set_name] = {}
    for window_name in SET_WINDOWS[set_name]:
        _, perturbed = build_sequences(WINDOWS[window_name], rsids)
        allele_set_sequences[set_name][window_name] = perturbed
    if set_name == 'haplotype_4':
        for window_name in WINDOWS:
            sequences[window_name] = (reference_sequences[window_name],
                                      allele_set_sequences[set_name][window_name])
print(f'built {sum(len(v) for v in allele_set_sequences.values())} perturbed sequences '
      f'across {len(BORZOI_ALLELE_SETS)} allele sets, all allele-checked')

In [ ]:
# @title Check the alleles against Ensembl { display-mode: "form" }
# @markdown A fourth, redundant check, and the only one needing network. Both alleles come
# @markdown from the fine mapping's own variant index, the reference base is verified against
# @markdown the fetched sequence in the cell above, and the orientation is verified against
# @markdown the phase table. Turn this on to also confirm each ALT is an allele Ensembl
# @markdown records for that rsID.
CHECK_ALLELES_WITH_ENSEMBL = False  # @param {type:"boolean"}

for rsid, (chromosome, position, ref, alt) in (VARIANTS.items()
                                               if CHECK_ALLELES_WITH_ENSEMBL else []):
    try:
        response = requests.get(
            f'https://rest.ensembl.org/variation/human/{rsid}',
            headers={'Content-Type': 'application/json'}, timeout=60,
        )
        record = response.json() if response.status_code == 200 else None
    except Exception as exc:  # noqa: BLE001
        print(f'  {rsid:10s} lookup failed ({exc!r})')
        continue
    if record is None:
        print(f'  {rsid:10s} not found in Ensembl')
        continue
    mappings = [m for m in record.get('mappings', [])
                if str(m.get('assembly_name', '')).startswith('GRCh38')]
    alleles = sorted({a for m in mappings
                      for a in str(m.get('allele_string', '')).split('/') if a})
    positions = sorted({int(m.get('start', -1)) for m in mappings})
    flag = 'OK' if (alt in alleles and position in positions) else 'CHECK'
    print(f'  {rsid:10s} using {ref}>{alt} at {position:,} | '
          f'Ensembl {"/".join(alleles)} at {positions}  {flag}')

### Memory during the Borzoi passes

Every forward pass produces a float32 array of 7,611 tracks by 6,144 bins — 187 MB. Holding
one per allele set would need several gigabytes, which is more than a Colab runtime has, and
exhausting it kills the kernel and loses everything.

Only the arrays the figures actually draw are kept: the two reference predictions and the
four-variant haplotype. Every other allele set is reduced to a per-track summary the moment
it is predicted and the array is freed, so peak memory stays flat however many sets are
scored. Those summaries are the same quantities the figures use — mean baseline, mean and
absolute mean difference, integrated difference and log2 fold change per track — and they are
written to disk as they are produced. An interrupted run resumes from them rather than
repeating the passes.

In [ ]:
# @title Run Borzoi, reducing each prediction as it is made { display-mode: "form" }
# @markdown Each forward pass produces a float32 array of (n_tracks x 6,144), which is 187 MB.
# @markdown Holding one per allele set would need several GB and is what exhausts a Colab
# @markdown runtime, so only the arrays the figures actually draw are kept: the two reference
# @markdown predictions and the published four-variant haplotype. Every other allele set is
# @markdown reduced to a per-track summary the moment it is predicted and the array is freed.
# @markdown
# @markdown Summaries are written to disk as they are produced, so a run that is interrupted
# @markdown can be resumed rather than repeated.
RESUME_FROM_DISK = True  # @param {type:"boolean"}

import gc

from borzoi_pytorch import Borzoi

# The sets whose full arrays the figures need. Everything else is summarised and discarded.
KEEP_FULL_ARRAYS = {'haplotype_4'}

SUMMARY_DIR = OUTPUT_DIR / 'allele_set_summaries'
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# The reduction needs to know which tracks and which bins to summarise over. Both are
# properties of the window and the gene, so they are settled here rather than in the figure
# cells, and the figure cells recompute their own from the arrays they keep.
_rna_index_all = rna_targets.index.to_numpy()
if DROP_ANTISENSE:
    _rna_index_all = rna_targets.index[rna_targets['strand'] != '-'].to_numpy()
print(f'summarising over {len(_rna_index_all):,} RNA tracks; windows containing {GENE}: '
      + ', '.join(w for w in WINDOWS
                  if 0 <= bin_of(GENE_COORDINATES[1], WINDOWS[w])
                  < OUTPUT_CONTEXT_LENGTH // BIN_SIZE))
FC_PSEUDOCOUNT_EARLY = 1e-3


def gene_bins(window_name: str):
    """First and last output bin of the gene in this window, or None if it is outside.

    The centred window puts the variants in the middle of the output, which pushes MAP3K1
    beyond its right edge — so there is no gene to quantify there, and a summary over that
    window would be taken across an empty slice. Checked rather than assumed.
    """
    n_bins = OUTPUT_CONTEXT_LENGTH // BIN_SIZE
    start = bin_of(GENE_COORDINATES[1], WINDOWS[window_name])
    if start >= n_bins or start < 0:
        return None
    return max(start, 0), n_bins


def summarise(baseline: np.ndarray, perturbed: np.ndarray, window_name: str) -> pd.DataFrame:
    """Per-track effect of one allele set, over the part of the gene inside this window.

    This is the whole of what is retained for a set, about 100 kB against the 187 MB of the
    array it replaces.
    """
    gene_start, _ = gene_bins(window_name)
    base = baseline[_rna_index_all, gene_start:]
    diff = perturbed[_rna_index_all, gene_start:] - base
    table = pd.DataFrame({
        'track_index': _rna_index_all,
        'description': targets.loc[_rna_index_all, 'description'].to_numpy(),
        'strand': targets.loc[_rna_index_all, 'strand'].to_numpy(),
        'mean_baseline': base.mean(axis=1),
        'mean_difference': diff.mean(axis=1),
        'mean_abs_difference': np.abs(diff).mean(axis=1),
        'integrated_difference': diff.sum(axis=1) * BIN_SIZE,
        'log2_fold_change': np.log2((perturbed[_rna_index_all, gene_start:].sum(axis=1)
                                     + FC_PSEUDOCOUNT_EARLY)
                                    / (base.sum(axis=1) + FC_PSEUDOCOUNT_EARLY)),
    })
    table['is_adipose'] = table['description'].str.contains(ADIPOSE_PATTERN, case=False)
    del base, diff
    return table


borzoi = Borzoi.from_pretrained(f'johahi/borzoi-replicate-{FOLD}').to(DEVICE).eval()
report_memory('model loaded')


@torch.no_grad()
def predict(sequence: str) -> np.ndarray:
    tensor = one_hot(sequence).to(DEVICE)
    output = borzoi(tensor)
    array = output.squeeze(0).float().cpu().numpy()
    del output, tensor
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return array


# --- reference passes, one per window, reused by every allele set on that window ----------
baseline_predictions = {}
for window_name, reference in reference_sequences.items():
    print(f'{window_name}: reference ...', flush=True)
    baseline_predictions[window_name] = predict(reference)
report_memory(f'{len(baseline_predictions)} reference predictions '
              f'({array_gb(*baseline_predictions.values()):.2f} GB)')

# --- one pass per allele set, reduced immediately ----------------------------------------
allele_set_predictions = {}          # full arrays, only for the sets the figures draw
allele_set_effects = {}              # per-track summaries, for everything

_n_total = sum(len(w) for w in allele_set_sequences.values())
_done = 0
for set_name, per_window in allele_set_sequences.items():
    for window_name, perturbed_sequence in per_window.items():
        _done += 1
        summary_path = SUMMARY_DIR / f'{set_name}__{window_name}.csv'

        if (RESUME_FROM_DISK and summary_path.exists()
                and set_name not in KEEP_FULL_ARRAYS
                and gene_bins(window_name) is not None):
            allele_set_effects[(set_name, window_name)] = pd.read_csv(summary_path)
            print(f'[{_done}/{_n_total}] {set_name} / {window_name}: from disk', flush=True)
            continue

        print(f'[{_done}/{_n_total}] {set_name} / {window_name} ...', flush=True)
        prediction = predict(perturbed_sequence)

        if gene_bins(window_name) is None:
            # No part of the gene is inside this window's output, so there is nothing to
            # summarise. The array is still kept if a figure draws it.
            print(f'    {GENE} is outside the {window_name} output window, no summary')
        else:
            summary = summarise(baseline_predictions[window_name], prediction, window_name)
            summary.to_csv(summary_path, index=False)
            allele_set_effects[(set_name, window_name)] = summary

        if set_name in KEEP_FULL_ARRAYS:
            allele_set_predictions.setdefault(set_name, {})[window_name] = prediction
        else:
            # The array has served its purpose; the summary is what the rest of the notebook
            # reads. Dropping it here is what keeps peak memory flat across the loop.
            del prediction
            gc.collect()

report_memory(f'{_done} allele-set passes done, '
              f'{sum(len(v) for v in allele_set_predictions.values())} full array(s) kept')

# The published panels read `predictions`, so keep it pointing at the original four.
predictions = {
    window_name: {'baseline': baseline_predictions[window_name],
                  'perturbed': allele_set_predictions['haplotype_4'][window_name]}
    for window_name in WINDOWS
}

# One table across every allele set, which is what the summaries are for.
ALLELE_SET_EFFECTS = pd.concat(
    [table.assign(allele_set=name, window=window)
     for (name, window), table in allele_set_effects.items()],
    ignore_index=True,
)
ALLELE_SET_EFFECTS.to_csv(OUTPUT_DIR / f'Tab_borzoi_{GENE}_allele_set_effects.csv',
                          index=False)
print(f'\n{len(allele_set_effects)} allele-set summaries '
      f'({len(ALLELE_SET_EFFECTS):,} rows) written to '
      f'Tab_borzoi_{GENE}_allele_set_effects.csv')

# The sequences are 524 kb strings and are not needed again once predicted.
release('allele_set_sequences', 'borzoi')
report_memory('model and sequences released')


## Supplementary Figure 8 — differences across all output channels

Every channel the model predicts, baseline against haplotype, as a heat map over the 6,144
output bins. The point is where in the genome and in which assays anything moves at all: the
strongest changes sit around the substituted bases themselves, with the RNA-seq channels
responding at the edge of *C5orf67*.

Panel **a** is the centred window (variants in the middle of the output, *MAP3K1* outside it);
panel **b** starts 100 bp upstream of rs459193 and brings *MAP3K1* — its promoter in particular —
into the predicted span. The colour scale is logarithmic, which is what makes the structure
visible at all; on a linear scale the whole map is flat. The map is drawn at the model's native
32 bp resolution — one column per output bin, no pooling — so the file is large; raise the dpi
rather than pooling if columns look coarse. Panels carry no titles: left is the centred window,
right the upstream one, and the cell prints the exact coordinates beneath the figure.

In [ ]:
# @title Supplementary Figure 8: output differences across channels { display-mode: "form" }
# @markdown Half-width, in bins, of the red box drawn around the variants.
BOX_HALF_WIDTH = 60  # @param {type:"slider", min:10, max:200, step:10}
# @markdown Figure size in inches. The map is drawn at the model's native resolution — all
# @markdown 7,611 channels x 6,144 bins, one column per 32 bp bin, with no pooling — so give
# @markdown it enough width and dpi that neighbouring bins stay distinguishable.
S8_FIG_WIDTH = 16  # @param {type:"slider", min:10, max:26, step:1}
S8_FIG_HEIGHT = 8  # @param {type:"slider", min:5, max:16, step:1}
# @markdown Rendering the full 7,611 x 6,144 matrix costs roughly 4 GB while matplotlib
# @markdown resamples it, which Colab has and a small VM may not. If the session dies here,
# @markdown raise this to pool *rows* (output channels) — columns, and so the 32 bp
# @markdown resolution along the genome, are never pooled.
S8_ROW_POOL = 1  # @param {type:"slider", min:1, max:8, step:1}


import gc

from matplotlib.ticker import FuncFormatter


# Assay block boundaries, so the y-axis can be labelled by assay rather than track index.
assay_bounds = {}
for assay, group in targets.groupby('assay', sort=False):
    assay_bounds[assay] = (int(group.index.min()), int(group.index.max()))

fig, axes = plt.subplots(1, 2, figsize=(S8_FIG_WIDTH, S8_FIG_HEIGHT))
for ax, (name, arrays) in zip(axes, predictions.items()):
    # Native 32 bp resolution: every output bin gets its own column. Matplotlib downsamples
    # for the screen but the saved file keeps the full matrix, so raise the dpi below rather
    # than pooling if the columns look coarse.
    # Kept in float32 and clipped in place: at full resolution the matrix is ~190 MB, and
    # every incidental copy (np.clip, an implicit float64 promotion inside the norm) costs
    # another one. Peak usage is a few GB, which Colab has and a small VM may not.
    difference = np.abs(
        arrays['perturbed'] - arrays['baseline'], dtype=np.float32
    )
    if S8_ROW_POOL > 1:
        usable = (difference.shape[0] // S8_ROW_POOL) * S8_ROW_POOL
        difference = difference[:usable].reshape(
            usable // S8_ROW_POOL, S8_ROW_POOL, difference.shape[1]
        ).max(axis=1)
    positive = difference[difference > 0]
    floor = float(np.percentile(positive[::37], 1)) if positive.size else 1e-12
    del positive
    # The log is taken here, in float32, and shown with a linear norm: LogNorm would build
    # a float64 masked copy of the whole matrix, which at this size is another 370 MB.
    np.maximum(difference, floor, out=difference)
    np.log10(difference, out=difference)
    image = ax.imshow(difference, aspect='auto', cmap='binary',
                      interpolation='nearest')
    centre = bin_of(LEFTMOST, WINDOWS[name])
    if centre >= 0:
        ax.add_patch(plt.Rectangle(
            (centre - BOX_HALF_WIDTH, 0), 2 * BOX_HALF_WIDTH, difference.shape[0],
            fill=False, edgecolor='red', linewidth=1.0,
        ))
    gene_bin = bin_of(GENE_COORDINATES[1], WINDOWS[name])
    if gene_bin >= 0:
        ax.axvline(gene_bin, color='tab:blue', lw=0.8, ls='dashed')
    for assay, (first, last) in assay_bounds.items():
        ax.text(-260, (first + last) / 2 / S8_ROW_POOL, assay, fontsize=8,
                va='center', ha='right')
        ax.axhline(last / S8_ROW_POOL, color='white', lw=0.4)
    ax.set_xlabel(f'Bins ({BIN_SIZE} bp resolution)', fontsize=11)
    ax.tick_params(axis='x', labelsize=10)
    ax.set_yticks([])
    colorbar = fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
    colorbar.ax.yaxis.set_major_formatter(
        FuncFormatter(lambda value, _: f'$10^{{{value:.0f}}}$')
    )
    colorbar.ax.tick_params(labelsize=9)
    del difference
    gc.collect()

axes[0].set_ylabel('Output channels', fontsize=11)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'SuppFig_borzoi_channel_differences.png', dpi=400, bbox_inches='tight')
plt.show()

print('Left: centred window, input starts '
      f'{WINDOWS["centred"]:,}. Right: upstream window, input starts '
      f'{WINDOWS["upstream"]:,}. Blue dashed line: {GENE} TSS; red box: the variants.')

## Supplementary Figures 9 and 10 — tracks in adipose contexts

The same comparison read one track at a time, for the adipose and mesenchymal contexts across
assays: CAGE, DNase, ATAC, the histone ChIP marks, and RNA-seq. The left column overlays
baseline and haplotype predictions; the right column plots their difference, which is where a
change of this size is actually visible.

**Supplementary Figure 9** uses the centred window — the variants are in the middle of the
output span, so this shows what happens *locally*, around the variants themselves.
**Supplementary Figure 10** uses the upstream window, which is the one containing *MAP3K1*: the
perturbations do reach the gene, but weakly, and in the opposite direction to the one expected
from the eQTL data. That directional mismatch is a known property of these models rather than a
result about the locus, and the manuscript reports it as such.

In [ ]:
# @title Supplementary Figures 9 and 10: per-track profiles { display-mode: "form" }
# @markdown Tracks drawn per figure, chosen from the adipose-matching tracks across assays.
N_PROFILE_TRACKS = 14  # @param {type:"slider", min:6, max:24, step:2}
# @markdown Half-width in bins of the plotted region, centred on the variants (Figure 9) or
# @markdown spanning the whole output (Figure 10 uses the full span regardless).
PROFILE_HALF_WIDTH = 3000  # @param {type:"slider", min:500, max:3072, step:500}
# @markdown Figure width in inches, and height per track row.
PROFILE_FIG_WIDTH = 24  # @param {type:"slider", min:13, max:34, step:1}
PROFILE_ROW_HEIGHT = 1.6  # @param {type:"slider", min:1.0, max:3.0, step:0.1}
# @markdown Font sizes for the axis tick numbers and for the track names on the left.
PROFILE_TICK_FONTSIZE = 11  # @param {type:"slider", min:6, max:18, step:1}
PROFILE_LABEL_FONTSIZE = 9  # @param {type:"slider", min:6, max:16, step:1}

# Adipose tracks across assays, one per description so the panel is not filled with the
# strand partner of the same experiment.
adipose_all = targets[
    targets['description'].str.contains(ADIPOSE_PATTERN, case=False)
    & (targets['strand'] != '-')
]
selection, seen = [], set()
for assay in ['CAGE', 'DNASE', 'ATAC', 'CHIP', 'RNA']:
    block = adipose_all[adipose_all['assay'] == assay]
    for index, row in block.iterrows():
        if row['description'] in seen:
            continue
        seen.add(row['description'])
        selection.append((int(index), row['description']))
        if sum(1 for i, _ in selection if targets.loc[i, 'assay'] == assay) >= 4:
            break
selection = selection[:N_PROFILE_TRACKS]
print(f'{len(selection)} tracks drawn per figure.')


def profile_figure(window_name: str, figure_label: str, half_width: int):
    arrays = predictions[window_name]
    n_bins = arrays['baseline'].shape[1]
    centre = bin_of(LEFTMOST, WINDOWS[window_name])
    if centre < 0 or half_width >= n_bins // 2:
        first, last = 0, n_bins
    else:
        first, last = max(0, centre - half_width), min(n_bins, centre + half_width)
    span = np.arange(first, last)
    # x axis in kb from the leftmost variant, as in the published panels
    offsets = ((span * BIN_SIZE) + WINDOWS[window_name] + CROP - LEFTMOST) / 1000

    fig, axes = plt.subplots(
        len(selection), 2,
        figsize=(PROFILE_FIG_WIDTH, PROFILE_ROW_HEIGHT * len(selection)),
        sharex=True, squeeze=False,
    )
    for row, (index, description) in enumerate(selection):
        baseline = arrays['baseline'][index, first:last]
        perturbed = arrays['perturbed'][index, first:last]
        difference = perturbed - baseline

        # Both profiles are filled down to zero rather than one filled and one drawn as a
        # line: coverage is a quantity above a zero floor, and filling both makes the
        # comparison a comparison of areas. The perturbed fill is the more transparent of
        # the two so the baseline underneath stays visible where they overlap.
        axes[row, 0].fill_between(offsets, baseline, 0, color='tab:purple', alpha=0.45,
                                  lw=0, label='Baseline')
        axes[row, 0].fill_between(offsets, perturbed, 0, color='tab:red', alpha=0.25,
                                  lw=0, label='Perturbed')
        axes[row, 0].plot(offsets, perturbed, color='tab:red', lw=0.5)

        # The difference is signed, so its fill runs to zero in both directions.
        axes[row, 1].fill_between(offsets, difference, 0, color='tab:blue', alpha=0.30,
                                  lw=0, label='Perturbed - Baseline')
        axes[row, 1].plot(offsets, difference, color='tab:blue', lw=0.5)
        axes[row, 1].axhline(0, color='grey', lw=0.5, ls='dotted')

        # Track names label the left column only, so the two panels can sit close together.
        axes[row, 0].set_ylabel(
            description[:46] + ('...' if len(description) > 46 else ''),
            fontsize=PROFILE_LABEL_FONTSIZE, rotation=0, ha='right', va='center',
        )
        for column in (0, 1):
            axes[row, column].tick_params(labelsize=PROFILE_TICK_FONTSIZE)
            axes[row, column].axvline(0, color='grey', lw=0.5, ls='dotted')

    axes[0, 0].legend(fontsize=PROFILE_TICK_FONTSIZE - 1, loc='upper right')
    axes[0, 1].legend(fontsize=PROFILE_TICK_FONTSIZE - 1, loc='upper right')
    for column in (0, 1):
        axes[-1, column].set_xlabel('Position (kb from rs459193)',
                                    fontsize=PROFILE_TICK_FONTSIZE + 1)
    fig.suptitle(figure_label, fontsize=13, y=1.002)
    fig.tight_layout()
    fig.subplots_adjust(wspace=0.07, hspace=0.25)
    return fig


figure9 = profile_figure(
    'centred',
    'Supplementary Figure 9. Predictions centred at rs459193, reference vs. four-variant '
    'haplotype',
    PROFILE_HALF_WIDTH,
)
figure9.savefig(OUTPUT_DIR / 'SuppFig_borzoi_profiles_variant_centred.png', dpi=250, bbox_inches='tight')
plt.show()

figure10 = profile_figure(
    'upstream',
    'Supplementary Figure 10. Predictions from an input starting 100 bp upstream of '
    f'rs459193 ({GENE} in the output window)',
    10**9,  # full span
)
figure10.savefig(OUTPUT_DIR / 'SuppFig_borzoi_profiles_upstream_window.png', dpi=250,
                 bbox_inches='tight')
plt.show()

## Figure 4a,b — *MAP3K1* effects across RNA tracks

Replaces the stacked `Figure3bc` panel. Both panels are the same height and sit side by side,
type is larger throughout, and the boxes are narrower. The scatter's fifteen-entry legend is
gone: adipose tracks are drawn in orange, which is the same cue panel a already uses on its
labels, so the two panels can be read together without a key.


In [ ]:
# @title Figure 4a,b: MAP3K1 effects across RNA tracks (side by side) { display-mode: "form" }
# Panels sit side by side at equal height. Adipose tracks are drawn in orange, matching
# the label colouring in panel a, so the two panels can be read together without a
# fifteen-entry legend.
WINDOW_FOR_FIGURE = 'upstream'
# Downstream cells (effect measures, enrichment) refer to this window by its
# original name. Keep both bound to the same value.
WINDOW_FOR_FIGURE3 = WINDOW_FOR_FIGURE

arrays = predictions[WINDOW_FOR_FIGURE]
start_bin = bin_of(GENE_COORDINATES[1], WINDOWS[WINDOW_FOR_FIGURE])
n_bins = arrays['baseline'].shape[1]
if start_bin < 0:
    raise ValueError(f'{GENE} does not start inside the {WINDOW_FOR_FIGURE} output window.')
region_bp = (n_bins - start_bin) * BIN_SIZE
print(f"Region quantified: bins {start_bin}-{n_bins} = 5' {region_bp:,} bp of {GENE}.")

rna_index = rna_targets.index.to_numpy()
if DROP_ANTISENSE:
    rna_index = rna_targets.index[rna_targets['strand'] != '-'].to_numpy()
    print(f'{len(rna_index):,} RNA tracks kept of {len(rna_targets):,} '
          '(minus-strand tracks dropped).')

baseline_region = arrays['baseline'][rna_index, start_bin:]
difference_region = arrays['perturbed'][rna_index, start_bin:] - baseline_region

track_effects = pd.DataFrame({
    'track_index': rna_index,
    'description': targets.loc[rna_index, 'description'].to_numpy(),
    'strand': targets.loc[rna_index, 'strand'].to_numpy(),
    'mean_baseline': baseline_region.mean(axis=1),
    'mean_difference': difference_region.mean(axis=1),
    'mean_abs_difference': np.abs(difference_region).mean(axis=1),
})
track_effects['is_adipose'] = track_effects['description'].str.contains(
    ADIPOSE_PATTERN, case=False
)
order = track_effects['mean_abs_difference'].to_numpy().argsort()[::-1]
track_effects = track_effects.iloc[order].reset_index(drop=True)
difference_region = difference_region[order]
track_effects.to_csv(OUTPUT_DIR / f'Tab_borzoi_{GENE}_track_effects.csv', index=False)

ADIPOSE_COLOUR = '#d95f02'
GREY = '#6e6e6e'


def short(label, width=40):
    label = label.replace('RNA:', '')
    return label if len(label) <= width else label[: width - 3] + '...'


n_box = min(N_TOP_BOX, len(track_effects))

with plt.rc_context({'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12,
                     'xtick.labelsize': 11, 'ytick.labelsize': 11}):
    fig, (ax_box, ax_scatter) = plt.subplots(
        1, 2, figsize=(15.5, 0.42 * n_box + 1.8), width_ratios=[1.35, 1]
    )

    # --- a) ranked tracks ------------------------------------------------
    ax_box.boxplot(
        [difference_region[i] for i in range(n_box)][::-1], vert=False, widths=0.32,
        flierprops={'marker': '.', 'markersize': 1.6, 'markerfacecolor': GREY,
                    'markeredgecolor': 'none', 'alpha': 0.5},
        medianprops={'color': ADIPOSE_COLOUR, 'linewidth': 1.4},
        boxprops={'linewidth': 0.9}, whiskerprops={'linewidth': 0.9},
        capprops={'linewidth': 0.9},
    )
    ax_box.set_yticklabels(
        [short(track_effects['description'].iloc[i]) for i in range(n_box)][::-1]
    )
    for tick, index in zip(ax_box.get_yticklabels(), range(n_box)[::-1]):
        if track_effects['is_adipose'].iloc[index]:
            tick.set_color(ADIPOSE_COLOUR)
    ax_box.axvline(0, ls='dotted', lw=1.0, color='grey')
    ax_box.set_xlabel('Difference (perturbed - baseline)')
    ax_box.set_title('a) Tracks ranked by mean absolute difference', loc='left')
    ax_box.ticklabel_format(axis='x', style='sci', scilimits=(-3, -3))

    # --- b) effect vs baseline -------------------------------------------
    other = ~track_effects['is_adipose']
    ax_scatter.scatter(track_effects.loc[other, 'mean_baseline'],
                       track_effects.loc[other, 'mean_difference'],
                       s=20, facecolor='none', edgecolor=GREY, alpha=0.45,
                       linewidth=0.7, label='other RNA tracks')
    adipose = track_effects['is_adipose']
    ax_scatter.scatter(track_effects.loc[adipose, 'mean_baseline'],
                       track_effects.loc[adipose, 'mean_difference'],
                       s=42, color=ADIPOSE_COLOUR, edgecolor='white', linewidth=0.6,
                       zorder=3, label='adipose-related')
    ax_scatter.axhline(0, ls='dashed', lw=1.0, color='grey')
    ax_scatter.set_xlabel('Mean baseline signal')
    ax_scatter.set_ylabel('Mean change (perturbed - baseline)')
    ax_scatter.set_title(f'b) Effect vs baseline expression, {GENE}', loc='left')
    ax_scatter.legend(fontsize=11, frameon=False, loc='lower left')
    ax_scatter.ticklabel_format(axis='y', style='sci', scilimits=(-3, -3))
    for spine in ('top', 'right'):
        ax_scatter.spines[spine].set_visible(False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'Fig_borzoi_{GENE}_tracks_and_baseline.png', dpi=400, bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / f'Fig_borzoi_{GENE}_tracks_and_baseline.pdf', bbox_inches='tight')
plt.show()


## Alternative effect measures

Mean absolute difference answers "how much did this track move", which is fine for ranking but
awkward to state in a legend: the units are scaled coverage, the sign is thrown away, and the
value depends on how much of the gene body sits in the window. Three other measures are computed
here from the same predictions, each answering a different question.

| Measure | Question | Units |
|---|---|---|
| `mean_difference` | Which way did it move, on average per bin? | scaled coverage |
| `integrated_difference` | How much predicted signal was gained or lost across the region? | coverage x bp |
| `exon_*` variants of both | The same, over exonic bins only | as above |
| `log2_fold_change` | By what **fraction** did predicted expression change? | log2 ratio |

The last is the one that usually reads best in a manuscript, because it is dimensionless and
comparable across tracks with very different expression levels: a track at high baseline and one
at low baseline can be said to change by the same percentage. The absolute measures cannot be
compared that way — a large coverage change in a highly expressed track may be a smaller
*relative* effect than a modest change in a quiet one.

Restricting to exons matters because intronic bins are mostly baseline noise in these
predictions: they dilute a mean and add variance to a sum without carrying much signal about
expression. Exon coordinates come from Ensembl for *MAP3K1*; if that request fails the notebook
falls back to defining exons empirically, as the bins whose baseline signal exceeds a quantile of
the region — which is what the original analysis did by eye with a threshold of 1.

**Integrated** means summed rather than averaged, times bin width. It is the measure least
sensitive to how much of the gene fits in the window in terms of interpretation — it is a total,
not a rate — but for exactly that reason it is *not* comparable to a run over a different region,
so quote it with the region stated.

Every measure is computed for every track. The figures below show all four side by side, with
adipose-related tracks highlighted, so the claim "the effect concentrates in adipose contexts"
can be checked under each rather than resting on the one that happened to be plotted.

In [ ]:
# @title Compute signed, exonic and integrated effect measures { display-mode: "form" }
# @markdown Where exon coordinates come from. "ensembl" queries the REST API for MAP3K1's
# @markdown exons; "signal" defines exonic bins empirically as those whose baseline exceeds a
# @markdown quantile of the region, pooled across the most-expressing tracks.
EXON_SOURCE = "ensembl"  # @param ["ensembl", "signal"]
# @markdown Quantile used by the "signal" definition (and by the fallback).
SIGNAL_EXON_QUANTILE = 0.90  # @param {type:"slider", min:0.5, max:0.99, step:0.01}
# @markdown Pseudocount added to both sums before the log2 ratio, so tracks with near-zero
# @markdown predicted expression cannot produce enormous fold changes out of numerical noise.
FC_PSEUDOCOUNT = 1e-3  # @param {type:"number"}

region_bins = np.arange(start_bin, n_bins)
region_starts = WINDOWS[WINDOW_FOR_FIGURE] + CROP + region_bins * BIN_SIZE


def ensembl_exons(chromosome: str, start: int, end: int, gene: str) -> list:
    """Exon intervals of `gene` in the region, from Ensembl REST (1-based, inclusive)."""
    response = requests.get(
        f'https://rest.ensembl.org/overlap/region/human/'
        f'{chromosome.removeprefix("chr")}:{start}-{end}',
        params={'feature': 'exon'},
        headers={'Content-Type': 'application/json'},
        timeout=120,
    )
    response.raise_for_status()
    exons = [
        (int(entry['start']), int(entry['end']))
        for entry in response.json()
        if entry.get('Parent') and str(entry.get('seq_region_name')) == chromosome.removeprefix('chr')
    ]
    return exons


exon_mask = None
if EXON_SOURCE == 'ensembl':
    try:
        exons = ensembl_exons(*GENE_COORDINATES, GENE)
        # A bin is exonic if it overlaps any annotated exon. Exons of all transcripts are
        # unioned: a bin is 32 bp and transcript models disagree at that resolution anyway.
        exon_mask = np.zeros(len(region_bins), dtype=bool)
        for exon_start, exon_end in exons:
            overlapping = (region_starts + BIN_SIZE > exon_start) & (region_starts <= exon_end)
            exon_mask |= overlapping
        print(f'{len(exons)} Ensembl exon records in the region; '
              f'{int(exon_mask.sum())} of {len(region_bins)} bins are exonic '
              f'({exon_mask.mean():.1%}).')
    except Exception as exc:  # noqa: BLE001
        print(f'Ensembl exon lookup failed ({exc!r}) — falling back to the signal definition.')

if exon_mask is None or not exon_mask.any():
    # Empirical definition: bins where the pooled baseline of the best-expressing tracks is
    # high. This is the original analysis's threshold-by-eye, made explicit and track-pooled.
    pooled = arrays['baseline'][rna_index, start_bin:]
    strongest = pooled[pooled.mean(axis=1).argsort()[::-1][:50]]
    profile = strongest.mean(axis=0)
    exon_mask = profile >= np.quantile(profile, SIGNAL_EXON_QUANTILE)
    print(f'Signal-defined exonic bins: {int(exon_mask.sum())} of {len(region_bins)} '
          f'({exon_mask.mean():.1%}), threshold at the {SIGNAL_EXON_QUANTILE:.0%} quantile.')

baseline_region = arrays['baseline'][rna_index, start_bin:]
perturbed_region = arrays['perturbed'][rna_index, start_bin:]
difference = perturbed_region - baseline_region


def log2_fold_change(baseline_values, perturbed_values, mask=None):
    """log2 of total perturbed over total baseline signal, with a pseudocount."""
    if mask is not None:
        baseline_values, perturbed_values = baseline_values[:, mask], perturbed_values[:, mask]
    return np.log2(
        (perturbed_values.sum(axis=1) + FC_PSEUDOCOUNT)
        / (baseline_values.sum(axis=1) + FC_PSEUDOCOUNT)
    )


metrics = pd.DataFrame({
    'track_index': rna_index,
    'description': targets.loc[rna_index, 'description'].to_numpy(),
    'strand': targets.loc[rna_index, 'strand'].to_numpy(),
    'mean_baseline': baseline_region.mean(axis=1),
    'exon_mean_baseline': baseline_region[:, exon_mask].mean(axis=1),
    # Signed, per bin
    'mean_difference': difference.mean(axis=1),
    'exon_mean_difference': difference[:, exon_mask].mean(axis=1),
    # Integrated over the region: a total, in coverage x bp
    'integrated_difference': difference.sum(axis=1) * BIN_SIZE,
    'exon_integrated_difference': difference[:, exon_mask].sum(axis=1) * BIN_SIZE,
    # Relative, dimensionless
    'log2_fold_change': log2_fold_change(baseline_region, perturbed_region),
    'exon_log2_fold_change': log2_fold_change(baseline_region, perturbed_region, exon_mask),
    # The original ranking measure, kept for comparison
    'mean_abs_difference': np.abs(difference).mean(axis=1),
})
metrics['is_adipose'] = metrics['description'].str.contains(ADIPOSE_PATTERN, case=False)
metrics['percent_change'] = (2 ** metrics['exon_log2_fold_change'] - 1) * 100
metrics.to_csv(OUTPUT_DIR / f'Tab_borzoi_{GENE}_effect_measures.csv', index=False)

print(f'\n{len(metrics)} tracks x {len(metrics.columns) - 4} measures written to '
      f'Tab_borzoi_{GENE}_effect_measures.csv')
print(f'Region: bins {start_bin}-{n_bins} ({region_bp:,} bp), '
      f'{int(exon_mask.sum()) * BIN_SIZE:,} bp exonic.')

# Quoted in the Results: how many tracks move down, adipose ones included.
_reduced = metrics['exon_log2_fold_change'] < 0
_adipose = metrics['is_adipose']
print(f'Direction: {int(_reduced.sum())} of {len(metrics)} tracks '
      f'({_reduced.mean():.0%}) show a reduced predicted expression, including '
      f'{int((_reduced & _adipose).sum())} of {int(_adipose.sum())} adipose-related '
      f'tracks.')
display(
    metrics.sort_values('exon_log2_fold_change')
    [['description', 'is_adipose', 'exon_mean_baseline', 'exon_mean_difference',
      'exon_integrated_difference', 'exon_log2_fold_change', 'percent_change']]
    .head(15)
)

### Do adipose tracks stand out under each measure?

The first figure is the direct test of the claim: for each measure, the adipose tracks against
everything else. If the effect really concentrates in adipose contexts, the orange distribution
sits away from the grey one under every measure, not just the one plotted in the manuscript.
Each panel carries a two-sided Mann-Whitney U test with the rank-biserial correlation as effect
size.

Two cautions on reading those p-values. The track panel is a convenience sample — tissues are
represented very unequally, and a handful of adipose experiments from one lab are not
independent observations — so the test describes the predictions, not adipose biology. And the
measures are strongly correlated with each other, so four small p-values are close to one
finding repeated, not four.

The second figure takes whichever measure you pick and shows it two ways: tracks ranked, and
plotted against baseline expression. The latter is where the ceiling effect lives — under the
absolute measures the wedge is unavoidable, since a track with no predicted expression cannot
change much; under the relative measure it should largely disappear, which is the point of using
a ratio. If adipose tracks still lead once the ceiling is removed, that is a stronger version of
the manuscript's claim than the original panel makes.

In [ ]:
# @title Figure 4: every measure, adipose vs the rest { display-mode: "form" }
# @markdown Measure used for the detail figure and the ranked panel.
FOCUS_MEASURE = "exon_log2_fold_change"  # @param ["exon_log2_fold_change", "log2_fold_change", "exon_mean_difference", "mean_difference", "exon_integrated_difference", "integrated_difference", "mean_abs_difference"]
# @markdown Tracks named in the ranked panel.
N_RANKED = 30  # @param {type:"slider", min:10, max:60, step:5}

MEASURES = {
    'mean_difference': 'Mean difference\n(signed, per bin)',
    'exon_mean_difference': 'Exonic mean difference\n(signed, per bin)',
    'exon_integrated_difference': 'Exonic integrated difference\n(coverage x bp)',
    'exon_log2_fold_change': 'Exonic log2 fold change\n(relative)',
}

adipose = metrics[metrics['is_adipose']]
other = metrics[~metrics['is_adipose']]
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, len(MEASURES), figsize=(4.1 * len(MEASURES), 5.4))
summary_rows = []
for ax, (column, label) in zip(axes, MEASURES.items()):
    groups = [other[column].to_numpy(), adipose[column].to_numpy()]
    ax.boxplot(groups, showfliers=False, widths=0.55,
               medianprops={'color': 'black'})
    for position, (values, colour) in enumerate(
        zip(groups, ['dimgrey', 'tab:orange']), start=1
    ):
        ax.scatter(position + rng.normal(0, 0.06, len(values)), values, s=10,
                   color=colour, alpha=0.35 if position == 1 else 0.9, zorder=3,
                   edgecolor='none')
    statistic, p_value = stats.mannwhitneyu(groups[1], groups[0], alternative='two-sided')
    rank_biserial = 2 * statistic / (len(groups[0]) * len(groups[1])) - 1
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f'Other\n(n={len(other)})', f'Adipose\n(n={len(adipose)})'],
                       fontsize=8)
    ax.axhline(0, ls='dashed', lw=0.8, color='grey')
    ax.set_title(f'{label}\np = {p_value:.2e}, r = {rank_biserial:.2f}', fontsize=9)
    summary_rows.append({
        'measure': column,
        'adipose_median': float(np.median(groups[1])),
        'other_median': float(np.median(groups[0])),
        'mannwhitney_p': p_value,
        'rank_biserial': rank_biserial,
    })

fig.suptitle(f'Figure 4. Effect of the four-variant haplotype on predicted {GENE} expression, '
             'by measure', fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'Fig_borzoi_{GENE}_adipose_vs_other_measures.png', dpi=250, bbox_inches='tight')
plt.show()

measure_summary = pd.DataFrame(summary_rows)
measure_summary.to_csv(OUTPUT_DIR / f'Tab_borzoi_{GENE}_measure_summary.csv', index=False)
display(measure_summary)

In [ ]:
# @title Supplementary Table 12: adipose separation under each effect measure
# One row per measure, and deliberately two kinds of column. The top-n count is what the
# manuscript has quoted; the rank-biserial and its bootstrap interval are what can be quoted
# without a re-run changing them. Where the two disagree, the second is the one to believe:
# a count of five is one draw from a distribution, and the interval says how wide that
# distribution is.
#
# `stable` marks the measures whose interval stays on one side of zero. A measure that is
# strong here and weak in the count column is not a contradiction — it means the enrichment
# is real but the top of the ranking is crowded.
_top_n = int(globals().get('ENRICH_TOP_N', 20))
_resamples = int(globals().get('N_RESAMPLES', 2000))

rows = []
for column in list(MEASURES) + ['mean_abs_difference']:
    values = metrics[column].to_numpy()
    flags = metrics['is_adipose'].to_numpy()
    # Ranked by absolute effect, as the published figure does.
    order = np.argsort(-np.abs(values))
    n_total, n_adipose = len(values), int(flags.sum())
    hits = int(flags[order[:_top_n]].sum())
    expected = _top_n * n_adipose / n_total

    statistic, p_rank = stats.mannwhitneyu(values[flags], values[~flags],
                                           alternative='two-sided')
    effect = 2 * statistic / (int((~flags).sum()) * n_adipose) - 1
    draws = []
    for _ in range(_resamples):
        a = values[flags][RNG.integers(0, n_adipose, n_adipose)]
        o = values[~flags][RNG.integers(0, n_total - n_adipose, n_total - n_adipose)]
        draws.append(2 * stats.mannwhitneyu(a, o, alternative='two-sided')[0]
                     / (len(a) * len(o)) - 1)
    low, high = float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

    rows.append({
        'measure': column,
        'adipose_in_top_n': hits,
        'expected': round(expected, 2),
        'fold_enrichment': round(hits / expected, 2) if expected else np.nan,
        'hypergeom_p': stats.hypergeom.sf(hits - 1, n_total, n_adipose, _top_n),
        'rank_biserial': round(effect, 3),
        'CI_low': round(low, 3),
        'CI_high': round(high, 3),
        'mannwhitney_p': p_rank,
        'stable': (low > 0) == (high > 0),
    })
measure_enrichment = pd.DataFrame(rows)
measure_enrichment.to_csv(OUTPUT_DIR / f'Tab_borzoi_{GENE}_measure_enrichment.csv',
                          index=False)
display(measure_enrichment)

_stable = measure_enrichment[measure_enrichment.stable]
if len(_stable):
    _best = _stable.reindex(_stable.rank_biserial.abs().sort_values(ascending=False).index)
    print(f'{len(_stable)} of {len(measure_enrichment)} measures separate adipose from the '
          f'rest with an interval\n  that stays on one side of zero. Strongest: '
          f'{_best.iloc[0].measure} ({_best.iloc[0].rank_biserial:+.3f}, '
          f'{_best.iloc[0].CI_low:+.3f} to {_best.iloc[0].CI_high:+.3f}).')
    print('  Quote a measure from this list rather than the top-n count of one of them.')
else:
    print('No measure separates the two groups with an interval clear of zero in this run.')
_unstable = measure_enrichment[~measure_enrichment.stable]
if len(_unstable):
    print(f'  Intervals crossing zero, i.e. not quotable from a single run: '
          f'{", ".join(_unstable.measure)}.')

## Hypergeometric test — adipose enrichment among the most-affected tracks

How surprising is the number of adipose tracks among the top twenty, given how many exist in the
track universe? The manuscript reports 8 of the top 20 against 33 adipose tracks in 1,543,
a fold enrichment of 18.7 at p = 1.9e-9.

The cell reports the test under both track universes, because the correction changes it. Dropping
antisense tracks removes near-zero-signal tracks that were never going to rank highly, which
shrinks the universe and raises the background rate — so the corrected fold enrichment is
necessarily smaller than the published one even when the same tracks top the ranking. The two
rows are not competing estimates of one quantity; they are the same test on two different track
sets, and the corrected row is the one that matches the corrected figure.

In [ ]:
# @title Adipose enrichment, and whether it survives moving the cut-off { display-mode: "form" }
# The published statistic counts how many adipose tracks land among the twenty most affected.
# That count is decided at the boundary, and the boundary is crowded: this cell prints how
# crowded, because it is the reason the number moves between runs while every conclusion the
# number is supposed to support stays exactly where it was.
#
# Three things are computed instead of one:
#
#   1. the count at the published cut-off, unchanged, so earlier numbers stay comparable
#   2. the same count swept across cut-offs, which separates "adipose tracks respond more"
#      from "twenty happens to be a good place to cut"
#   3. the Mann-Whitney test over every track and its rank-biserial effect size, with a
#      bootstrap interval — neither cuts the ranking anywhere, so neither can be moved by a
#      track crossing a boundary
#
# Quote the third kind. The first is reported for continuity and should be described as what
# it is: one cut-off through a continuous ranking.
ENRICH_TOP_N = 20  # @param {type:"slider", min:5, max:50, step:5}
SWEEP_CUTOFFS = "10, 20, 30, 40, 50, 75, 100"  # @param {type:"string"}
# @markdown Resamples for the bootstrap intervals. 2,000 is enough for a 95% interval.
N_RESAMPLES = 2000  # @param {type:"integer"}
# @markdown How near the cut-off a track has to be to count as "could have gone either way".
TIE_TOLERANCE = 0.01  # @param {type:"number"}

_cutoffs = sorted({int(x) for x in SWEEP_CUTOFFS.replace(',', ' ').split()})


def hypergeometric(flags, order, top_n):
    """Adipose count, expectation, fold enrichment and p at one cut-off."""
    n_total, n_adipose = len(flags), int(flags.sum())
    top_n = min(top_n, n_total)
    hits = int(flags[order[:top_n]].sum())
    expected = top_n * n_adipose / n_total
    return {
        'top_n': top_n,
        'adipose_in_top_n': hits,
        'expected': round(expected, 2),
        'fold_enrichment': round(hits / expected, 2) if expected else np.nan,
        'pvalue': stats.hypergeom.sf(hits - 1, n_total, n_adipose, top_n),
    }


def rank_test(values, flags):
    """Mann-Whitney over all tracks, with the rank-biserial correlation as effect size.

    Rank-biserial is the probability that a randomly chosen adipose track exceeds a randomly
    chosen other one, rescaled to [-1, 1]. It uses every track, so no boundary exists for a
    track to cross, and it is invariant to any monotone rescaling of the measure.
    """
    adipose_values, other_values = values[flags], values[~flags]
    statistic, p_value = stats.mannwhitneyu(adipose_values, other_values,
                                            alternative='two-sided')
    effect = 2 * statistic / (len(adipose_values) * len(other_values)) - 1
    return statistic, p_value, effect


def bootstrap_effect(values, flags, n_resamples):
    """Percentile interval for the rank-biserial, resampling tracks within each group."""
    adipose_values, other_values = values[flags], values[~flags]
    draws = []
    for _ in range(n_resamples):
        a = adipose_values[RNG.integers(0, len(adipose_values), len(adipose_values))]
        o = other_values[RNG.integers(0, len(other_values), len(other_values))]
        statistic = stats.mannwhitneyu(a, o, alternative='two-sided')[0]
        draws.append(2 * statistic / (len(a) * len(o)) - 1)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


def bootstrap_fold(values, flags, top_n, n_resamples):
    """Percentile interval for the fold enrichment at one cut-off, resampling tracks."""
    n = len(values)
    draws = []
    for _ in range(n_resamples):
        idx = RNG.integers(0, n, n)
        f, v = flags[idx], values[idx]
        if not f.any():
            continue
        result = hypergeometric(f, np.argsort(-v), top_n)
        draws.append(result['fold_enrichment'])
    return float(np.nanpercentile(draws, 2.5)), float(np.nanpercentile(draws, 97.5))


rows, sweep_rows = [], []
for label, subset in (
    ('published (all RNA tracks)', rna_targets.index.to_numpy()),
    ('corrected (sense + unstranded)',
     rna_targets.index[rna_targets['strand'] != '-'].to_numpy()),
):
    baseline_subset = arrays['baseline'][subset, start_bin:]
    difference_subset = arrays['perturbed'][subset, start_bin:] - baseline_subset
    descriptions = targets.loc[subset, 'description']
    flags = descriptions.str.contains(ADIPOSE_PATTERN, case=False).to_numpy()

    values = np.abs(difference_subset).mean(axis=1)   # the published ranking measure
    order = np.argsort(-values)

    at_published = hypergeometric(flags, order, ENRICH_TOP_N)
    statistic, p_rank, effect = rank_test(values, flags)
    effect_low, effect_high = bootstrap_effect(values, flags, N_RESAMPLES)
    fold_low, fold_high = bootstrap_fold(values, flags, ENRICH_TOP_N, N_RESAMPLES)

    # How decided is the decision at the cut-off? The gap to the next track, and how many
    # tracks sit within TIE_TOLERANCE of it, say whether the count was ever really settled.
    cut_value = values[order[at_published['top_n'] - 1]]
    next_value = values[order[at_published['top_n']]] if at_published['top_n'] < len(values) \
        else np.nan
    relative_gap = (cut_value - next_value) / cut_value if cut_value else np.nan
    near_cut = int((np.abs(values - cut_value) <= TIE_TOLERANCE * cut_value).sum())

    rows.append({
        'track_universe': label,
        'tracks_total': len(subset),
        'adipose_total': int(flags.sum()),
        **at_published,
        'fold_CI_low': round(fold_low, 2), 'fold_CI_high': round(fold_high, 2),
        'rank_biserial': round(effect, 3),
        'rank_biserial_CI_low': round(effect_low, 3),
        'rank_biserial_CI_high': round(effect_high, 3),
        'mannwhitney_p': p_rank,
        'gap_at_cutoff': round(float(relative_gap), 4),
        'tracks_within_tolerance_of_cutoff': near_cut,
    })
    for cutoff in _cutoffs:
        sweep_rows.append({'track_universe': label, **hypergeometric(flags, order, cutoff)})

enrichment = pd.DataFrame(rows)
sweep = pd.DataFrame(sweep_rows)
enrichment.to_csv(OUTPUT_DIR / 'Tab_borzoi_adipose_enrichment.csv', index=False)
sweep.to_csv(OUTPUT_DIR / 'Tab_borzoi_adipose_enrichment_sweep.csv', index=False)
display(enrichment)
display(sweep.pivot(index='top_n', columns='track_universe',
                    values=['adipose_in_top_n', 'fold_enrichment', 'pvalue']))

# --- what can be quoted --------------------------------------------------------------------
_main = enrichment.iloc[-1]                       # the corrected universe
_sweep_main = sweep[sweep.track_universe == _main.track_universe]
_folds = _sweep_main.fold_enrichment
_worst_p = _sweep_main.pvalue.max()

print(f'\nAt the cut-off: {int(_main.adipose_in_top_n)} of the top {int(_main.top_n)} tracks '
      f'are adipose, fold {_main.fold_enrichment}, p = {_main.pvalue:.2g}.')
print(f'  The track at the cut-off leads the next one by {_main.gap_at_cutoff:.2%}, and '
      f'{int(_main.tracks_within_tolerance_of_cutoff)} tracks sit within '
      f'{TIE_TOLERANCE:.0%} of it.')
print(f'  Bootstrap interval for that fold enrichment: '
      f'{_main.fold_CI_low} to {_main.fold_CI_high}. A number whose interval is that wide '
      f'should not\n  be quoted to two decimal places, or compared between runs.')
print(f'\nAcross cut-offs {min(_cutoffs)} to {max(_cutoffs)}: fold enrichment '
      f'{_folds.min():.1f} to {_folds.max():.1f}, worst p = {_worst_p:.2g}.')
print(f'Without any cut-off: rank-biserial {_main.rank_biserial:+.3f} '
      f'(95% CI {_main.rank_biserial_CI_low:+.3f} to {_main.rank_biserial_CI_high:+.3f}), '
      f'Mann-Whitney p = {_main.mannwhitney_p:.2g}.')

_stable = (_main.rank_biserial_CI_low > 0) == (_main.rank_biserial_CI_high > 0)
print('\nSuggested wording — replaces a count that changes with a statistic that does not:')
if _stable:
    print(f'  "Adipose-related tracks respond more strongly than the rest across the whole\n'
          f'   ranking (rank-biserial {_main.rank_biserial:+.2f}, 95% CI '
          f'{_main.rank_biserial_CI_low:+.2f} to {_main.rank_biserial_CI_high:+.2f},\n'
          f'   Mann-Whitney p = {_main.mannwhitney_p:.1g}), and are enriched among the most '
          f'affected at every\n   cut-off tested from {min(_cutoffs)} to {max(_cutoffs)} '
          f'tracks (fold {_folds.min():.1f} to {_folds.max():.1f})."')
else:
    print('  The bootstrap interval for the effect size crosses zero in this run. Report that '
          'honestly:\n   the separation is not established on this measure. The exonic '
          'measures in the cell above\n   are the ones to check — see the stability table '
          'there.')

note(enrich_top_n=int(_main.top_n),
     enrich_adipose_in_top_n=int(_main.adipose_in_top_n),
     enrich_fold=float(_main.fold_enrichment),
     enrich_rank_biserial=float(_main.rank_biserial),
     enrich_mannwhitney_p=float(_main.mannwhitney_p),
     enrich_cutoff_gap=float(_main.gap_at_cutoff))

In [ ]:
# @title Save the figures { display-mode: "form" }
# @markdown Zips `borzoi_figures/` beside itself. In Colab it also downloads the archive,
# @markdown because Colab storage is wiped when the runtime disconnects; locally the figures
# @markdown are already on disk and the zip is only a convenience.
import shutil

archive = shutil.make_archive(str(Path(OUTPUT_DIR).parent / 'borzoi_figures'), 'zip',
                              OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path.name:48s} {path.stat().st_size / 1024:8.1f} KB')

try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print(f'\nFigures are in {OUTPUT_DIR}\nArchive: {archive}')

### End of Section A — handing off

Section B redefines several names that Section A also uses (`OUTPUT_DIR`, `MEASURES`,
`predictions`, `metrics`, `adipose`, `enrichment`, and others), because both notebooks were
written to stand alone. The cell below copies everything Section C needs into a single
`BORZOI` dictionary first, so nothing is lost when Section B rebinds those names.

Run it before continuing. If you skip it, Section C will tell you rather than silently
comparing AlphaGenome against itself.

In [ ]:
# @title A-final. Snapshot the Borzoi results for Sections B and C
from pathlib import Path

BORZOI = {
    # data and results
    'targets': targets,
    'rna_targets': rna_targets,
    'metrics': metrics,
    'track_effects': track_effects,
    'measures': dict(MEASURES),
    'measure_enrichment': measure_enrichment,
    'enrichment_by_universe': enrichment,
    'exon_mask': exon_mask,
    'region': {'start_bin': start_bin, 'n_bins': n_bins, 'region_bp': region_bp,
               'window': WINDOW_FOR_FIGURE},
    'rna_index': rna_index,
    # geometry and helpers, reused by the Section C sweep
    'windows': dict(WINDOWS),
    'variants': dict(VARIANTS),
    'gene': GENE,
    'gene_coordinates': GENE_COORDINATES,
    'leftmost': LEFTMOST,
    'context_length': BORZOI_CONTEXT_LENGTH,
    'output_length': OUTPUT_CONTEXT_LENGTH,
    'bin_size': BIN_SIZE,
    'crop': CROP,
    'adipose_pattern': ADIPOSE_PATTERN,
    'fold': FOLD,
    'drop_antisense': DROP_ANTISENSE,
    'build_sequences': build_sequences,
    'fetch_reference': fetch_reference,
    'one_hot': one_hot,
    'bin_of': bin_of,
    'output_dir': Path(OUTPUT_DIR),
}

# The raw arrays are deliberately not snapshotted: nothing downstream reads them, and at
# 187 MB each they are the largest thing Section A holds. Releasing them here gives Section B
# a clean runtime to work in. Section C re-runs the model rather than reusing predictions.
_freed = array_gb(*[a for w in predictions.values() for a in w.values()],
                  *baseline_predictions.values())
release('predictions', 'baseline_predictions', 'allele_set_predictions',
        'reference_sequences', 'sequences')
print(f'Released {_freed:.2f} GB of Section A prediction arrays.')
report_memory('Section A handed off')

print(f'Snapshotted {len(BORZOI)} objects from Section A.')
print(f'  {len(BORZOI["metrics"]):,} RNA tracks scored over '
      f'{BORZOI["region"]["region_bp"]:,} bp of {GENE}')
print(f'  median exonic log2 fold change: '
      f'{BORZOI["metrics"]["exon_log2_fold_change"].median():.3e} '
      f'(adipose {BORZOI["metrics"].loc[BORZOI["metrics"]["is_adipose"], "exon_log2_fold_change"].median():.3e})')
print('\nSection B may now redefine any of these names freely.')

# Everything Section A settled that could differ in another run, written where the figures
# are. Diffing two of these files is the fastest way to explain two different answers.
note(borzoi_fold=FOLD,
     borzoi_drop_antisense=DROP_ANTISENSE,
     borzoi_window=WINDOW_FOR_FIGURE,
     borzoi_variants=';'.join(sorted(VARIANTS)),
     borzoi_n_variants=len(VARIANTS),
     borzoi_rna_tracks=int(len(rna_targets)))
write_provenance()


---

# Section B — AlphaGenome

The AlphaGenome analysis of the same locus, unchanged, writing to `alphagenome_outputs/`. It
needs an API key: locally from `ALPHA_GENOME_API_KEY` in the environment or in a `.env` file
at the repository root, in Colab from a secret of that name. See *Running locally* at the top
for the full list of places the setup cell looks.

Two notes on running it after Section A rather than alone:

- The setup cell rebinds `OUTPUT_DIR`, so Section B's figures land in its own folder while
  Section A's stay where they were written. Section C reads both.
- Names shared with Section A (`MEASURES`, `predictions`, `metrics`, `adipose`, `enrichment`)
  are now Section B's. Section A's copies live in `BORZOI`.

## 1. Setup

We load the model client once and reuse it throughout. The key comes from the first of these
that has one, so nothing needs configuring twice:

1. `ALPHA_GENOME_API_KEY` (or `ALPHAGENOME_API_KEY`) in the environment. VS Code puts the
   contents of `${workspaceFolder}/.env` there when the kernel starts.
2. A `.env` file at the repository root or beside the notebook, read directly — so a key
   added after the kernel started works without restarting it.
3. `~/.alphagenome_api_key`, a file containing just the key.
4. A Colab secret named `ALPHA_GENOME_API_KEY` (key icon in the left sidebar, with "Notebook
   access" enabled).
5. Failing all of those, a `getpass` prompt. What you type is held in memory for this session
   and written nowhere.

The key itself is never printed — only which of the five it came from. Keys go in `.env`,
which is git-ignored; never in a cell.

This cell also switches on Colab's interactive dataframe viewer, so the score tables below can
be sorted and filtered in place rather than printed as static text. Outside Colab that is a
no-op and the tables print as usual.

In [ ]:
# @title Setup: imports, API key and model client
import io
import os
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

try:
    from alphagenome.data import gene_annotation, genome
    from alphagenome.data import transcript as transcript_utils
    from alphagenome.interpretation import ism
    from alphagenome.models import dna_client, variant_scorers
    from alphagenome.visualization import plot_components
except ImportError as error:
    # Importing alphagenome pulls in typing_extensions, typeguard and jaxtyping. If the
    # install cell upgraded any of them after this session had already imported an older
    # copy — which Colab and a long-lived kernel both do — the interpreter keeps the old
    # module in memory and the new one fails against it, most often as
    # `cannot import name 'NoExtraItems' from 'typing_extensions'`. Nothing is wrong with
    # the installation; the session is simply older than it is.
    raise ImportError(
        f'{error}\n\n  alphagenome could not be imported. If the message above names a '
        'symbol that does exist\n  in the installed version, this session imported an '
        'older copy before it was upgraded:\n  restart the kernel and run from the top. '
        'Otherwise run the install cell at the top first.') from error

API_KEY_NAMES = ('ALPHA_GENOME_API_KEY', 'ALPHAGENOME_API_KEY')


def _read_dotenv(path: Path) -> dict:
    """The KEY=value pairs in a .env file. Quotes and a leading `export` are tolerated."""
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.removeprefix('export ').partition('=')
        values[key.strip()] = value.strip().strip('\'"')
    return values


def _alphagenome_api_key():
    """The API key and where it came from. Never returns the key by printing it."""
    # 1. The environment. VS Code loads ${workspaceFolder}/.env into the kernel at startup,
    #    so a key placed there before the kernel started arrives here.
    for name in API_KEY_NAMES:
        value = os.environ.get(name, '').strip()
        if value:
            return value, f'the environment variable {name}'

    # 2. A .env file read directly, which also catches one written after the kernel started.
    notebook_dir = Path(globals().get('NOTEBOOK_DIR', Path.cwd()))
    searched = [notebook_dir, *notebook_dir.parents][:8]
    if 'REPO_ROOT' in globals() and REPO_ROOT:
        searched.insert(0, Path(REPO_ROOT))
    for folder in dict.fromkeys(searched):
        env_file = folder / '.env'
        if not env_file.is_file():
            continue
        values = _read_dotenv(env_file)
        for name in API_KEY_NAMES:
            if values.get(name):
                return values[name], f'{name} in {env_file}'

    # 3. A key file in the home folder, for working across several checkouts.
    key_file = Path.home() / '.alphagenome_api_key'
    if key_file.is_file() and key_file.read_text().strip():
        return key_file.read_text().strip(), str(key_file)

    # 4. Colab secrets.
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata

            value = (userdata.get('ALPHA_GENOME_API_KEY') or '').strip()
            if value:
                return value, 'the Colab secret ALPHA_GENOME_API_KEY'
        except Exception as error:  # noqa: BLE001  (secret absent, or access not granted)
            print(f'Colab secret not readable ({type(error).__name__}).')

    # 5. Ask. Held in memory for this session only.
    from getpass import getpass

    value = getpass('AlphaGenome API key (not echoed, not saved): ').strip()
    if not value:
        raise RuntimeError(
            'No AlphaGenome API key. Put a line reading ALPHA_GENOME_API_KEY=your-key in a '
            '.env file\n  at the repository root, or export it before starting the kernel. '
            'Keys are issued at\n  https://deepmind.google.com/science/alphagenome')
    return value, 'the prompt (this session only)'


_api_key, _api_key_source = _alphagenome_api_key()
dna_model = dna_client.create(_api_key)
del _api_key                       # keep it out of the namespace the rest of the cells see

# Interactive, filterable dataframes in Colab (no-op elsewhere).
try:
    from google.colab import data_table

    data_table.enable_dataframe_formatter()
except ImportError:
    pass

# The predictions come from a server, so the client version is the only handle this
# notebook has on which model answered. Recorded for the same reason the Borzoi fold is.
try:
    import importlib.metadata as _meta

    _alphagenome_version = _meta.version('alphagenome')
except Exception:  # noqa: BLE001
    _alphagenome_version = 'unknown'
if 'note' in globals():
    note(alphagenome_client=_alphagenome_version, alphagenome_key_source=_api_key_source)

print(f'Model client ready (alphagenome {_alphagenome_version}), key from '
      f'{_api_key_source}.')

## 2. Configuration

Everything that defines the analysis lives in the next two cells. The first holds the data
being studied — genes, variants, tissue/cell-type ontology terms — and is meant to be edited
as code. The second exposes the run-time knobs as Colab form controls, so you can change the
context length, the ISM window or which sections run without touching code.

The eight ontology terms are the adipose and mesenchymal contexts relevant to insulin
resistance; they are the same terms listed in supplementary table ST8.

In [ ]:
# @title Genes, variants and tissue ontology terms
# --- Genes -----------------------------------------------------------------
GENES = ['ANKRD55', 'MAP3K1']   # Locus interval must span both.
FOCAL_GENE = 'MAP3K1'           # Gene used for gene-level (RNA-seq) scoring.

# --- Variants of interest (hg38) -------------------------------------------
# Built from CANONICAL_VARIANTS (shared cell near the top) rather than written out again,
# so Section B cannot use a different allele from Section A. Order is preserved.
VARIANT_TABLE = '\n'.join(
    ['rsid,CHROM,POS,REF,ALT']
    + [f'{rsid},{c},{p},{r},{a}' for rsid, (c, p, r, a) in CANONICAL_VARIANTS.items()]
)
FOCAL_VARIANT = 'rs256903'  # Used for the single-variant REF/ALT figure.
# Section B scores every variant in CANONICAL_VARIANTS individually, so both
# credible sets are covered without any further change here.

# --- Tissue / cell-type ontology terms -------------------------------------
ONTOLOGY_TERMS = {
    'UBERON:0001013': 'Adipose tissue',
    'UBERON:0014455': 'Subcutaneous abdominal adipose',
    'UBERON:0015143': 'Mesenteric fat',
    'UBERON:0010414': 'Omental fat',
    'CL:0002615': 'Adipocyte of omental tissue',
    'CL:0002617': 'Breast adipocyte',
    'CL:0002579': 'Omentum preadipocyte',
    'CL:0002570': 'Mesenchymal stem cell of adipose',
}
PRIMARY_ONTOLOGY = 'UBERON:0001013'  # Single term for summary comparisons.
# Tissues shown side by side in the section 9 background comparison. Terms with no
# RNA-seq track are reported and skipped rather than raising.
COMPARISON_ONTOLOGIES = ['UBERON:0001013', 'CL:0002570']
# Genes scored in the section 9 background comparison. Both must fall inside the
# variant-centred window; SETD9 sits ~250 kb upstream of the variants, so it does.
COMPARISON_GENES = ['MAP3K1', 'SETD9']

# --- Derived objects -------------------------------------------------------
variants_df = pd.read_csv(io.StringIO(VARIANT_TABLE))
VARIANTS = [
    genome.Variant(
        chromosome=row.CHROM,
        position=int(row.POS),
        reference_bases=row.REF,
        alternate_bases=row.ALT,
        name=row.rsid,
    )
    for row in variants_df.itertuples()
]
VARIANT_BY_RSID = {v.name: v for v in VARIANTS}
assert_canonical(
    {v.name: (v.chromosome, v.position, v.reference_bases, v.alternate_bases)
     for v in VARIANTS},
    'Section B (AlphaGenome)',
)
RSID_BY_ID = {str(v): v.name for v in VARIANTS}  # 'chr5:56513311:C>A' -> 'rs256903'

print(f'{len(VARIANTS)} variants, {len(ONTOLOGY_TERMS)} ontology terms.')
variants_df

In [ ]:
# @title Analysis settings { display-mode: "form", run: "auto" }

# @markdown **Model context.** Length of the input sequence around each variant.
# @markdown 1MB is required for section 4 (both genes in one window) and for the
# @markdown gene-level MAP3K1 scorer used on the upstream variants.
sequence_length = "1MB"  # @param ["2KB", "16KB", "100KB", "500KB", "1MB"]

# @markdown **Focal variant** for the single-variant REF/ALT figure (section 6). Any rsID
# @markdown in CANONICAL_VARIANTS is valid; the listed ones are the five the manuscript
# @markdown reports.
FOCAL_VARIANT = "rs256903"  # @param ["rs459193", "rs256903", "rs173964", "rs256904", "rs3843467"]
assert FOCAL_VARIANT in CANONICAL_VARIANTS, (
    f'{FOCAL_VARIANT} is not in this run. Available: {sorted(CANONICAL_VARIANTS)}')

# @markdown ---
# @markdown **In silico mutagenesis (section 8).** Cost scales as
# @markdown `3 x width x 2 backgrounds` per variant, so start small.
RUN_ISM = True  # @param {type:"boolean"}
# @markdown Variants mutagenised. `main_five` matches the supplementary figure it feeds;
# @markdown `all` covers every credible-set variant and costs proportionally more.
ISM_SCOPE = "main_five"  # @param ["main_five", "focal_only", "all"]
ISM_WIDTH = 256  # @param {type:"slider", min:32, max:1024, step:32}
ISM_SCORER_KEY = "RNA_SEQ"  # @param ["RNA_SEQ", "ATAC", "CHIP_TF"]
# @markdown How to collapse the several tracks an ontology term may have into one
# @markdown number. This is per term — terms are never averaged together.
ISM_TRACK_AGG = "mean"  # @param ["mean", "max"]
# @markdown Put each tissue's REF and ALT logos on one symmetric y-axis. Turn this
# @markdown off to autoscale every panel independently, as the original code did.
ISM_SHARED_YLIM = True  # @param {type:"boolean"}

# @markdown ---
# @markdown **Background comparison (section 9)** and parallelism. The comparison scores
# @markdown every alternative allele at each variant's position, so it too defaults to the
# @markdown five the manuscript figures report.
RUN_BACKGROUND = True  # @param {type:"boolean"}
BACKGROUND_SCOPE = "main_five"  # @param ["main_five", "all"]
MAX_WORKERS = 4  # @param {type:"slider", min:1, max:8, step:1}

# --- Derived settings ------------------------------------------------------
SEQUENCE_LENGTH = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]
ISM_VARIANTS = {
    'focal_only': [FOCAL_VARIANT],
    'main_five': [r for r in MANUSCRIPT_RSIDS if r in CANONICAL_VARIANTS],
    'all': None,
}[ISM_SCOPE]
TRACK_AGG = {'mean': np.nanmean, 'max': np.nanmax}[ISM_TRACK_AGG]

OUTPUT_DIR = Path(globals().get('RESULTS_ROOT', Path.cwd())) / 'alphagenome_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

n_ism = len(ISM_VARIANTS or VARIANTS) * 2 * 3 * ISM_WIDTH if RUN_ISM else 0
print(f'Context: {SEQUENCE_LENGTH:,} bp | focal variant: {FOCAL_VARIANT}')
print(f'ISM: {"on" if RUN_ISM else "off"}, '
      f'{len(ISM_VARIANTS or VARIANTS)} variant(s) '
      f'({ISM_WIDTH} bp, {ISM_SCORER_KEY}, tracks per term -> {ISM_TRACK_AGG}) '
      f'-> ~{n_ism:,} scored variants')
print(f'Outputs -> {OUTPUT_DIR.resolve()}')

### Output naming

Every analysis below is computed once over all 25 credible-set variants and reported twice:
at the **main** scope, the four fine-mapped variants plus rs3843467, which is what the
main-text figures show, and at the **supplementary** scope, all of them. Because the variant
scorers work one variant at a time, the main-text numbers are a strict subset of the same
run — nothing is scored twice, and the two cannot disagree.

Files are written into a subfolder per scope and carry the scope in their name, so the two
versions of a figure never overwrite one another. A manifest records every file, its scope,
and which manuscript item it feeds.

In [ ]:
# @title Output naming { display-mode: "form" }
# Every file produced below is named for what it contains and which scope it belongs to,
# so the main-text and supplementary versions of the same analysis never collide.
#
#   scoped_path('variant_prioritisation', 'main_5variants', '.pdf')
#     -> <outputs>/main_5variants/Fig_variant_prioritisation__main_5variants.pdf
#
# The scope subfolder means a whole set can be handed over as one directory.
from pathlib import Path

FIGURE_PREFIX = {'figure': 'Fig', 'supp_figure': 'SuppFig', 'table': 'Tab',
                 'diagnostic': 'Diag'}

MANIFEST = []


def scoped_path(name, scope, suffix, kind='figure', manuscript_item=None,
                description=None):
    """Path for an output, and a manifest row recording what it feeds."""
    folder = Path(OUTPUT_DIR) / scope
    folder.mkdir(parents=True, exist_ok=True)
    prefix = FIGURE_PREFIX.get(kind, 'Out')
    path = folder / f'{prefix}_{name}__{scope}{suffix}'
    MANIFEST.append({'file': str(path.relative_to(OUTPUT_DIR)), 'scope': scope,
                     'kind': kind, 'manuscript_item': manuscript_item or '',
                     'description': description or name})
    return path


def write_manifest():
    """One CSV listing every file produced, its scope and where it belongs."""
    import pandas as pd

    if not MANIFEST:
        print('nothing recorded yet')
        return None
    table = pd.DataFrame(MANIFEST).drop_duplicates(subset='file')
    path = Path(OUTPUT_DIR) / 'output_manifest.csv'
    table.to_csv(path, index=False)
    print(f'{len(table)} output(s) recorded in {path}')
    return table


print(f'outputs -> {OUTPUT_DIR}')
print(f'  {MAIN_SCOPE_NAME}/  main-text figures and tables')
print(f'  {SUPP_SCOPE_NAME}/  the same analyses over every credible-set variant')


In [ ]:
# @title Plot helpers: colour-coded variant annotation
# Four of the five variants sit within 3.6 kb of each other, so text labels overlap at
# any zoom level wide enough to show the locus. We colour each variant's line instead
# and identify them in a figure legend.
PALETTE = [
    '#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
    '#ff7f00', '#a65628', '#f781bf', '#666666',
]
VARIANT_COLORS = {
    v.name: PALETTE[i % len(PALETTE)] for i, v in enumerate(VARIANTS)
}


def variant_annotation(variants=None, alpha: float = 0.9, show_labels: bool = False):
    # Labels are off by default: passing labels=None with use_default_labels=False
    # suppresses them entirely, rather than drawing empty strings that still
    # reserve vertical space above the axes.
    variants = list(VARIANTS if variants is None else variants)
    return plot_components.VariantAnnotation(
        variants,
        colors=[VARIANT_COLORS[v.name] for v in variants],
        alpha=alpha,
        labels=[v.name for v in variants] if show_labels else None,
        use_default_labels=False,
    )


def add_variant_legend(fig, variants=None, loc: str = 'upper right', ncol: int = 1):
    variants = list(VARIANTS if variants is None else variants)
    handles = [
        Line2D(
            [0], [0], color=VARIANT_COLORS[v.name], lw=4,
            label=(
                f'{v.name}   {v.position:,}  '
                f'{v.reference_bases}>{v.alternate_bases}'
            ),
        )
        for v in variants
    ]
    fig.legend(
        handles=handles, loc=loc, ncol=ncol, fontsize=9, framealpha=0.9,
        bbox_to_anchor=(0.995, 0.995) if loc == 'upper right' else None,
    )
    return fig


for name, color in VARIANT_COLORS.items():
    print(f'{name:12s} {color}')

## 3. Gene annotation

Transcript models come from GENCODE v46 (hg38). We keep protein-coding genes with the
highest transcript support level, and build two extractors: one returning all supported
transcripts, one returning only the longest transcript per gene (used for the wide locus
plots, where full transcript sets would be unreadable).

The loader falls back to a column-wise Arrow conversion if `pd.read_feather` raises, so a
pandas/pyarrow mismatch degrades to a warning rather than stopping the notebook.

In [ ]:
# @title Load GENCODE gene annotations
GTF_URL = (
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)


def read_feather_robust(url: str) -> pd.DataFrame:
    '''Reads a Feather file, working around mismatched pandas/pyarrow pairs.

    pandas hands the whole Arrow table to pyarrow's block-based converter, which is
    the piece that breaks when the two packages disagree about the pandas major
    version. Converting one column at a time never touches that code path.
    '''
    try:
        return pd.read_feather(url)
    except Exception as exc:  # noqa: BLE001
        print(f'pd.read_feather failed ({exc!r}); falling back to column-wise read.')
        import fsspec
        import pyarrow.feather as feather

        with fsspec.open(url) as handle:
            table = feather.read_table(handle)
        return pd.DataFrame(
            {name: table.column(name).to_pandas() for name in table.column_names}
        )


gtf = read_feather_robust(GTF_URL)

gtf_transcript = gene_annotation.filter_transcript_support_level(
    gene_annotation.filter_protein_coding(gtf), ['1']
)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcript)

gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(gtf_transcript)
longest_transcript_extractor = transcript_utils.TranscriptExtractor(gtf_longest_transcript)

print(f'Loaded {len(gtf):,} GTF rows.')

## 4. Locus definition

The analysis window is the **widest model context (1 Mb)** centred so that it contains both
*ANKRD55* and *MAP3K1*. The two genes span roughly 800 kb, so a single 1 Mb input sequence
covers the whole locus including all five variants — meaning one prediction gives us both
genes in the same coordinate frame.

Note that a *variant-centred* 1 Mb window (used from section 7 onwards) is not the same
interval as this locus window; scoring is always done in a window centred on the variant,
which is what the ST8 `scored_interval` column records.

In [ ]:
# @title Define the locus interval spanning both genes
# Union of the two gene bodies, expanded to the model's input length.
gene_intervals = {g: gene_annotation.get_gene_interval(gtf, gene_symbol=g) for g in GENES}
for name, iv in gene_intervals.items():
    print(f'{name:10s} {iv}  ({iv.width:,} bp, strand {iv.strand})')

span = genome.Interval(
    chromosome=next(iter(gene_intervals.values())).chromosome,
    start=min(iv.start for iv in gene_intervals.values()),
    end=max(iv.end for iv in gene_intervals.values()),
)
LOCUS_INTERVAL = span.resize(SEQUENCE_LENGTH)

print(f'\nGene span : {span} ({span.width:,} bp)')
print(f'Locus     : {LOCUS_INTERVAL} ({LOCUS_INTERVAL.width:,} bp)')

# Sanity checks: both genes and all variants should fit inside the locus window.
outside = [n for n, iv in gene_intervals.items() if not LOCUS_INTERVAL.contains(iv)]
if outside:
    print(f'\nWARNING: {", ".join(outside)} do not fit in a {SEQUENCE_LENGTH:,} bp window. '
          'Set sequence_length to 1MB in the settings form for sections 4-5.')
else:
    print('\nBoth genes are contained in the locus interval.')
for v in VARIANTS:
    assert LOCUS_INTERVAL.contains(v.reference_interval), f'{v.name} outside the window.'

In [ ]:
# @title Figure 1: variant positions across the locus
locus_transcripts = longest_transcript_extractor.extract(LOCUS_INTERVAL)

fig = plot_components.plot(
    [plot_components.TranscriptAnnotation(locus_transcripts)],
    annotations=[variant_annotation()],
    interval=LOCUS_INTERVAL,
    title=f'Variants of interest across the {" / ".join(GENES)} locus',
)
add_variant_legend(fig)
fig.savefig(OUTPUT_DIR / 'Diag_locus_overview.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Predicted RNA-seq across the locus

Reference-genome predictions (no variant) for the eight adipose/mesenchymal ontology terms,
over the full 1 Mb locus. This establishes the baseline: which of the two genes AlphaGenome
expects to be expressed in adipose contexts, and on which strand.

RNA-seq tracks are stranded, so each ontology term contributes at least one track per strand;
genes on the minus strand only show signal in the negative-strand tracks. Some terms map to
several experiments, so the figure can get tall — drop entries from `ONTOLOGY_TERMS` if so.

In [ ]:

# @title Predict RNA-seq across the locus
locus_output = dna_model.predict_interval(
    interval=LOCUS_INTERVAL,
    requested_outputs=[dna_client.OutputType.RNA_SEQ],
    ontology_terms=list(ONTOLOGY_TERMS),
)

rna = locus_output.rna_seq
print(f'RNA-seq predictions: {rna.values.shape[0]:,} positions x {rna.values.shape[1]} tracks')
missing = set(ONTOLOGY_TERMS) - set(rna.metadata['ontology_curie'])
if missing:
    print('Terms without RNA-seq tracks:', ', '.join(sorted(missing)))
rna.metadata[['name', 'ontology_curie', 'biosample_name', 'strand']]

In [ ]:
# @title Figure 2: predicted RNA-seq across the locus
fig = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(locus_transcripts),
        plot_components.Tracks(
            tdata=rna,
            ylabel_template='{biosample_name} ({strand})',
            filled=True,
            max_num_tracks=100,
        ),
    ],
    annotations=[variant_annotation(alpha=0.6)],
    interval=LOCUS_INTERVAL,
    title=(
        f'Predicted RNA-seq across the {" / ".join(GENES)} locus '
        '(adipose and mesenchymal tracks)'
    ),
)
add_variant_legend(fig)
fig.savefig(OUTPUT_DIR / 'SuppFig_locus_expression_landscape.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. REF vs ALT predictions for a focal variant

`predict_variant` runs the model twice on the same interval — once with the reference base and
once with the alternate — so the two tracks are directly comparable. We overlay them for
RNA-seq and ATAC around the focal variant. This is the visual counterpart of the ATAC scores
tabulated in section 7.2.

In [ ]:
# @title Figure 3: REF vs ALT tracks for the focal variant
focal = VARIANT_BY_RSID[FOCAL_VARIANT]
focal_interval = focal.reference_interval.resize(SEQUENCE_LENGTH)

focal_output = dna_model.predict_variant(
    interval=focal_interval,
    variant=focal,
    requested_outputs=[dna_client.OutputType.RNA_SEQ, dna_client.OutputType.ATAC],
    ontology_terms=list(ONTOLOGY_TERMS),
)

ref_alt_colors = {'REF': 'dimgrey', 'ALT': 'red'}
zoom = focal.reference_interval.resize(2**15)  # ~32 kb around the variant.

fig = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcript_extractor.extract(zoom)),
        plot_components.OverlaidTracks(
            tdata={
                'REF': focal_output.reference.rna_seq,
                'ALT': focal_output.alternate.rna_seq,
            },
            colors=ref_alt_colors,
            ylabel_template='RNA-seq: {biosample_name} ({strand})',
            max_num_tracks=100,
        ),
        plot_components.OverlaidTracks(
            tdata={
                'REF': focal_output.reference.atac,
                'ALT': focal_output.alternate.atac,
            },
            colors=ref_alt_colors,
            ylabel_template='ATAC: {biosample_name}',
            max_num_tracks=100,
        ),
    ],
    annotations=[variant_annotation([focal])],
    interval=zoom,
    title=f'REF vs ALT predictions around {focal.name} ({focal})',
)
add_variant_legend(fig, [focal])
fig.savefig(OUTPUT_DIR / f'Diag_ref_alt_profiles_{focal.name}.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Batch variant scoring — supplementary table ST8

Instead of inspecting tracks by eye, `score_variants` reduces each variant to one number per
track. We use the two recommended `CenterMaskScorer` configurations that ST8 is built from:

- **ATAC** — `CenterMaskScorer(ATAC, width=501, DIFF_LOG2_SUM)`
- **CHIP_TF** — `CenterMaskScorer(CHIP_TF, width=501, DIFF_LOG2_SUM)`

Both aggregate the log2 ALT − REF difference over a 501 bp window centred on the variant, in a
variant-centred 1 Mb input sequence. Because these are not gene-centric scorers, every track in
the model is scored — including tissues outside our ontology list — which is exactly what the
P300 and KLF/SP tables need.

`tidy_scores` flattens the resulting `AnnData` objects into one row per (variant, track, scorer).
It returns the `variant_id` and `scored_interval` columns as `Variant` and `Interval` objects
rather than strings, so we convert them once before use — `Variant` is an unhashable dataclass
and any pandas operation that hashes the column would otherwise fail.
The `raw_score` is the aggregated difference; `quantile_score` is that value's rank within the
distribution of scores for a background set of common variants, so it is comparable across
scorers and tracks.

In [ ]:
# @title Score every variant with the ATAC and ChIP-TF scorers
SELECTED_SCORERS = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['ATAC'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['CHIP_TF'],
]
for s in SELECTED_SCORERS:
    print(s)

scoring_intervals = [v.reference_interval.resize(SEQUENCE_LENGTH) for v in VARIANTS]

raw_scores = dna_model.score_variants(
    intervals=scoring_intervals,
    variants=VARIANTS,
    variant_scorers=SELECTED_SCORERS,
    max_workers=MAX_WORKERS,
)

df_scores = variant_scorers.tidy_scores(raw_scores)

# tidy_scores leaves Variant and Interval objects in these two columns rather than
# strings. Variant is an unhashable dataclass, so anything that hashes the column
# (.map, .groupby, .merge, drop_duplicates) raises TypeError; converting once here
# also makes the CSVs match the 'chr5:56513311:C>A' formatting used in ST8.
df_scores['variant_id'] = df_scores['variant_id'].astype(str)
df_scores['scored_interval'] = df_scores['scored_interval'].astype(str)

df_scores['rsid'] = df_scores['variant_id'].map(RSID_BY_ID)

# Quantile scores are attached only when the scorer provides them; fall back to the
# raw score for sorting so the tables below work either way.
SORT_KEY = 'quantile_score' if 'quantile_score' in df_scores.columns else 'raw_score'
if SORT_KEY == 'raw_score':
    print('No quantile scores returned — tables will be sorted by raw_score.')
    BASE_SCORE_COLUMNS = ['raw_score']
else:
    BASE_SCORE_COLUMNS = ['raw_score', 'quantile_score']
df_scores.to_csv(scoped_path('ST8_all_variant_scores', SUPP_SCOPE_NAME, '.csv',
                             kind='table',
                             manuscript_item='Supplementary Table 8 (source data)',
                             description='Every variant effect score returned by the ATAC '
                                         'and ChIP-TF scorers'), index=False)

print(f'\n{len(df_scores):,} rows: {df_scores.output_type.value_counts().to_dict()}')
df_scores.head()

### 7.1 P300 binding differences across all tissues

EP300 is a co-activator that marks active enhancers, so a predicted loss of EP300 binding is
evidence that a variant disrupts enhancer activity. AlphaGenome only has EP300 ChIP-seq tracks
for cell lines and tissues outside our adipose list, so this table deliberately looks **beyond
the ontology terms** and reports every EP300 track. Rows are sorted by `quantile_score`, most
negative (strongest predicted loss of binding) first.

In [ ]:
# @title ST8a: P300 binding differences
BASE_COLUMNS = [
    'variant_id', 'rsid', 'scored_interval', 'output_type', 'variant_scorer',
    'track_name', 'track_strand', 'Assay title', 'biosample_name', 'biosample_type',
]
TF_COLUMNS = BASE_COLUMNS + ['transcription_factor'] + BASE_SCORE_COLUMNS
ATAC_COLUMNS = BASE_COLUMNS + ['ontology_curie', 'ontology_label'] + BASE_SCORE_COLUMNS

def _write_scoped(table, name, item_main, item_supp, description):
    """Write the same table at both scopes: the main-text subset and everything."""
    for scope, rsids in SCOPES.items():
        subset = table[table['rsid'].isin(rsids)]
        item = item_main if scope == MAIN_SCOPE_NAME else item_supp
        path = scoped_path(name, scope, '.csv', kind='table',
                           manuscript_item=item, description=description)
        subset.to_csv(path, index=False)
        print(f'  {scope}: {len(subset):5d} rows -> {path.name}')


p300_table = (
    df_scores[
        (df_scores['output_type'] == 'CHIP_TF')
        & (df_scores['transcription_factor'] == 'EP300')
    ]
    .sort_values(SORT_KEY)
    .loc[:, TF_COLUMNS]
    .reset_index(drop=True)
)
_write_scoped(p300_table, 'ST8a_EP300_binding_differences',
              'Supplementary Table 8a', 'Supplementary Table 8a (all variants)',
              'EP300 ChIP-seq differences per variant and track')

print(f'{len(p300_table)} EP300 track scores across {p300_table.rsid.nunique()} variants.')
p300_table.head(20)

### 7.2 ATAC-seq differences in the adipose subset

Here we do the opposite: keep a single output type (chromatin accessibility) and restrict to
the adipose and mesenchymal ontology terms defined in section 2. A negative score means the
alternate allele is predicted to *close* chromatin in that tissue.

In [ ]:
# @title ST8b: ATAC differences in the adipose subset
atac_table = (
    df_scores[
        (df_scores['output_type'] == 'ATAC')
        & (df_scores['ontology_curie'].isin(ONTOLOGY_TERMS))
    ]
    .assign(ontology_label=lambda d: d['ontology_curie'].map(ONTOLOGY_TERMS))
    .sort_values(SORT_KEY)
    .loc[:, ATAC_COLUMNS]
    .reset_index(drop=True)
)
_write_scoped(atac_table, 'ST8b_ATAC_adipose_differences',
              'Supplementary Table 8b', 'Supplementary Table 8b (all variants)',
              'ATAC differences in adipose depots per variant and track')

print(f'{len(atac_table)} adipose ATAC track scores.')
atac_table

### 7.3 KLF and SP binding differences

The KLF/SP family binds GC-rich motifs and includes several regulators of adipocyte
differentiation and insulin sensitivity, which makes them the mechanistically interesting
factors at this locus. As with P300, we scan **all** ChIP-TF tracks rather than the adipose
subset, because these factors are profiled in cell lines.

The pattern matches `SP` followed by a single digit (SP1–SP9) or `KLF` followed by digits,
which avoids unrelated genes such as *SPI1* or *SP110*.

In [ ]:
# @title ST8c: KLF and SP binding differences
KLF_SP_PATTERN = r'^(SP\d|KLF\d+)$'

klf_sp_table = (
    df_scores[
        (df_scores['output_type'] == 'CHIP_TF')
        & (df_scores['transcription_factor'].str.match(KLF_SP_PATTERN, na=False))
    ]
    .sort_values(SORT_KEY)
    .loc[:, TF_COLUMNS]
    .reset_index(drop=True)
)
_write_scoped(klf_sp_table, 'ST8c_KLF_SP_binding_differences',
              'Supplementary Table 8c', 'Supplementary Table 8c (all variants)',
              'SP and KLF ChIP-seq differences per variant and track')

print(
    f'{len(klf_sp_table)} track scores for '
    f'{klf_sp_table.transcription_factor.nunique()} factors: '
    + ', '.join(sorted(klf_sp_table.transcription_factor.unique()))
)
klf_sp_table.head(20)

### 7.4 Figure 3b — variant prioritisation across the three readouts

The three tables above each say the same thing about rs256903, but only once you read all three.
This panel puts them on one axis. Scores are quantiles of the genome-wide score distribution, so
the ATAC, EP300 and SP/KLF columns are comparable despite measuring different things, and negative
means the alternate allele closes chromatin or reduces predicted binding.


In [ ]:
# @title Figure 3b: variant prioritisation across ATAC, EP300 and SP/KLF { display-mode: "form" }
ADIPOSE_TISSUE_NOTE = 'ATAC-seq, adipose depots'
FOCAL_VARIANT = 'rs256903'
FOCAL_COLOUR = '#d62728'
OTHER_COLOUR = '#8c8c8c'

panel_specs = [
    (ADIPOSE_TISSUE_NOTE, atac_table),
    ('EP300 ChIP-seq, all tracks', p300_table),
    ('SP / KLF ChIP-seq, all tracks', klf_sp_table),
]
def draw_prioritisation(rsid_order, scope, kind, manuscript_item):
    """The panel, drawn for whichever variants the scope contains.

    Height scales with the number of rows so the 5-variant and 25-variant versions are
    both legible; the underlying scores are identical.
    """
    row_of = {rsid: i for i, rsid in enumerate(rsid_order[::-1])}
    jitter = np.random.default_rng(0)
    height = max(3.2, 0.42 * len(rsid_order) + 1.4)

    with plt.rc_context({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
                         'xtick.labelsize': 10, 'ytick.labelsize': 10}):
        fig, axes = plt.subplots(1, len(panel_specs), figsize=(11.2, height), sharey=True)

        for ax, (title, table) in zip(axes, panel_specs):
            for rsid in rsid_order:
                values = table.loc[table['rsid'] == rsid, 'quantile_score'].to_numpy()
                if len(values) == 0:
                    continue
                colour = FOCAL_COLOUR if rsid == FOCAL_VARIANT else OTHER_COLOUR
                y = row_of[rsid] + jitter.uniform(-0.16, 0.16, len(values))
                ax.scatter(values, y, s=26, facecolor=colour, edgecolor='white',
                           linewidth=0.5, alpha=0.9, zorder=3)
                ax.plot([np.median(values)] * 2,
                        [row_of[rsid] - 0.30, row_of[rsid] + 0.30],
                        color=colour, linewidth=2.0, zorder=4)
            ax.axvline(0, color='0.35', linestyle=':', linewidth=0.9, zorder=1)
            for row in range(len(rsid_order)):
                ax.axhline(row, color='0.92', linewidth=0.7, zorder=0)
            ax.set_xlim(-1.08, 1.08)
            ax.set_xticks([-1, -0.5, 0, 0.5, 1])
            ax.set_title(title)
            ax.set_xlabel('Quantile score (ALT - REF)')
            ax.annotate('closes / loses binding', xy=(-1.0, -0.45), fontsize=7.5,
                        color='0.4', ha='left', va='center')
            ax.annotate('opens / gains binding', xy=(1.0, -0.45), fontsize=7.5,
                        color='0.4', ha='right', va='center')
            for spine in ('top', 'right'):
                ax.spines[spine].set_visible(False)

        axes[0].set_yticks(range(len(rsid_order)))
        axes[0].set_yticklabels(rsid_order[::-1])
        axes[0].set_ylim(-0.85, len(rsid_order) - 0.4)
        fig.tight_layout()
        for _suffix, _kwargs in (('.png', {'dpi': 400}), ('.pdf', {})):
            fig.savefig(
                scoped_path('variant_prioritisation', scope, _suffix, kind=kind,
                            manuscript_item=manuscript_item,
                            description='Variant effect scores across ATAC, EP300 and '
                                        'SP/KLF, one row per variant'),
                bbox_inches='tight', **_kwargs)
    plt.show()
    return fig


# Main text shows the five; the supplement shows every credible-set variant. Same scores.
draw_prioritisation(SCOPES[MAIN_SCOPE_NAME], MAIN_SCOPE_NAME, 'figure', 'Figure 3b')
draw_prioritisation(SCOPES[SUPP_SCOPE_NAME], SUPP_SCOPE_NAME, 'supp_figure',
                    'Supplementary Figure (variant prioritisation, all variants)')


for _scope, _rsids in SCOPES.items():
    print(f'\n[{_scope}] variants with a negative median quantile score:')
    for title, table in panel_specs:
        medians = (table[table['rsid'].isin(_rsids)]
                   .groupby('rsid')['quantile_score'].median())
        negative = medians[medians < 0].sort_values()
        print(f'  {title}: {", ".join(negative.index) if len(negative) else "none"}')


## 8. *In silico* mutagenesis on REF and ALT backgrounds

ISM asks which *positions* around a variant matter, by scoring every possible single-nucleotide
substitution in a window and plotting the result as a sequence logo. Tall letters mark
positions where changing the base changes the prediction — typically transcription factor motifs.

We run it twice per variant:

- **REF background** (`interval_variant=None`) — the unmodified reference sequence.
- **ALT background** (`interval_variant=variant`) — the alternate allele is written into the
  sequence first, then every position is mutated on top of it.

Comparing the two answers a question a single ISM run cannot: does the variant *create* or
*destroy* the motif around it? If a motif is visible on the REF background but flat on the ALT
background, the alternate allele has disrupted it.

**Which tracks does this use?** `score_ism_variants` takes no `ontology_terms` argument, so the
model scores *every* track of the requested output type — all RNA-seq tracks, for all genes in
the window. The scoring pass is therefore tissue-independent, and it is only the *reading* of
those scores that is tissue-specific: `track_scalar` selects the rows for `FOCAL_GENE` and the
columns whose `ontology_curie` matches one term, then collapses whatever tracks remain with
`TRACK_AGG` (mean by default). Note this collapses the tracks *within* one term; the terms
themselves are never pooled, so each tissue keeps its own pair of rows. One logo per tissue,
all from a single scored set — which is why
adding ontology terms costs nothing extra, and why a term with no RNA-seq track is simply
skipped rather than being an error. The cell after the run prints the exact tracks behind each
logo.

With the default `ISM_SCORER_KEY = 'RNA_SEQ'` the scorer is gene-centric, so each number is the
predicted change in *MAP3K1* expression in that tissue. Switching it to `ATAC` or `CHIP_TF`
gives a center-mask scorer instead: no gene dimension, and each number becomes the local
accessibility or binding change at the variant. `track_scalar` detects which kind it is from
the returned object, so nothing else needs changing.

Note the context interval stays at 1 Mb: shorter contexts would exclude *MAP3K1* from the
window for the upstream variants, and the gene-level scorer needs the gene present.

In [ ]:
# @title ISM helper functions
ISM_SCORER = variant_scorers.RECOMMENDED_VARIANT_SCORERS[ISM_SCORER_KEY]
IS_GENE_SCORER = ISM_SCORER_KEY in ('RNA_SEQ', 'RNA_SEQ_ACTIVE', 'CAGE', 'PROCAP')


def track_scalar(adata, ontology: str, gene: str | None = None, agg=None) -> float:
    '''Collapses one score matrix to a single number for one ontology term.

    Gene-centric scorers (RNA_SEQ) return a (gene x track) matrix, so we select the
    focal gene; center-mask scorers (ATAC, CHIP_TF) return a (1 x track) matrix and
    need no gene selection. Whether the matrix is gene-centric is read off the object
    itself, so this helper works for either scorer. Multiple tracks of the same
    ontology term are combined with TRACK_AGG.
    '''
    onto_mask = (adata.var['ontology_curie'] == ontology).to_numpy()
    if 'gene_name' in adata.obs.columns:
        gene_mask = (adata.obs['gene_name'] == (gene or FOCAL_GENE)).to_numpy()
    else:
        gene_mask = np.ones(adata.X.shape[0], dtype=bool)
    values = adata.X[np.ix_(gene_mask, onto_mask)]
    if values.size == 0 or np.all(np.isnan(values)):
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return float((agg or TRACK_AGG)(values))


def ism_logo_matrix(scores, ontology: str, ism_interval: genome.Interval,
                    agg=None) -> np.ndarray:
    '''Turns a list of per-variant ISM scores into a (width, 4) sequence-logo matrix.'''
    return ism.ism_matrix(
        [track_scalar(entry[0], ontology, agg=agg) for entry in scores],
        variants=[entry[0].uns['variant'] for entry in scores],
        interval=ism_interval,
    )


def ism_track_table(scores) -> pd.DataFrame:
    # Lists the tracks that feed each tissue's logo, so the aggregation is inspectable.
    var = scores[0][0].var
    columns = [c for c in ['name', 'biosample_name', 'strand', 'Assay title'] if c in var]
    rows = []
    for term, label in ONTOLOGY_TERMS.items():
        subset = var[var['ontology_curie'] == term]
        if subset.empty:
            rows.append({'ontology_curie': term, 'tissue': label, 'n_tracks': 0})
            continue
        for _, track in subset.iterrows():
            rows.append({
                'ontology_curie': term,
                'tissue': label,
                'n_tracks': len(subset),
                **{c: track[c] for c in columns},
            })
    return pd.DataFrame(rows)


def available_ontologies(scores) -> list[str]:
    '''Ontology terms from the configuration that actually have tracks in this output.'''
    present = set(scores[0][0].var['ontology_curie'])
    return [term for term in ONTOLOGY_TERMS if term in present]


print(f'ISM scorer: {ISM_SCORER}')
print(f'Window: {ISM_WIDTH} bp -> {3 * ISM_WIDTH:,} scored variants per background')

In [ ]:
# @title Run ISM (expensive — see the cost note at the top)
ism_results = {}

if RUN_ISM:
    targets = ISM_VARIANTS or [v.name for v in VARIANTS]
    for rsid in targets:
        variant = VARIANT_BY_RSID[rsid]
        context_interval = variant.reference_interval.resize(SEQUENCE_LENGTH)
        ism_interval = variant.reference_interval.resize(ISM_WIDTH)
        print(f'\n{rsid} ({variant}) — mutating {ism_interval}')

        ism_results[rsid] = {
            'interval': ism_interval,
            'ref': dna_model.score_ism_variants(
                interval=context_interval,
                ism_interval=ism_interval,
                variant_scorers=[ISM_SCORER],
                interval_variant=None,
                max_workers=MAX_WORKERS,
            ),
            'alt': dna_model.score_ism_variants(
                interval=context_interval,
                ism_interval=ism_interval,
                variant_scorers=[ISM_SCORER],
                interval_variant=variant,
                max_workers=MAX_WORKERS,
            ),
        }
    print(f'\nISM complete for: {", ".join(ism_results)}')
else:
    print('RUN_ISM is False — skipping.')

In [ ]:
# @title Which tracks feed each tissue's logo
if ism_results:
    first = next(iter(ism_results.values()))['ref']
    adata = first[0][0]
    print(f'Scored per ISM variant: {adata.X.shape[0]} gene rows x '
          f'{adata.X.shape[1]} tracks (all tracks, not just the adipose subset).')
    if 'gene_name' in adata.obs.columns:
        print(f'Rows kept for the logos: gene_name == {FOCAL_GENE!r}')
    print(f'Tracks per tissue collapsed with: {TRACK_AGG.__name__}\n')
    display(ism_track_table(first))
else:
    print('No ISM results in this session.')

Each variant gets one figure with a REF and an ALT logo per tissue. The two logos of a tissue
share a y-axis so their heights can be compared directly, and the variant position is marked.

In [ ]:
# @title Figure 4: REF/ALT sequence logos per variant
for rsid, result in ism_results.items():
    variant = VARIANT_BY_RSID[rsid]
    ism_interval = result['interval']
    tissues = available_ontologies(result['ref'])
    if not tissues:
        print(f'{rsid}: no configured ontology term has {ISM_SCORER_KEY} tracks — skipped.')
        continue

    components = []
    for term in tissues:
        m_ref = ism_logo_matrix(result['ref'], term, ism_interval)
        m_alt = ism_logo_matrix(result['alt'], term, ism_interval)
        # Shared symmetric limits make REF and ALT heights comparable; without them
        # each panel autoscales and an asymmetry can be an artefact of the axes.
        if ISM_SHARED_YLIM:
            limit = float(max(np.abs(m_ref).max(), np.abs(m_alt).max())) * 1.05 or 1.0
            ylim = (-limit, limit)
        else:
            ylim = None
        label = ONTOLOGY_TERMS[term]
        components += [
            plot_components.SeqLogo(
                scores=m_ref, scores_interval=ism_interval,
                ylabel=f'{label}\nREF background', ylim=ylim,
            ),
            plot_components.SeqLogo(
                scores=m_alt, scores_interval=ism_interval,
                ylabel=f'{label}\nALT background', ylim=ylim,
            ),
        ]

    fig = plot_components.plot(
        components,
        annotations=[variant_annotation([variant])],
        interval=ism_interval,
        fig_width=18,
        title=(
            f'ISM around {rsid} ({variant}) — '
            f'{ISM_SCORER_KEY}'
            + (f' / {FOCAL_GENE}' if IS_GENE_SCORER else '')
        ),
    )
    fig.savefig(OUTPUT_DIR / f'Diag_ism_logos_{rsid}.png', dpi=200, bbox_inches='tight')
    plt.show()

### Mean versus max aggregation

When an ontology term has several RNA-seq tracks, the two have very different behaviour. `mean`
is unbiased; `max` is systematically above zero even on noise, and because `ism_matrix`
mean-centres each position and draws the background base at minus the mean of its three
alternatives, that bias becomes a uniform shift of every letter. A panel can end up entirely on
one side of zero for statistical rather than biological reasons.

This cell renders both from the already-scored results, so comparing them costs nothing. The
printed table is the part to read: if `drawn_mean` is far from zero and `one_sided` is True, the
panel's apparent direction is the aggregation bias, not the sequence. Features that persist
across both aggregators are the ones worth interpreting.

In [ ]:
# @title Compare mean vs max aggregation, no re-scoring { run: "auto" }
COMPARE_VARIANT = ""  # @param {type:"string"}
# @markdown Leave blank to use the first variant with ISM results.
COMPARE_ONTOLOGY = ""  # @param {type:"string"}
# @markdown Leave blank to use PRIMARY_ONTOLOGY.

if ism_results:
    rsid = COMPARE_VARIANT.strip() or next(iter(ism_results))
    result = ism_results[rsid]
    term = COMPARE_ONTOLOGY.strip() or PRIMARY_ONTOLOGY
    available = available_ontologies(result['ref'])
    if term not in available:
        print(f'{term} has no tracks; using {available[0]} instead.')
        term = available[0]

    n_tracks = int((result['ref'][0][0].var['ontology_curie'] == term).sum())
    print(f'{rsid} — {ONTOLOGY_TERMS[term]} ({term}), {n_tracks} track(s)')
    if n_tracks == 1:
        print('Single track: mean and max are identical here.')

    components, rows = [], []
    for agg_name, agg in [('mean', np.nanmean), ('max', np.nanmax)]:
        for background in ('ref', 'alt'):
            matrix = ism_logo_matrix(result[background], term, result['interval'], agg=agg)
            drawn = matrix[matrix != 0]
            rows.append({
                'aggregation': agg_name,
                'background': background.upper(),
                'drawn_mean': drawn.mean(),
                'drawn_min': drawn.min(),
                'drawn_max': drawn.max(),
                'one_sided': bool(np.all(drawn < 0) or np.all(drawn > 0)),
            })
            components.append(
                plot_components.SeqLogo(
                    scores=matrix, scores_interval=result['interval'],
                    ylabel=f'{agg_name}\n{background.upper()}',
                )
            )

    fig = plot_components.plot(
        components,
        annotations=[variant_annotation([VARIANT_BY_RSID[rsid]])],
        interval=result['interval'],
        fig_width=18,
        title=f'{rsid} — {ONTOLOGY_TERMS[term]}: mean vs max aggregation',
    )
    fig.savefig(OUTPUT_DIR / f'Diag_ism_aggregation_{rsid}.png', dpi=200,
                bbox_inches='tight')
    plt.show()
    display(pd.DataFrame(rows))
else:
    print('No ISM results in this session — run section 8 first.')

## 9. Variant effects on *MAP3K1* versus background variants

A raw effect size is hard to interpret on its own: is a shift in predicted *MAP3K1* expression
large, or is it what any substitution at that position would produce? We compare each variant
against two nulls:

1. **Allelic background** — the other possible alternate alleles at the *same* position. This
   isolates the effect of the specific base change from the effect of the position.
2. **Local sequence background** — every ISM substitution in the surrounding window
   (available for free from section 8, if it was run). This is the broader distribution of
   effects achievable anywhere nearby.

AlphaGenome's own `quantile_score` (section 7) is a third, genome-wide null based on common
variants; the two nulls here are local and therefore stricter.

The comparison is drawn once per gene in `COMPARISON_GENES` and once per tissue in
`COMPARISON_ONTOLOGIES` — *MAP3K1* and *SETD9*, in bulk adipose tissue and mesenchymal stem cell
of adipose, by default. *SETD9* sits roughly 250 kb upstream of the variants and well inside the
scored window, so it comes free from the same scoring pass: the gene-level scorer returns every
gene in the interval, and adding one costs no extra API calls. It serves as a within-window
comparator — a variant that moves *MAP3K1* but not *SETD9* is acting with some specificity,
whereas one that moves both similarly is more likely reflecting a broad regional effect. Reading them side by side separates an effect the
model predicts across adipose contexts from one confined to the progenitor state, which matters
here because *MAP3K1* regulation at this locus has been implicated in adipocyte differentiation
rather than mature adipocyte function. The two panels are on independent x-axes, since a
gene-level score is only comparable within a track set.

In [ ]:
# @title Score the allelic background variants
BASES = ['A', 'C', 'G', 'T']

# Restricted to the variants the section 9 figures report, unless widened above.
BACKGROUND_VARIANTS_IN_SCOPE = [
    v for v in VARIANTS
    if BACKGROUND_SCOPE == 'all' or v.name in MANUSCRIPT_RSIDS
]
print(f'background comparison on {len(BACKGROUND_VARIANTS_IN_SCOPE)} variant(s): '
      f'{[v.name for v in BACKGROUND_VARIANTS_IN_SCOPE]}')

background_variants = [
    genome.Variant(
        chromosome=v.chromosome,
        position=v.position,
        reference_bases=v.reference_bases,
        alternate_bases=base,
        name=f'{v.name}_bg_{base}',
    )
    for v in BACKGROUND_VARIANTS_IN_SCOPE
    for base in BASES
    if base not in (v.reference_bases, v.alternate_bases)
]

if RUN_BACKGROUND:
    eval_variants = BACKGROUND_VARIANTS_IN_SCOPE + background_variants
    eval_intervals = [v.reference_interval.resize(SEQUENCE_LENGTH) for v in eval_variants]

    bg_scores = dna_model.score_variants(
        intervals=eval_intervals,
        variants=eval_variants,
        variant_scorers=[variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']],
        max_workers=MAX_WORKERS,
    )

    genes_present = set(bg_scores[0][0].obs['gene_name'])
    missing_genes = [g for g in COMPARISON_GENES if g not in genes_present]
    if missing_genes:
        print('Not in the scored window, skipping: ' + ', '.join(missing_genes))
    genes = [g for g in COMPARISON_GENES if g in genes_present]

    rows = []
    for variant, score in zip(eval_variants, bg_scores):
        for gene in genes:
            for term in ONTOLOGY_TERMS:
                rows.append({
                    'variant': str(variant),
                    'name': variant.name,
                    'rsid': variant.name.split('_bg_')[0],
                    'is_observed': '_bg_' not in variant.name,
                    'position': variant.position,
                    'alt': variant.alternate_bases,
                    'gene': gene,
                    'ontology_curie': term,
                    'ontology_label': ONTOLOGY_TERMS[term],
                    'score': track_scalar(score[0], term, gene=gene),
                })
    allelic_df = pd.DataFrame(rows).dropna(subset=['score'])
    allelic_df.to_csv(OUTPUT_DIR / 'Tab_allelic_background_scores.csv', index=False)
    print(f'Scored {len(eval_variants)} variants '
          f'({len(background_variants)} background) for '
          f'{", ".join(genes)} across {allelic_df.ontology_curie.nunique()} tissues.')
    allelic_df.head()
else:
    allelic_df = pd.DataFrame()
    print('RUN_BACKGROUND is False — skipping.')

In [ ]:
# @title Figure 5: gene effects vs background variants
# One figure per gene in COMPARISON_GENES, one column per tissue, one row per variant.
# Grey shows the nulls, the coloured diamond is the observed variant.
if not allelic_df.empty:
    scored_terms = set(allelic_df['ontology_curie'])
    tissues = [t for t in COMPARISON_ONTOLOGIES if t in scored_terms]
    skipped = [t for t in COMPARISON_ONTOLOGIES if t not in scored_terms]
    if skipped:
        print('No RNA-seq tracks, skipping: '
              + ', '.join(f'{ONTOLOGY_TERMS[t]} ({t})' for t in skipped))

    # Only the variants that were actually scored get a row. The background comparison runs
    # on BACKGROUND_SCOPE, which is narrower than VARIANTS by default, so laying the figure
    # out over every variant would leave most rows empty and make the populated ones small.
    SCORED_VARIANTS = [v for v in VARIANTS if v.name in set(allelic_df['rsid'])]
    if len(SCORED_VARIANTS) < len(VARIANTS):
        print(f'{len(SCORED_VARIANTS)} of {len(VARIANTS)} variants were scored against the '
              'background; the rest are left out of the figure rather than drawn empty.')
    if not SCORED_VARIANTS:
        raise ValueError('no variant has background scores — check BACKGROUND_SCOPE.')

    summary = []
    for gene in allelic_df['gene'].unique():
        gene_df = allelic_df[allelic_df['gene'] == gene]
        rng = np.random.default_rng(0)
        fig, axes = plt.subplots(
            len(SCORED_VARIANTS), len(tissues),
            figsize=(5.8 * len(tissues), 1.9 * len(SCORED_VARIANTS)),
            sharex='col', squeeze=False,
        )

        for col, term in enumerate(tissues):
            tissue_df = gene_df[gene_df['ontology_curie'] == term]

            for row, variant in enumerate(SCORED_VARIANTS):
                ax = axes[row, col]
                rsid = variant.name
                sub = tissue_df[tissue_df['rsid'] == rsid]
                observed = sub.loc[sub['is_observed'], 'score']
                observed = float(observed.iloc[0]) if len(observed) else np.nan
                allelic_bg = sub.loc[~sub['is_observed'], 'score'].to_numpy()

                # Local background from section 8, when ISM used the same scorer.
                local_bg = np.array([])
                if rsid in ism_results and ISM_SCORER_KEY == 'RNA_SEQ':
                    local_bg = np.array([
                        track_scalar(entry[0], term, gene=gene)
                        for entry in ism_results[rsid]['ref']
                    ])
                    local_bg = local_bg[~np.isnan(local_bg)]

                if local_bg.size:
                    ax.scatter(local_bg, rng.normal(0, 0.06, local_bg.size),
                               s=6, color='lightgrey', alpha=0.5, zorder=1)
                ax.scatter(allelic_bg, np.zeros_like(allelic_bg), s=45,
                           color='dimgrey', zorder=3)
                ax.scatter([observed], [0], s=110, marker='D',
                           color=VARIANT_COLORS[rsid], edgecolor='black',
                           linewidth=0.6, zorder=4)
                ax.axvline(0, ls='dotted', color='black', lw=1)

                reference_bg = local_bg if local_bg.size else allelic_bg
                summary.append({
                    'gene': gene,
                    'rsid': rsid,
                    'variant': str(variant),
                    'ontology_curie': term,
                    'tissue': ONTOLOGY_TERMS[term],
                    'observed_score': observed,
                    'allelic_bg_n': allelic_bg.size,
                    'allelic_bg_mean': allelic_bg.mean() if allelic_bg.size else np.nan,
                    'local_bg_n': local_bg.size,
                    'percentile_vs_background': (
                        100 * np.mean(reference_bg < observed)
                        if reference_bg.size else np.nan
                    ),
                })

                ax.set_yticks([])
                ax.spines[['left', 'right', 'top']].set_visible(False)
                if col == 0:
                    ax.set_ylabel(rsid, rotation=0, ha='right', va='center')

            axes[0, col].set_title(ONTOLOGY_TERMS[term], fontsize=10)
            axes[-1, col].set_xlabel(f'{gene} ALT - REF (log2)')

        legend_handles = [
            Line2D([0], [0], marker='o', color='none', markerfacecolor='lightgrey',
                   markersize=6, label='local ISM background'),
            Line2D([0], [0], marker='o', color='none', markerfacecolor='dimgrey',
                   markersize=8, label='other alleles, same position'),
            Line2D([0], [0], marker='D', color='none', markerfacecolor='white',
                   markeredgecolor='black', markersize=9, label='observed variant'),
        ]
        fig.legend(handles=legend_handles, loc='lower center', ncol=3,
                   frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.03))
        fig.suptitle(f'{gene} effect vs background variants', y=1.01)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'Fig_{gene}_effect_vs_mutational_background.png', dpi=200,
                    bbox_inches='tight')
        plt.show()

    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(OUTPUT_DIR / 'Tab_mutational_background_summary.csv', index=False)
    display(summary_df)

### 9b. Is in silico mutagenesis informative for these variants?

The sequence logos above show what the model has learned around each variant, but they do not
by themselves establish that the observed allele matters. Two comparisons are more decisive,
and both are computed here: where the observed allele falls within the exhaustive in silico mutagenesis
of the surrounding 256 bp, and where it falls among the three possible substitutions at its
own position. The answer is sobering. Only three of twenty variant, gene and tissue
combinations fall beyond the 1st or 99th percentile of the local background, and the observed
allele is the most extreme of the three possible ones in four of twenty. This is what should be
expected here. T2D is highly polygenic, the individual effect of any one common variant is
small, and these variants sit 300 kbp or more from the promoters of the candidate target genes,
a distance at which current models are known to attenuate distal effects heavily. Reading a
direction off the attributions is therefore not warranted, and the background comparison, not
the logo, is the informative summary.


In [ ]:
# @title 9b. Supplementary figure: ISM effects against the mutational background
# Panel a: percentile of the observed allele within the local background.
# Panel b: the observed allele against the other two possible alleles at the same position.
TISSUES_FOR_FIGURE = ['Adipose tissue', 'Mesenchymal stem cell of adipose']
SHORT_TISSUE = {'Adipose tissue': 'Adipose tissue',
                'Mesenchymal stem cell of adipose': 'AMSC'}
GENE_COLOUR = {'MAP3K1': '#1f5fa8', 'SETD9': '#c2691f'}
RSID_ORDER = [r for r in CANONICAL_VARIANTS if r in set(summary_df['rsid'])]

summary = summary_df[summary_df['tissue'].isin(TISSUES_FOR_FIGURE)]
alleles = allelic_df[allelic_df['ontology_label'].isin(TISSUES_FOR_FIGURE)]
row_of = {rsid: i for i, rsid in enumerate(RSID_ORDER[::-1])}

with plt.rc_context({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
                     'xtick.labelsize': 10, 'ytick.labelsize': 10}):
    fig, (ax_pct, ax_alleles) = plt.subplots(1, 2, figsize=(12.4, 4.4))

    ax_pct.axvspan(0, 1, color='0.85', zorder=0)
    ax_pct.axvspan(99, 100, color='0.85', zorder=0)
    ax_pct.axvline(50, color='0.45', ls=':', lw=1.1, zorder=1)
    for _, row in summary.iterrows():
        first = row['tissue'] == TISSUES_FOR_FIGURE[0]
        ax_pct.scatter(row['percentile_vs_background'],
                       row_of[row['rsid']] + (0.17 if first else -0.17),
                       s=80, marker='o' if first else 's',
                       facecolor=GENE_COLOUR[row['gene']], edgecolor='white',
                       linewidth=0.8, zorder=3)
    ax_pct.set_xlim(-2, 102)
    ax_pct.set_xlabel('Percentile within the 768-substitution local ISM null')
    ax_pct.set_title('a) Most variants are unremarkable against\nevery substitution in the surrounding 256 bp')

    for (gene, tissue, rsid), group in alleles.groupby(['gene', 'ontology_label', 'rsid']):
        first = tissue == TISSUES_FOR_FIGURE[0]
        y = row_of[rsid] + (0.17 if first else -0.17)
        others = group.loc[~group['is_observed'], 'score'].to_numpy() * 1e3
        ax_alleles.plot(others, [y] * len(others), ls='', marker='x', color='0.55',
                        markersize=7, mew=1.4, zorder=2)
        ax_alleles.scatter(group.loc[group['is_observed'], 'score'].iloc[0] * 1e3, y,
                           s=80, marker='o' if first else 's',
                           facecolor=GENE_COLOUR[gene], edgecolor='white',
                           linewidth=0.8, zorder=3)
    ax_alleles.axvline(0, color='0.45', ls=':', lw=1.1, zorder=1)
    ax_alleles.set_xlabel(r'Predicted expression change, ALT - REF (log2, $\times10^{-3}$)')
    ax_alleles.set_title('b) The observed allele is rarely the most\n'
                         'extreme of the three at its own position')

    for ax in (ax_pct, ax_alleles):
        for i in range(len(RSID_ORDER)):
            ax.axhline(i, color='0.93', lw=0.8, zorder=0)
        ax.set_yticks(range(len(RSID_ORDER)))
        ax.set_yticklabels(RSID_ORDER[::-1])
        ax.set_ylim(-0.7, len(RSID_ORDER) - 0.3)
        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)

    legend_handles = [
        plt.Line2D([], [], marker='o', ls='', color=GENE_COLOUR['MAP3K1'], label='MAP3K1'),
        plt.Line2D([], [], marker='o', ls='', color=GENE_COLOUR['SETD9'], label='SETD9'),
        plt.Line2D([], [], marker='o', ls='', color='0.4', label='Adipose tissue'),
        plt.Line2D([], [], marker='s', ls='', color='0.4', label='AMSC'),
        plt.Line2D([], [], marker='x', ls='', color='0.55', mew=1.4,
                   label='other alleles at the same position (b)'),
    ]
    fig.legend(handles=legend_handles, fontsize=10, loc='lower center', ncol=5,
               frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'SuppFig_ism_vs_mutational_background.png', dpi=400, bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / 'SuppFig_ism_vs_mutational_background.pdf', bbox_inches='tight')
plt.show()

extreme = ((summary['percentile_vs_background'] < 1)
           | (summary['percentile_vs_background'] > 99)).sum()
most_extreme = sum(
    abs(group.loc[group['is_observed'], 'score'].iloc[0])
    > group.loc[~group['is_observed'], 'score'].abs().max()
    for _, group in alleles.groupby(['gene', 'ontology_label', 'rsid'])
)
print(f'Beyond the 1st/99th percentile of the local background: {extreme} of {len(summary)}')
print(f'Observed allele the most extreme of the three possible: {most_extreme} '
      f'of {alleles.groupby(["gene", "ontology_label", "rsid"]).ngroups}')


## 10. Sequence-level allele substitution: each variant alone, and the four together

Everything up to here uses `predict_variant` or `score_variants`, both of which substitute
**one** variant into the interval. This section replaces that mechanism with direct sequence
editing, which lifts that restriction: we fetch the reference sequence for the locus window,
write alternate alleles into it, and call `predict_sequence` on the result. One edit gives the
single-variant case; four edits give the risk haplotype. Both are handled by the same code, so
the single-variant and multi-variant figures are directly comparable — they differ only in how
many bases were changed.

This reproduces the Borzoi panels of manuscript Figure 3b,c with AlphaGenome, and extends them:
Borzoi was run with the four linked variants inserted simultaneously, whereas here each variant
also gets its own pair of panels.

**Allele sets.** An *allele set* is just a list of variants written into the sequence together.
By default there are five: `rs459193`, `rs256903`, `rs173964` and `rs256904` on their own, and
`haplotype_4` with all four at once. `rs3843467` is left out of the haplotype, matching the
Borzoi analysis — it sits ~46 kb away and is in lower LD with the other four — but it can be
added to `SINGLE_VARIANT_RSIDS` to get its own panels.

**Why `predict_sequence` and not `predict_variant`.** `predict_variant` takes a single `Variant`
object and so cannot represent four simultaneous edits. Fetching the reference and editing it
ourselves also means the *baseline* prediction is made from the same fetched sequence as the
perturbed one: if the fetch were subtly wrong, both predictions would be wrong in the same way
and their difference would still be the effect of the substituted alleles. Before anything is
predicted, every variant's REF allele is checked against the fetched sequence; a mismatch means
the coordinates or the assembly are wrong, so that check raises rather than warns.

**Window.** The locus interval from section 4 is reused, since it is built to contain *ANKRD55*
and *MAP3K1* together along with every variant — one window covers all the genes we want to read
out. *SETD9* falls inside it as well and so comes free.

**What is quantified.** For each gene, predicted RNA-seq is sliced to the gene body and reduced
per track to the mean baseline signal and the mean perturbed-minus-baseline difference. Only
tracks on the gene's own strand are kept: RNA-seq is stranded, so antisense tracks carry
essentially no signal for that gene and would otherwise add a cloud of near-zero points that
reflects orientation rather than biology.

**Cost and memory.** RNA-seq is 1 bp resolution, so a 1 Mb window across every track in the model
is several GB per prediction. Two things keep this tractable. Predictions are batched over
ontology terms and reduced to gene slices before the next batch is requested, so peak memory is
one batch. And the baseline is predicted **once per batch and shared by every allele set**, so
`n_batches x (1 + n_allele_sets)` model calls are needed rather than `2 x n_batches x
n_allele_sets` — with five sets that is roughly a 40% saving. Per-position arrays (needed only
for the box panel) are retained for the genes listed in `HAPLO_POSITION_GENES`; other genes keep
their summary statistics and are drawn with the scatter panel alone.

**How to read the result.** These differences are raw predicted-coverage units, not a calibrated
effect size, and they are small. The informative pattern is not the magnitude of any one track
but the *consistency*: whether the shift is one-directional across independent tracks, and
whether the tracks moved most are the ones with real baseline expression of the gene. The
scatter panel makes the second point visually — the empty wedge at low baseline signal is a
ceiling effect, since a track that barely expresses the gene has little room to move.

Section 10d builds the adipose track list; 10e then compares the sets: the sum of the four single-variant shifts against the shift
measured with all four present. Agreement means the model is treating them additively; a gap
means it is not, which is itself worth knowing before any single-variant score is used as a
proxy for the haplotype.

In [ ]:
# @title 10a. Build the reference sequence and one edited sequence per allele set { display-mode: "form" }
# @markdown Variants given their own panels, one edit each. Comma-separated rsIDs, or
# @markdown `auto` for the five the manuscript reports.
SINGLE_VARIANT_RSIDS = "auto"  # @param {type:"string"}
# @markdown Variants written into the sequence *together* as one haplotype. Leave blank to skip
# @markdown the multi-variant set. The Borzoi figure this reproduces used these four and left
# @markdown out rs3843467 (further away, lower LD).
HAPLOTYPE_RSIDS = "rs459193,rs256903,rs173964,rs256904"  # @param {type:"string"}
# @markdown Also write each block in as its own haplotype, so a block is scored as a unit
# @markdown rather than one variant at a time.
BLOCK_HAPLOTYPES = True  # @param {type:"boolean"}
# @markdown Genes to quantify. Each must fall inside the locus window; genes that only
# @markdown partially overlap it are clipped and reported.
HAPLOTYPE_GENES = "MAP3K1,SETD9,ANKRD55"  # @param {type:"string"}
# @markdown Optional local/GCS hg38 FASTA. Leave blank to fetch the sequence over HTTP
# @markdown instead (UCSC, falling back to Ensembl) — no multi-GB download needed.
FASTA_PATH = ""  # @param {type:"string"}
# @markdown Check every alternate allele against Ensembl's variation record before
# @markdown predicting. The alleles come from the fine mapping's own variant index and are
# @markdown already checked against the hand-checked variants, the phase table and the
# @markdown fetched reference base, so this is a fourth confirmation that needs network.
VERIFY_ALLELES_WITH_ENSEMBL = False  # @param {type:"boolean"}

import gc

import requests

if SINGLE_VARIANT_RSIDS.strip().lower() == 'auto':
    SINGLE_RSIDS = sorted((r for r in MANUSCRIPT_RSIDS if r in VARIANT_BY_RSID),
                          key=lambda r: VARIANT_BY_RSID[r].position)
    print(f'SINGLE_VARIANT_RSIDS = auto -> {SINGLE_RSIDS}')
else:
    SINGLE_RSIDS = [s.strip() for s in SINGLE_VARIANT_RSIDS.split(',') if s.strip()]
HAPLO_RSIDS = [s.strip() for s in HAPLOTYPE_RSIDS.split(',') if s.strip()]
HAPLO_GENE_LIST = [s.strip() for s in HAPLOTYPE_GENES.split(',') if s.strip()]
HAPLO_VARIANTS = [VARIANT_BY_RSID[r] for r in HAPLO_RSIDS]

# An allele set is simply a named list of variants to write into the sequence at once:
# length 1 for the single-variant panels, length 4 for the haplotype. Everything
# downstream — prediction, reduction, plotting — treats the two identically.
ALLELE_SETS = {rsid: [VARIANT_BY_RSID[rsid]] for rsid in SINGLE_RSIDS}
HAPLO_SET_NAME = f'haplotype_{len(HAPLO_RSIDS)}' if HAPLO_RSIDS else None
if HAPLO_SET_NAME:
    ALLELE_SETS[HAPLO_SET_NAME] = HAPLO_VARIANTS

# One set per block. Each is scored against the same baseline as everything else, so the
# block shifts and the single-variant shifts are directly comparable, which is what the
# additivity panel needs.
if BLOCK_HAPLOTYPES:
    for _block, _rsids in GROUP_RSIDS.items():
        _members = [VARIANT_BY_RSID[r] for r in _rsids if r in VARIANT_BY_RSID]
        if len(_members) > 1:
            ALLELE_SETS[_block] = _members
print(f'allele sets: {list(ALLELE_SETS)}')

# The alleles are written into the *locus* window from section 4, not a variant-centred
# one: that interval is built to contain ANKRD55 and MAP3K1 together as well as every
# variant, so baseline and perturbed predictions share one coordinate frame that already
# covers every gene we want to read out.
HAPLO_INTERVAL = LOCUS_INTERVAL
assert HAPLO_INTERVAL.width in dna_client.SUPPORTED_SEQUENCE_LENGTHS.values(), (
    f'{HAPLO_INTERVAL.width:,} bp is not a supported input length — set '
    'sequence_length to 1MB in the settings form and re-run section 4.'
)
EDITED_VARIANTS = {v.name: v for variants in ALLELE_SETS.values() for v in variants}
for _v in EDITED_VARIANTS.values():
    assert _v.is_snv, f'{_v.name} is not an SNV; the substitution code below assumes SNVs.'
    assert HAPLO_INTERVAL.contains(_v.reference_interval), f'{_v.name} outside the window.'


def reference_sequence(interval: genome.Interval) -> str:
    """Reference sequence for an interval, uppercased.

    Routed through Section A's fetcher, which tries UCSC, then Ensembl, then a local FASTA,
    retries each chunk, and caches the result to Drive. A host being unreachable or returning
    a 500 on one chunk is therefore survivable, and a re-run costs nothing.
    """
    if FASTA_PATH.strip():
        from alphagenome.io import fasta

        print(f'Reading sequence from {FASTA_PATH}')
        return fasta.FastaExtractor(FASTA_PATH.strip()).extract(interval).upper()

    if 'fetch_reference' not in globals():
        raise RuntimeError(
            'fetch_reference is not defined — run the Section A sequence cell first, or set '
            'FASTA_PATH above to an indexed hg38 FASTA.')
    print(f'Reference sequence for {interval.chromosome}:{interval.start:,}-'
          f'{interval.end:,}')
    return fetch_reference(interval.chromosome, interval.start, interval.end).upper()


haplo_ref_sequence = reference_sequence(HAPLO_INTERVAL)
assert len(haplo_ref_sequence) == HAPLO_INTERVAL.width, (
    f'Got {len(haplo_ref_sequence):,} bp, expected {HAPLO_INTERVAL.width:,}.'
)

# Coordinate sanity check. If a fetched base disagrees with the variant's REF allele the
# coordinates or the assembly are wrong, and every number downstream would be meaningless —
# so this raises rather than warns.
mismatches = []
for variant in EDITED_VARIANTS.values():
    offset = variant.start - HAPLO_INTERVAL.start
    observed = haplo_ref_sequence[offset:offset + len(variant.reference_bases)]
    flag = 'OK' if observed == variant.reference_bases else 'MISMATCH'
    print(f'  {variant.name:10s} {variant.chromosome}:{variant.position:,} '
          f'expected {variant.reference_bases} / found {observed}  {flag}')
    if flag == 'MISMATCH':
        mismatches.append(variant.name)
if mismatches:
    raise ValueError(
        f'REF allele mismatch for {", ".join(mismatches)} — the fetched sequence is not '
        'hg38, or the variant coordinates are wrong.'
    )


def edited_sequence(variants, sequence: str = None,
                    interval: genome.Interval = None) -> str:
    """Writes the alternate alleles of `variants` into the reference sequence.

    The single-variant and haplotype cases differ only in the length of `variants`,
    which is the whole point of routing both through this function.
    """
    sequence = haplo_ref_sequence if sequence is None else sequence
    interval = HAPLO_INTERVAL if interval is None else interval
    bases = list(sequence)
    for variant in variants:
        bases[variant.start - interval.start] = variant.alternate_bases
    return ''.join(bases)


# Cross-check against the alleles Section A actually predicted with. Both sections now
# derive from CANONICAL_VARIANTS, so this should never fire — it is here because it did
# fire before that change (rs173964 was A>C in Section A and A>G here), and a silent
# disagreement makes the two models' haplotypes incomparable while looking identical.
# It raises rather than warns, because every number below depends on it.
_reference_alleles = (
    {rsid: tuple(entry) for rsid, entry in BORZOI['variants'].items()}
    if 'BORZOI' in globals() else
    {rsid: tuple(CANONICAL_VARIANTS[rsid]) for rsid in HAPLOTYPE_RSIDS_CANONICAL}
)
_source = 'Section A as run' if 'BORZOI' in globals() else 'CANONICAL_VARIANTS'

disagreements = []
for rsid, (chrom, position, ref, alt) in _reference_alleles.items():
    variant = VARIANT_BY_RSID.get(rsid)
    if variant is None:
        continue
    here = (variant.chromosome, variant.position,
            variant.reference_bases, variant.alternate_bases)
    if here != (chrom, position, ref, alt):
        disagreements.append(
            f'  {rsid}: Section B {here[2]}>{here[3]} at {here[0]}:{here[1]:,} | '
            f'{_source} {ref}>{alt} at {chrom}:{position:,}'
        )
if disagreements:
    raise ValueError(
        'The two models are not writing the same alleles:\n'
        + '\n'.join(disagreements)
        + '\n\nFix CANONICAL_VARIANTS and re-run Section A; do not proceed, the '
          'cross-model comparison in Section C would be meaningless.'
    )
print(f'\nAlleles agree with {_source} for all '
      f'{len(_reference_alleles)} shared variant(s).')

# --- Allele verification against Ensembl ------------------------------------
# The REF check above only confirms the reference base. This confirms that the ALT we are
# writing in is an allele Ensembl actually records for the rsID, at the position we think.
def ensembl_variant(rsid: str) -> dict | None:
    response = requests.get(
        f'https://rest.ensembl.org/variation/human/{rsid}',
        headers={'Content-Type': 'application/json'}, timeout=60,
    )
    if response.status_code != 200:
        return None
    return response.json()


if VERIFY_ALLELES_WITH_ENSEMBL:
    print('\nVerifying alleles against Ensembl:')
    problems = []
    for variant in EDITED_VARIANTS.values():
        try:
            record = ensembl_variant(variant.name)
        except Exception as exc:  # noqa: BLE001
            print(f'  {variant.name:10s} lookup failed ({exc!r}) — not verified.')
            continue
        if record is None:
            print(f'  {variant.name:10s} not found in Ensembl — not verified.')
            continue
        mappings = [m for m in record.get('mappings', [])
                    if str(m.get('assembly_name', '')).startswith('GRCh38')]
        alleles, positions = set(), set()
        for mapping in mappings:
            alleles.update(str(mapping.get('allele_string', '')).split('/'))
            positions.add(int(mapping.get('start', -1)))
        observed = '/'.join(sorted(a for a in alleles if a))
        allele_ok = variant.alternate_bases in alleles
        position_ok = variant.position in positions
        flag = 'OK' if (allele_ok and position_ok) else 'CHECK'
        print(f'  {variant.name:10s} using {variant.reference_bases}>'
              f'{variant.alternate_bases} at {variant.position:,}; '
              f'Ensembl GRCh38 {observed} at {sorted(positions)}  {flag}')
        if not allele_ok:
            problems.append(f'{variant.name}: ALT {variant.alternate_bases} is not an '
                            f'observed allele (Ensembl lists {observed})')
        if not position_ok:
            problems.append(f'{variant.name}: position {variant.position:,} does not '
                            f'match Ensembl {sorted(positions)}')
    if problems:
        print('\nALLELE PROBLEMS — resolve before quoting any number below:')
        for problem in problems:
            print(f'  - {problem}')
    else:
        print('  All alleles and positions match Ensembl GRCh38.')

ALLELE_SEQUENCES = {}
for name, variants in ALLELE_SETS.items():
    sequence = edited_sequence(variants)
    n_diff = sum(a != b for a, b in zip(haplo_ref_sequence, sequence))
    assert n_diff == len(variants), (
        f'{name}: sequences differ at {n_diff} positions, expected {len(variants)}.'
    )
    ALLELE_SEQUENCES[name] = sequence

# Kept for backwards compatibility with any cell that referred to the haplotype directly.
haplo_alt_sequence = ALLELE_SEQUENCES.get(HAPLO_SET_NAME)
gc.collect()

print(f'\nWindow: {HAPLO_INTERVAL} ({HAPLO_INTERVAL.width:,} bp)')
print(f'{len(ALLELE_SETS)} allele set(s), each verified against the reference:')
for name, variants in ALLELE_SETS.items():
    print(f'  {name:14s} {len(variants)} edit(s): '
          + ', '.join(f'{v.name} {v.reference_bases}>{v.alternate_bases}' for v in variants))

In [ ]:
# @title 10b. Predict every allele set against a shared baseline { display-mode: "form" }
# @markdown Score every RNA-seq track in the model (as the Borzoi figure did), or only the
# @markdown adipose ontology terms from section 2.
HAPLO_ALL_TRACKS = True  # @param {type:"boolean"}
# @markdown Ontology terms per API call. RNA-seq is 1 bp resolution, so a 1 Mb window with
# @markdown every track is several GB — batching keeps peak memory to one batch. Each batch
# @markdown is a separate model run, so fewer, larger batches are cheaper but need more RAM.
# @markdown Set to 0 to request everything in a single call.
HAPLO_TERM_BATCH = 80  # @param {type:"slider", min:0, max:400, step:20}
# @markdown Genes for which per-position differences are kept (needed for the box panel).
# @markdown Every gene keeps its per-track summary either way; genes left out here are drawn
# @markdown with the scatter panel only. Keeping all genes for all allele sets is what makes
# @markdown this cell expensive in RAM, not in API calls.
HAPLO_POSITION_GENES = "MAP3K1"  # @param {type:"string"}
# @markdown Positions kept per track for the box plot. Summary statistics always use every
# @markdown position; this only thins what gets stored and drawn.
HAPLO_MAX_POSITIONS = 20000  # @param {type:"integer"}
# @markdown Pseudocount added to both totals before the log2 ratio, so tracks with almost no
# @markdown predicted expression cannot produce huge fold changes out of numerical noise.
HAPLO_FC_PSEUDOCOUNT = 1e-3  # @param {type:"number"}
# @markdown Tracks kept per gene. "gene_and_unstranded" keeps the gene's own strand plus
# @markdown unstranded tracks — this is the correct default and the one that matches the
# @markdown Borzoi section. AlphaGenome annotates unstranded assays as '.', and GTEx RNA-seq
# @markdown is unstranded, so "gene" silently discards every GTEx track, including
# @markdown Adipose_Subcutaneous and Adipose_Visceral_Omentum. "gene" and "both" are kept
# @markdown only for comparison with earlier runs.
HAPLO_STRAND_MODE = "gene_and_unstranded"  # @param ["gene_and_unstranded", "gene", "both"]
# @markdown Bin width in bp before differencing. AlphaGenome RNA-seq is 1 bp resolution;
# @markdown Borzoi predicts 32 bp bins, so 32 makes the per-position spread in the box
# @markdown panel comparable between the two. Per-track means are unaffected.
HAPLO_BIN_SIZE = 32  # @param {type:"integer"}
# @markdown Width of the cropped region used by the appendix, measured from each gene's
# @markdown 5' end. The Borzoi run only had the first 55,744 bp of MAP3K1 inside its output
# @markdown window, so this reproduces that region of interest. Both the full gene body and
# @markdown this crop are reduced from the *same* predictions, so the appendix is free.
# @markdown Set to 0 to skip the cropped region set entirely.
APPENDIX_GENE_MAX_BP = 55744  # @param {type:"integer"}

import gc
import time

POSITION_GENES = {s.strip() for s in HAPLO_POSITION_GENES.split(',') if s.strip()}

# --- Gene regions to quantify over -----------------------------------------
haplo_regions = {}
for gene in HAPLO_GENE_LIST:
    try:
        gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene)
    except Exception as exc:  # noqa: BLE001
        print(f'{gene:10s} not found in the annotation ({exc!r}) — skipped.')
        continue
    region = gene_interval.intersect(HAPLO_INTERVAL)
    if region is None:
        print(f'{gene:10s} does not overlap the window — skipped.')
        continue
    clipped = region.width < gene_interval.width
    haplo_regions[gene] = {'region': region, 'strand': gene_interval.strand}
    print(f'{gene:10s} {region}  ({region.width:,} bp, strand {gene_interval.strand})'
          + (f'  CLIPPED from {gene_interval.width:,} bp' if clipped else ''))
if not haplo_regions:
    raise ValueError('No requested gene overlaps the window.')


def crop_to_five_prime(regions: dict, width: int) -> dict:
    """Trims each gene region to `width` bp from its 5' end, which end depending on strand."""
    cropped = {}
    for gene, entry in regions.items():
        region, strand = entry['region'], entry['strand']
        if region.width > width:
            region = (genome.Interval(region.chromosome, region.end - width, region.end)
                      if strand == '-' else
                      genome.Interval(region.chromosome, region.start, region.start + width))
        cropped[gene] = {'region': region, 'strand': strand}
    return cropped


# Two region sets reduced from one set of predictions: the full gene body (the analysis
# proper) and a 5'-end crop matching the region the Borzoi output window happened to cover
# (the appendix). Adding the second costs slicing time, not API calls.
PRIMARY_REGION_SET = 'full_gene'
REGION_SETS = {PRIMARY_REGION_SET: haplo_regions}
if APPENDIX_GENE_MAX_BP:
    REGION_SETS['cropped_5prime'] = crop_to_five_prime(haplo_regions, APPENDIX_GENE_MAX_BP)
    print(f"\nAppendix region set: 5' {APPENDIX_GENE_MAX_BP:,} bp of each gene "
          + ', '.join(f'{gene} {entry["region"].width:,} bp'
                      for gene, entry in REGION_SETS['cropped_5prime'].items()))

# --- Which ontology terms to request ----------------------------------------
if HAPLO_ALL_TRACKS:
    rna_metadata = dna_model.output_metadata(
        organism=dna_client.Organism.HOMO_SAPIENS
    ).rna_seq
    haplo_terms = sorted(set(rna_metadata['ontology_curie'].dropna()))
else:
    haplo_terms = list(ONTOLOGY_TERMS)

batches = (
    [haplo_terms[i:i + HAPLO_TERM_BATCH]
     for i in range(0, len(haplo_terms), HAPLO_TERM_BATCH)]
    if HAPLO_TERM_BATCH > 0 else [haplo_terms]
)
n_calls = len(batches) * (1 + len(ALLELE_SETS))
print(f'\nTracks per gene: {HAPLO_STRAND_MODE}; values binned at '
      f'{HAPLO_BIN_SIZE} bp before differencing.')
if HAPLO_STRAND_MODE == 'gene':
    print('  WARNING: "gene" drops unstranded tracks, i.e. every GTEx sample. Use '
          '"gene_and_unstranded" unless reproducing an earlier run.')
print(f'\n{len(haplo_terms)} ontology terms in {len(batches)} batch(es) x '
      f'(1 baseline + {len(ALLELE_SETS)} allele sets) = {n_calls} model call(s).')
print('The baseline is predicted once per batch and reused by every allele set, which is '
      f'{2 * len(batches) * len(ALLELE_SETS) - n_calls} call(s) fewer than predicting a '
      'baseline per set.')


# --- Exon masks -------------------------------------------------------------
# Intronic bins are mostly baseline noise in these predictions: they dilute a mean and add
# variance to a sum without carrying much about expression, so the exon-restricted measures
# are the informative ones. Three sources are tried in order, and which one was used is
# recorded per gene and written into every output table:
#
#   1. the GENCODE GTF already loaded in section 4 — offline, deterministic, and the same
#      annotation the gene coordinates come from;
#   2. the Ensembl REST API, with retries;
#   3. an empirical definition from the model's own baseline predictions, computed later
#      once the first batch is in hand.
#
# The whole-region fallback used previously is gone. It produced exon_* columns numerically
# identical to the unrestricted ones, which is indistinguishable from a real result unless
# you happen to compare the two columns — a silent failure of exactly the kind that is worth
# engineering out.
EXON_SIGNAL_QUANTILE = 0.90  # quantile used by the empirical fallback
EXON_LOOKUP_RETRIES = 3


def gtf_exon_intervals(gene: str) -> list:
    """Exons of `gene` from the GENCODE dataframe loaded in section 4, 1-based inclusive."""
    frame = gtf
    feature_column = next(
        (c for c in ('Feature', 'feature', 'type') if c in frame.columns), None
    )
    name_column = next(
        (c for c in ('gene_name', 'gene_id', 'gene') if c in frame.columns), None
    )
    start_column = next((c for c in ('Start', 'start') if c in frame.columns), None)
    end_column = next((c for c in ('End', 'end') if c in frame.columns), None)
    if not all([feature_column, name_column, start_column, end_column]):
        raise RuntimeError(f'unexpected GTF columns: {list(frame.columns)[:12]}')
    exons = frame[
        (frame[feature_column].astype(str).str.lower() == 'exon')
        & (frame[name_column].astype(str) == gene)
    ]
    if exons.empty:
        raise RuntimeError(f'no exon rows for {gene}')
    # pyranges-style frames are 0-based half-open; +1 puts starts on the 1-based footing
    # the mask builder expects, and one base of slack at a 32 bp bin size is immaterial.
    return [(int(row[start_column]) + 1, int(row[end_column]))
            for _, row in exons.iterrows()]


def ensembl_exon_intervals(region) -> list:
    last_error = None
    for attempt in range(1, EXON_LOOKUP_RETRIES + 1):
        try:
            response = requests.get(
                f'https://rest.ensembl.org/overlap/region/human/'
                f'{region.chromosome.removeprefix("chr")}:{region.start + 1}-{region.end}',
                params={'feature': 'exon'},
                headers={'Content-Type': 'application/json'},
                timeout=120,
            )
            response.raise_for_status()
            return [(int(entry['start']), int(entry['end'])) for entry in response.json()]
        except Exception as error:  # noqa: BLE001
            last_error = error
            if attempt < EXON_LOOKUP_RETRIES:
                time.sleep(2 ** attempt)
    raise last_error


def _bin_mask(base_mask: np.ndarray) -> np.ndarray:
    """Bins a per-base mask; a bin is exonic if any of its bases is."""
    if HAPLO_BIN_SIZE <= 1:
        return base_mask
    usable = (len(base_mask) // HAPLO_BIN_SIZE) * HAPLO_BIN_SIZE
    return base_mask[:usable].reshape(-1, HAPLO_BIN_SIZE).any(axis=1)


def build_exon_mask(gene: str, region) -> tuple:
    """(mask, source) over binned positions, or (None, 'pending') for the empirical route."""
    for source, getter in (('GENCODE GTF', lambda: gtf_exon_intervals(gene)),
                           ('Ensembl REST', lambda: ensembl_exon_intervals(region))):
        try:
            exons = getter()
        except Exception as error:  # noqa: BLE001
            print(f'    {source} unavailable for {gene} ({type(error).__name__}: {error})')
            continue
        base_mask = np.zeros(region.width, dtype=bool)
        for exon_start, exon_end in exons:
            first = max(0, exon_start - 1 - region.start)
            last = min(region.width, exon_end - region.start)
            if last > first:
                base_mask[first:last] = True
        binned = _bin_mask(base_mask)
        if binned.any():
            return binned, source
        print(f'    {source} returned no exon overlapping this region for {gene}')
    return None, 'pending'


def signal_exon_mask(baseline_values: np.ndarray) -> np.ndarray:
    """Empirical exons: bins where the best-expressing tracks carry most of the signal."""
    strongest = baseline_values[:, baseline_values.mean(axis=0).argsort()[::-1][:50]]
    profile = strongest.mean(axis=1)
    mask = profile >= np.quantile(profile, EXON_SIGNAL_QUANTILE)
    return mask if mask.any() else np.ones(len(profile), dtype=bool)


print('\nExon masks:')
for region_set, regions in REGION_SETS.items():
    for gene, entry in regions.items():
        mask, source = build_exon_mask(gene, entry['region'])
        entry['exon_mask'] = mask
        entry['exon_source'] = source
        if mask is None:
            print(f'  {region_set:15s} {gene:10s} annotation unavailable — will be defined '
                  f'empirically from the baseline predictions '
                  f'({EXON_SIGNAL_QUANTILE:.0%} quantile)')
        else:
            print(f'  {region_set:15s} {gene:10s} {int(mask.sum()):6d} / {len(mask):6d} '
                  f'exonic bins ({mask.mean():.1%}) — {source}')


def _gene_slice(rna, gene: str, regions: dict):
    """Gene-body slice as (values, metadata), with the strand rule applied to both.

    Antisense tracks are dropped because RNA-seq is stranded and they carry essentially no
    signal for the gene — a cloud of near-zero points reflecting orientation rather than a
    real null. Unstranded tracks are *kept*: AlphaGenome marks them '.', GTEx RNA-seq is
    unstranded, and filtering on the gene's strand alone therefore removes every GTEx
    track — including Adipose_Subcutaneous and Adipose_Visceral_Omentum, the two most
    directly comparable to the GTEx evidence in this study. The Borzoi section keeps
    unstranded tracks, so this also makes the two sections' track rules match.

    The filtering is done here on a boolean mask rather than with filter_to_*_strand(),
    because those helpers cannot express "this strand or unstranded".
    """
    region, strand = regions[gene]['region'], regions[gene]['strand']
    sliced = rna.slice_by_interval(region)
    metadata = sliced.metadata.reset_index(drop=True)
    if HAPLO_STRAND_MODE == 'both':
        keep = np.ones(len(metadata), dtype=bool)
    elif HAPLO_STRAND_MODE == 'gene':
        keep = (metadata['strand'] == strand).to_numpy()
    else:  # gene_and_unstranded
        keep = metadata['strand'].isin([strand, '.']).to_numpy()
    return sliced.values[:, keep], metadata.loc[keep].reset_index(drop=True)


def _binned(values: np.ndarray) -> np.ndarray:
    """Averages 1 bp values into HAPLO_BIN_SIZE bins, dropping a ragged final bin.

    Binning is applied identically to baseline and perturbed before differencing, so it
    changes the spread shown in the box panel but not the per-track means.
    """
    if HAPLO_BIN_SIZE <= 1:
        return values
    usable = (values.shape[0] // HAPLO_BIN_SIZE) * HAPLO_BIN_SIZE
    if usable == 0:
        return values
    return values[:usable].reshape(
        usable // HAPLO_BIN_SIZE, HAPLO_BIN_SIZE, values.shape[1]
    ).mean(axis=1)


def _summarise(base_values, base_metadata, pert_values, gene: str, exon_mask=None,
               exon_source: str = 'unknown'):
    """Per-track effect measures for one gene.

    Four measures, because mean absolute difference ranks well but reads badly: the signed
    mean per bin; the signed total across the region (integrated, in coverage x bp); each of
    those restricted to exonic bins; and the log2 ratio of total perturbed to total baseline,
    which is dimensionless and so comparable between tracks at very different expression
    levels.
    """
    difference = pert_values - base_values
    n_positions = difference.shape[0]
    if exon_mask is None or len(exon_mask) != n_positions:
        exon_mask = np.ones(n_positions, dtype=bool)
    bin_bp = max(1, HAPLO_BIN_SIZE)

    def fold_change(mask):
        return np.log2(
            (pert_values[mask].sum(axis=0) + HAPLO_FC_PSEUDOCOUNT)
            / (base_values[mask].sum(axis=0) + HAPLO_FC_PSEUDOCOUNT)
        )

    everything = np.ones(n_positions, dtype=bool)
    summary = pd.DataFrame({
        'track_name': list(base_metadata['name']),
        'biosample_name': list(
            base_metadata.get('biosample_name', base_metadata['name'])
        ),
        'strand': list(base_metadata['strand']),
        'mean_baseline': base_values.mean(axis=0),
        'exon_mean_baseline': base_values[exon_mask].mean(axis=0),
        'mean_difference': difference.mean(axis=0),
        'exon_mean_difference': difference[exon_mask].mean(axis=0),
        'integrated_difference': difference.sum(axis=0) * bin_bp,
        'exon_integrated_difference': difference[exon_mask].sum(axis=0) * bin_bp,
        'log2_fold_change': fold_change(everything),
        'exon_log2_fold_change': fold_change(exon_mask),
        'mean_abs_difference': np.abs(difference).mean(axis=0),
    })
    summary['percent_change'] = (2 ** summary['exon_log2_fold_change'] - 1) * 100
    summary['exon_source'] = exon_source
    summary['exon_bins'] = int(exon_mask.sum())
    summary['region_bins'] = n_positions
    if gene not in POSITION_GENES:
        return summary, None
    stride = max(1, difference.shape[0] // max(1, HAPLO_MAX_POSITIONS))
    return summary, difference[::stride].copy()


effect_summary_by_region = {
    region: {name: {} for name in ALLELE_SETS} for region in REGION_SETS
}
effect_diff_by_region = {
    region: {name: {} for name in ALLELE_SETS} for region in REGION_SETS
}

for index, batch in enumerate(batches, start=1):
    print(f'  batch {index}/{len(batches)} ({len(batch)} terms) — baseline ...', flush=True)
    baseline_output = dna_model.predict_sequence(
        sequence=haplo_ref_sequence,
        requested_outputs=[dna_client.OutputType.RNA_SEQ],
        ontology_terms=batch,
        interval=HAPLO_INTERVAL,
    )
    # Reduce the baseline to gene slices once, then discard the full-window prediction:
    # every allele set in this batch is compared against these same arrays.
    baseline_slices = {}
    for region_set, regions in REGION_SETS.items():
        for gene in regions:
            values, metadata = _gene_slice(baseline_output.rna_seq, gene, regions)
            if len(metadata) == 0:
                continue
            binned_values = _binned(values.astype(np.float32))
            entry = regions[gene]
            if entry.get('exon_mask') is None:
                # Empirical fallback, resolved once from the first batch's baseline and
                # then reused, so every batch shares one definition of "exonic".
                entry['exon_mask'] = signal_exon_mask(binned_values)
                entry['exon_source'] = (
                    f'empirical ({EXON_SIGNAL_QUANTILE:.0%} quantile of pooled baseline)'
                )
                print(f'    {region_set}/{gene}: exons defined empirically — '
                      f'{int(entry["exon_mask"].sum())} / {len(entry["exon_mask"])} bins')
            baseline_slices[(region_set, gene)] = (binned_values, metadata)
    del baseline_output
    gc.collect()

    for name, variants in ALLELE_SETS.items():
        print(f'    {name} ...', flush=True)
        perturbed_output = dna_model.predict_sequence(
            sequence=ALLELE_SEQUENCES[name],
            requested_outputs=[dna_client.OutputType.RNA_SEQ],
            ontology_terms=batch,
            interval=HAPLO_INTERVAL,
        )
        for (region_set, gene), (base_values, base_metadata) in baseline_slices.items():
            pert_values = _binned(
                _gene_slice(perturbed_output.rna_seq, gene,
                            REGION_SETS[region_set])[0].astype(np.float32)
            )
            summary, difference = _summarise(
                base_values, base_metadata, pert_values, gene,
                REGION_SETS[region_set][gene].get('exon_mask'),
                REGION_SETS[region_set][gene].get('exon_source', 'unknown'),
            )
            effect_summary_by_region[region_set][name].setdefault(gene, []).append(summary)
            if difference is not None:
                effect_diff_by_region[region_set][name].setdefault(gene, []).append(difference)
            del pert_values
        del perturbed_output
        gc.collect()

    del baseline_slices
    gc.collect()

# --- Concatenate batches and rank tracks ------------------------------------
for region_set in REGION_SETS:
  effect_summary = effect_summary_by_region[region_set]
  effect_diff = effect_diff_by_region[region_set]
  for name in ALLELE_SETS:
    for gene in list(effect_summary[name]):
        summary = pd.concat(effect_summary[name][gene], ignore_index=True)
        pieces = effect_diff[name].get(gene)
        difference = np.concatenate(pieces, axis=1) if pieces else None

        # Drop any track a term appeared in twice, then rank by mean absolute
        # difference, keeping summary rows and per-position columns in the same order.
        keep = ~summary['track_name'].duplicated().to_numpy()
        summary = summary[keep].reset_index(drop=True)
        if difference is not None:
            difference = difference[:, keep]

        order = summary['mean_abs_difference'].to_numpy().argsort()[::-1]
        summary = summary.iloc[order].reset_index(drop=True)
        summary['gene'] = gene
        summary['allele_set'] = name
        summary['n_edits'] = len(ALLELE_SETS[name])
        summary['region_set'] = region_set
        effect_summary[name][gene] = summary
        effect_diff[name][gene] = (
            difference[:, order] if difference is not None else None
        )

# The analysis proper runs on the full gene body; `effect_summary` / `effect_diff` keep
# pointing at it, and the appendix reads the cropped tables explicitly.
effect_summary = effect_summary_by_region[PRIMARY_REGION_SET]
effect_diff = effect_diff_by_region[PRIMARY_REGION_SET]

effect_table = pd.concat(
    [summary
     for region_map in effect_summary_by_region.values()
     for gene_map in region_map.values()
     for summary in gene_map.values()],
    ignore_index=True,
)
effect_table.to_csv(OUTPUT_DIR / 'allele_set_track_effects.csv', index=False)

# Kept under their old names so section 10c-style code written against the haplotype
# alone still works.
if HAPLO_SET_NAME:
    haplo_summary = effect_summary[HAPLO_SET_NAME]
    haplo_diff = effect_diff[HAPLO_SET_NAME]

for name in ALLELE_SETS:
    print(f'\n{name} (full gene body):')
    for gene, summary in effect_summary[name].items():
        negative = int((summary['mean_difference'] < 0).sum())
        print(f'  {gene:10s} {len(summary):4d} tracks, {negative:4d} '
              f'({negative / len(summary):.0%}) with reduced predicted expression, '
              f'median difference {summary["mean_difference"].median():.3e}')

In [ ]:
# @title 10b-check. Verify the exon restriction actually restricted anything
# @markdown A previous run silently produced `exon_*` columns identical to the unrestricted
# @markdown ones, because the exon lookup failed and the mask fell back to the whole region.
# @markdown Numerically that is indistinguishable from a real result unless the two columns
# @markdown are compared, so they are compared here, before any figure is drawn.
rows = []
problems = []
for region_set, region_map in effect_summary_by_region.items():
    for allele_set, gene_map in region_map.items():
        for gene, summary in gene_map.items():
            identical = np.allclose(summary['mean_difference'],
                                    summary['exon_mean_difference'])
            source = summary['exon_source'].iloc[0]
            rows.append({
                'region_set': region_set, 'allele_set': allele_set, 'gene': gene,
                'exon_source': source,
                'exon_bins': int(summary['exon_bins'].iloc[0]),
                'region_bins': int(summary['region_bins'].iloc[0]),
                'exon_fraction': summary['exon_bins'].iloc[0] / summary['region_bins'].iloc[0],
                'exon_equals_whole_region': identical,
            })
            if identical:
                problems.append(f'{region_set}/{allele_set}/{gene} (source: {source})')

exon_check = pd.DataFrame(rows).drop_duplicates(['region_set', 'gene'])
exon_check.to_csv(OUTPUT_DIR / 'exon_mask_check.csv', index=False)
display(exon_check)

if problems:
    print('\nPROBLEM — the exonic measures equal the whole-region measures for:')
    for problem in problems:
        print(f'  {problem}')
    print('\nDo not report any exon_* number from this run. Re-run 10b; if the GENCODE GTF '
          'and Ensembl are both unavailable the empirical fallback should still produce a '
          'restricted mask, so an exact match here means something is wrong with the mask '
          'itself rather than with the lookup.')
    raise SystemExit('Exon restriction is not in effect — stopping.')

sources = set(exon_check['exon_source'])
print('\nExon restriction in effect for every gene and region set.')
print(f'Sources used: {", ".join(sorted(sources))}')
if any(source.startswith('empirical') for source in sources):
    print('NOTE: at least one gene used the empirical definition, which derives exons from '
          'the model\'s own\npredictions rather than from annotation. State this in the '
          'methods if those numbers are reported.')

In [ ]:
# @title 10c. Figure 6: ranked track shifts and shift vs. baseline expression { display-mode: "form" }
# @markdown Genes to draw. Comma-separated, or blank for every gene quantified in 10b.
PLOT_GENES = "MAP3K1"  # @param {type:"string"}
# @markdown Allele sets to draw. Comma-separated, or blank for all of them.
PLOT_ALLELE_SETS = ""  # @param {type:"string"}
# @markdown Tracks shown in the box plot (ranked by mean absolute difference).
N_TOP_BOX = 20  # @param {type:"slider", min:5, max:40, step:1}
# @markdown Tracks highlighted in colour on the scatter panel.
N_TOP_SCATTER = 15  # @param {type:"slider", min:0, max:30, step:1}


def _short(label: str, width: int = 38) -> str:
    return label if len(label) <= width else label[: width - 3] + '...'


def plot_allele_set_effect(summary, difference, title: str, gene: str,
                           n_top_box: int = N_TOP_BOX,
                           n_top_scatter: int = N_TOP_SCATTER):
    """Draws the two Borzoi-style panels for one allele set and one gene.

    Panel 1 (box) needs per-position differences, so it is drawn only when `difference`
    is not None — that is, for genes listed in HAPLO_POSITION_GENES. Panel 2 (scatter)
    needs only the per-track means and is always drawn.
    """
    labels = [f'RNA:{row.biosample_name} ({row.strand})' for row in summary.itertuples()]
    n_box = min(n_top_box, len(summary)) if difference is not None else 0
    n_colour = min(n_top_scatter, len(summary))

    if n_box:
        fig, (ax_box, ax_scatter) = plt.subplots(
            2, 1, figsize=(11, 4 + 0.34 * n_box + 6),
            height_ratios=[0.34 * n_box + 1, 6],
        )
    else:
        fig, ax_scatter = plt.subplots(figsize=(11, 6))
        ax_box = None

    # --- Panel 1: distribution of per-position differences, ranked ----------
    # Tracks are ordered by mean |difference|, so the top rows are the tracks the
    # alleles move most; the spread within a row is across positions in the gene body.
    if ax_box is not None:
        data = [difference[:, i] for i in range(n_box)][::-1]
        ax_box.boxplot(
            data, vert=False, widths=0.6, showfliers=True,
            flierprops={'marker': '.', 'markersize': 2, 'markerfacecolor': 'black',
                        'markeredgecolor': 'none'},
            medianprops={'color': 'darkorange'},
        )
        ax_box.set_yticklabels([_short(labels[i]) for i in range(n_box)][::-1], fontsize=8)
        ax_box.set_xlabel('Difference (Perturbed - Baseline)')
        ax_box.set_title('Tracks ranked by mean absolute difference', fontsize=10)
        ax_box.axvline(0, ls='dotted', lw=0.8, color='grey')

    # --- Panel 2: effect size against how much the track is expressed at all ---
    # The empty wedge at low baseline signal is the point of this panel: a track with
    # little predicted expression of the gene has little room to change.
    ax_scatter.scatter(
        summary['mean_baseline'], summary['mean_difference'],
        s=14, facecolor='none', edgecolor='dimgrey', alpha=0.45,
        linewidth=0.6, label='All RNA tracks',
    )
    colours = plt.get_cmap('tab10')
    for i in range(n_colour):
        ax_scatter.scatter(
            summary['mean_baseline'].iloc[i], summary['mean_difference'].iloc[i],
            s=26, color=colours(i % 10), zorder=3, label=_short(labels[i], 30),
        )
    ax_scatter.axhline(0, ls='dashed', lw=0.8, color='grey')
    ax_scatter.set_xlabel('Mean Baseline Signal')
    ax_scatter.set_ylabel('Mean Change (Perturbed - Baseline)')
    ax_scatter.set_title(f'Mean Signal vs Mean Change for RNA Tracks — {gene}', fontsize=10)
    ax_scatter.legend(loc='center left', bbox_to_anchor=(1.01, 0.5),
                      fontsize=6.5, frameon=True)

    fig.suptitle(title, fontsize=12, y=1.02)
    fig.tight_layout()
    return fig


def draw_allele_set_figures(summaries, differences, genes=None, sets=None,
                            tag: str = '', region_label: str = 'full gene body'):
    """Draws Figure 6 for every (allele set, gene) pair in the given tables.

    Shared by the analysis proper and the appendix; `tag` distinguishes the filenames
    and `region_label` says which region the numbers came from.
    """
    genes = genes or [g.strip() for g in PLOT_GENES.split(',') if g.strip()] or list(haplo_regions)
    sets = sets or [s.strip() for s in PLOT_ALLELE_SETS.split(',') if s.strip()] or list(ALLELE_SETS)
    for name in sets:
        variants = ALLELE_SETS[name]
        edits = ', '.join(f'{v.name} {v.reference_bases}>{v.alternate_bases}'
                          for v in variants)
        for gene in genes:
            summary = summaries[name].get(gene)
            if summary is None:
                print(f'{name} / {gene}: no tracks on the gene strand — skipped.')
                continue
            difference = differences[name].get(gene)
            if difference is None:
                print(f'{name} / {gene}: per-position differences not retained '
                      '(add the gene to HAPLO_POSITION_GENES for the box panel).')
            label = ('single variant' if len(variants) == 1
                     else f'{len(variants)}-variant haplotype')
            fig = plot_allele_set_effect(
                summary, difference,
                title=f'{gene}: effect of {name} ({label}) — {edits}  [{region_label}]',
                gene=gene,
            )
            fig.savefig(OUTPUT_DIR / f'SuppFig_alphagenome_{tag}{name}_{gene}_ranked_tracks.png',
                        dpi=200, bbox_inches='tight')
            plt.show()


draw_allele_set_figures(effect_summary, effect_diff)
print('Wrote allele_set_track_effects.csv and one SuppFig_alphagenome_<allele_set>_<gene>_ranked_tracks.png per panel.')

### 10d. Which tracks count as adipose-related?

The enrichment test is only as good as the track list it is testing. Matching three keywords —
`fat`, `adipose`, `adipocyte` — against the biosample name is what the manuscript's Borzoi
analysis did, and on AlphaGenome's track panel it misses real adipose contexts: anything named
after the depot rather than the tissue (*omentum*, *epiploic*), cell models named after the line
(*SGBS*, *3T3-L1*), and terms where the adipose identity lives in the ontology rather than in the
string.

This cell replaces the keyword test with a three-way classification, and shows its work.

**Ontology descent (primary).** Each track carries an ontology CURIE. Rather than looking up
several hundred CURIEs one at a time, the cell asks EBI's OLS for the *descendants* of a handful
of adipose root terms — adipose tissue, fat pad, fat cell, preadipocyte, mesenchymal stem cell of
adipose tissue — which is a few requests in total, and then intersects that descendant set with
the CURIEs actually present in the model's metadata. This is the part that is genuinely
exhaustive: any term the ontology places under one of those roots is caught regardless of how it
is spelled. The resolved label of every root is printed, so a wrong root CURIE is visible
immediately rather than silently narrowing the search.

**String matching (secondary).** A broadened pattern list runs over the biosample name and,
where available, the ontology label and synonyms. Patterns are split in two. *Strong* patterns
are ones that cannot mean anything but adipose — `adipos`, `adipocyt`, `fat pad`, standalone
`fat`, `omentum`/`omental`, `epiploic`, `lipocyte`, `SGBS`, `3T3-L1`, `panniculus adiposus` — and
are accepted automatically. *Review* patterns are ones that are adipose only in context —
`subcutaneous`, `visceral`, `mesenteric`, `perirenal`, `stromal vascular`, bare
`mesenchymal stem cell` — and are listed for inspection rather than included, because
*subcutaneous skin*, *mesenteric artery* and *bone-marrow MSC* all match them and none is fat.
Set `INCLUDE_REVIEW_TIER` to include them anyway, or move individual CURIEs with
`MANUAL_INCLUDE_CURIES`.

**Manual override (final).** Two CURIE lists that force a track in or out, applied last, for
anything the first two passes get wrong.

The cell prints the included tracks grouped by the evidence that admitted them, the review tier
in full, and a paste-ready `ONTOLOGY_TERMS` dictionary — the discovery here is useful well beyond
the enrichment test, since sections 5–9 all run on that dictionary and it currently holds eight
terms chosen by hand. If OLS is unreachable the cell falls back to string matching alone and says
so; the classification is then less complete but nothing breaks.

In [ ]:
# @title 10d. Identify every adipose-related track { display-mode: "form" }
# @markdown Output types whose track metadata is classified. RNA_SEQ is what the enrichment
# @markdown test needs; add ATAC or CHIP_TF to reuse the same track list elsewhere.
CLASSIFY_OUTPUT_TYPES = "RNA_SEQ"  # @param {type:"string"}
# @markdown Query EBI OLS for the descendants of the adipose root terms. A few requests in
# @markdown total, not one per track. Turn off to use string matching alone.
USE_ONTOLOGY_LOOKUP = True  # @param {type:"boolean"}
# @markdown Treat the ambiguous "review" matches (subcutaneous, visceral, mesenteric, bare
# @markdown mesenchymal stem cell...) as adipose. Off by default — most of them are not.
INCLUDE_REVIEW_TIER = False  # @param {type:"boolean"}
# @markdown Comma-separated CURIEs forced in / out, applied after everything else.
MANUAL_INCLUDE_CURIES = ""  # @param {type:"string"}
MANUAL_EXCLUDE_CURIES = ""  # @param {type:"string"}

import re
import time

import requests

# Root terms whose ontology descendants are all adipose by construction. Their resolved
# labels are printed below: if a CURIE here is wrong, the label will not read as expected.
ADIPOSE_ROOTS = {
    'UBERON:0001013': 'adipose tissue (expected)',
    'UBERON:0003916': 'fat pad (expected)',
    'CL:0000136': 'fat cell / adipocyte (expected)',
    'CL:0002334': 'preadipocyte (expected)',
    'CL:0002570': 'mesenchymal stem cell of adipose tissue (expected)',
}

# Unambiguous: any track matching one of these is adipose-related.
STRONG_PATTERNS = {
    'adipose': r'adipos',              # adipose, adiposus, adiposity
    'adipocyte': r'adipocyt',          # adipocyte, preadipocyte, adipocytic
    'fat_pad': r'fat\s*pad',
    'fat': r'\bfat\b',                 # 'omental fat', 'brown fat', 'fat tissue'
    'omentum': r'omentum|omental',
    'epiploic': r'epiplo',             # epiploic appendage / appendix epiploica
    'lipocyte': r'lipocyt',
    'panniculus': r'panniculus\s+adiposus',
    'sgbs': r'\bsgbs\b',               # Simpson-Golabi-Behmel preadipocyte line
    '3t3l1': r'3t3[\s\-]?l1',
    'brown_white_fat': r'\b(brown|white|beige|perivascular)\s+fat\b',
    'depot_fat': r'\b(epididymal|gonadal|perigonadal|inguinal|epicardial|pericardial|'
                 r'periaortic|perirenal|retroperitoneal)\s+(fat|adipos)',
}

# Adipose only in context — reported, not included, unless INCLUDE_REVIEW_TIER is on.
# 'subcutaneous skin tissue', 'mesenteric artery' and bone-marrow MSCs all land here.
REVIEW_PATTERNS = {
    'subcutaneous': r'subcutaneous',
    'visceral': r'visceral',
    'mesenteric': r'mesenter',
    'perirenal': r'perirenal|retroperitoneal',
    'stromal_vascular': r'stromal[\s\-]?vascular|\bsvf\b',
    'mesenchymal_stem_cell': r'mesenchymal\s+stem\s+cell|\bmsc\b',
    'preadipose_line': r'\bstromal\s+cell\b',
}

OLS_BASE = 'https://www.ebi.ac.uk/ols4/api'


def _ols_term(curie: str) -> dict | None:
    """Looks a CURIE up in OLS and returns the first matching term document."""
    response = requests.get(f'{OLS_BASE}/terms', params={'obo_id': curie}, timeout=60)
    response.raise_for_status()
    terms = response.json().get('_embedded', {}).get('terms', [])
    return terms[0] if terms else None


def _ols_descendants(term: dict) -> dict:
    """All hierarchical descendants of a term as {curie: label}, following paging.

    The descendants link is read off the term document rather than constructed by hand,
    which keeps this working across OLS URL-encoding conventions.
    """
    link = term.get('_links', {}).get('hierarchicalDescendants', {}).get('href')
    if not link:
        return {}
    found, url, params = {}, link, {'size': 500}
    while url:
        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        payload = response.json()
        for entry in payload.get('_embedded', {}).get('terms', []):
            if entry.get('obo_id'):
                found[entry['obo_id']] = entry.get('label', '')
        url = payload.get('_links', {}).get('next', {}).get('href')
        params = None  # the next link already carries the paging parameters
        time.sleep(0.1)
    return found


ontology_adipose = {}   # curie -> label, from OLS
ontology_available = False
if USE_ONTOLOGY_LOOKUP:
    try:
        for root, expectation in ADIPOSE_ROOTS.items():
            term = _ols_term(root)
            if term is None:
                print(f'  {root:16s} NOT FOUND in OLS — check this CURIE.')
                continue
            descendants = _ols_descendants(term)
            ontology_adipose[root] = term.get('label', '')
            ontology_adipose.update(descendants)
            print(f'  {root:16s} resolved to "{term.get("label")}"  '
                  f'[{expectation}] — {len(descendants)} descendant term(s)')
        ontology_available = bool(ontology_adipose)
    except Exception as exc:  # noqa: BLE001
        print(f'\nOLS lookup failed ({exc!r}) — falling back to string matching alone. '
              'The classification below is then less complete.')
if ontology_available:
    print(f'\n{len(ontology_adipose)} adipose terms in the ontology closure.')


def _match(patterns: dict, text: str) -> list:
    return [name for name, pattern in patterns.items()
            if re.search(pattern, text, flags=re.IGNORECASE)]


# --- Classify every track of the requested output types ---------------------
output_types = [t.strip().upper() for t in CLASSIFY_OUTPUT_TYPES.split(',') if t.strip()]
all_metadata = dna_model.output_metadata(organism=dna_client.Organism.HOMO_SAPIENS)
manual_include = {c.strip() for c in MANUAL_INCLUDE_CURIES.split(',') if c.strip()}
manual_exclude = {c.strip() for c in MANUAL_EXCLUDE_CURIES.split(',') if c.strip()}

rows = []
for output_type in output_types:
    metadata = getattr(all_metadata, output_type.lower(), None)
    if metadata is None:
        print(f'{output_type}: no metadata on this model — skipped.')
        continue
    for record in metadata.to_dict('records'):
        curie = record.get('ontology_curie') or ''
        biosample = str(record.get('biosample_name') or record.get('name') or '')
        ontology_label = ontology_adipose.get(curie, '')
        text = f'{biosample} {ontology_label}'

        strong = _match(STRONG_PATTERNS, text)
        review = _match(REVIEW_PATTERNS, text)
        by_ontology = curie in ontology_adipose

        if by_ontology:
            tier = 'ontology'
        elif strong:
            tier = 'strong_string'
        elif review:
            tier = 'review'
        else:
            tier = 'not_adipose'

        included = tier in ('ontology', 'strong_string') or (
            tier == 'review' and INCLUDE_REVIEW_TIER
        )
        if curie in manual_include:
            tier, included = 'manual_include', True
        if curie in manual_exclude:
            tier, included = 'manual_exclude', False

        rows.append({
            'output_type': output_type,
            'track_name': record.get('name'),
            'biosample_name': biosample,
            'ontology_curie': curie,
            'ontology_label': ontology_label,
            'strand': record.get('strand'),
            'tier': tier,
            'string_evidence': ','.join(strong) or ('review:' + ','.join(review) if review else ''),
            'is_adipose': included,
        })

track_classification = pd.DataFrame(rows)
track_classification.to_csv(OUTPUT_DIR / 'Tab_alphagenome_adipose_tracks.csv', index=False)

ADIPOSE_TRACKS = set(
    track_classification.loc[track_classification['is_adipose'], 'track_name']
)
ADIPOSE_CURIES = dict(
    track_classification[track_classification['is_adipose']]
    .drop_duplicates('ontology_curie')
    .set_index('ontology_curie')
    .apply(lambda row: row['ontology_label'] or row['biosample_name'], axis=1)
)
ADIPOSE_CURIES.pop('', None)


def is_adipose_track(track_name: str = None, biosample_name: str = None) -> bool:
    """Membership test used by the enrichment analysis.

    Falls back to matching the biosample name against the strong patterns when a track
    name is not in the classification table (e.g. tracks renamed downstream).
    """
    if track_name is not None and track_name in ADIPOSE_TRACKS:
        return True
    if track_name is not None and track_name in set(track_classification['track_name']):
        return False
    return bool(_match(STRONG_PATTERNS, biosample_name or ''))


included = track_classification[track_classification['is_adipose']]
print(f'\n{len(included)} of {len(track_classification)} tracks classified as '
      f'adipose-related, across {included["ontology_curie"].nunique()} ontology terms.')
print(included['tier'].value_counts().to_string())

keyword_only = track_classification['biosample_name'].str.contains(
    'fat|adipose|adipocyte', case=False, na=False
)
print(f'\nThe three-keyword test used for the Borzoi analysis would have found '
      f'{int(keyword_only.sum())} tracks; this finds {len(included)}. '
      f'{int((included["is_adipose"] & ~keyword_only.reindex(included.index, fill_value=False)).sum())} '
      'were added by ontology descent or the broadened patterns.')

print('\nIncluded ontology terms — paste into ONTOLOGY_TERMS in section 2 to widen the '
      'whole notebook (note this increases cost in sections 5-9):')
for curie, label in sorted(ADIPOSE_CURIES.items(), key=lambda item: item[1]):
    print(f"    '{curie}': '{label}',")

review = track_classification[track_classification['tier'] == 'review']
if len(review):
    print(f'\n{len(review)} track(s) in the review tier — ambiguous, currently '
          f'{"INCLUDED" if INCLUDE_REVIEW_TIER else "EXCLUDED"}. '
          'Inspect and move individual CURIEs with the manual lists if needed:')
    display(review[['track_name', 'biosample_name', 'ontology_curie',
                    'string_evidence']].drop_duplicates('biosample_name'))

### 10e. Comparing the allele sets

Two questions the per-set figures cannot answer on their own.

**Is the haplotype effect the sum of its parts?** Each single-variant set and the haplotype set
were measured on the same tracks against the same baseline, so the four single-variant shifts can
simply be added and plotted against the shift measured with all four alleles present. Points on
the diagonal mean the model is behaving additively at this locus; systematic departure means a
single-variant score is not a safe proxy for the haplotype, in whichever direction the departure
runs. This is a statement about the model, not about the biology of the locus.

**Are adipose tracks over-represented among the most affected?** The manuscript reports a
hypergeometric enrichment of adipose-related tracks among the top Borzoi tracks (fold enrichment
18.7, p = 1.9e-9). The same test is run here on AlphaGenome's track set for each allele set,
but using the classification from 10d rather than three keywords: adipose tracks are counted
among the top `ENRICH_TOP_N` by mean absolute difference, against the background of all tracks
scored for the gene. Which tracks count as adipose is by far the most consequential choice in
this test — an under-inclusive list depresses both the background rate and the hit count, and
the two do not cancel — which is what 10d is for. Note that the two track universes are different — AlphaGenome and Borzoi do not
have the same panel of RNA-seq experiments — so the fold enrichments are not directly comparable
between the two models; what carries over is whether the effect concentrates in adipose contexts
at all.

Both are cheap: everything below reuses the tables computed in 10b and makes no further model
calls.

In [ ]:
# @title 10e. Additivity of the single-variant effects and adipose track enrichment { display-mode: "form" }
# @markdown Gene the comparison is made for.
COMPARE_GENE = "MAP3K1"  # @param {type:"string"}
# @markdown Top tracks (by mean absolute difference) tested for adipose enrichment.
ENRICH_TOP_N = 20  # @param {type:"slider", min:5, max:50, step:5}
# @markdown Fallback keywords, used only if 10d has not been run. When it has, the
# @markdown classification from that cell (ontology descent + broadened patterns) is used
# @markdown instead, which is considerably more complete.
ADIPOSE_KEYWORDS = "fat,adipose,adipocyte"  # @param {type:"string"}

from scipy import stats

keywords = [k.strip().lower() for k in ADIPOSE_KEYWORDS.split(',') if k.strip()]


def _adipose_flags(summary):
    """Per-row adipose flag, preferring the 10d classification over raw keywords."""
    if 'is_adipose_track' in globals():
        return pd.Series(
            [is_adipose_track(track_name=row.track_name,
                              biosample_name=row.biosample_name)
             for row in summary.itertuples()],
            index=summary.index,
        )
    return summary['biosample_name'].str.lower().apply(
        lambda label: any(keyword in label for keyword in keywords)
    )


if 'is_adipose_track' not in globals():
    print('Section 10d has not been run — falling back to keyword matching on '
          f'{keywords}, which misses depot- and cell-line-named tracks.')


def compare_allele_sets(summaries, gene: str = None, tag: str = '',
                        region_label: str = 'full gene body'):
    """Additivity panel + adipose enrichment for one set of per-track tables.

    Shared by the analysis proper and the appendix, so the two differ only in which
    region the tables were reduced over.
    """
    gene = gene or COMPARE_GENE
    print(f'--- {gene}, {region_label} ---')
    single_sets = [name for name, variants in ALLELE_SETS.items() if len(variants) == 1]

    if gene not in summaries[next(iter(ALLELE_SETS))]:
        raise ValueError(f'{gene} was not quantified in 10b; '
                         f'available: {list(summaries[next(iter(ALLELE_SETS))])}')

    # --- One wide table: mean_difference per track, one column per allele set ----
    wide = None
    for name in ALLELE_SETS:
        summary = summaries[name][gene]
        columns = summary[['track_name', 'biosample_name', 'strand',
                           'mean_baseline', 'mean_difference']].rename(
            columns={'mean_difference': name}
        )
        wide = columns if wide is None else wide.merge(
            columns[['track_name', name]], on='track_name', how='inner'
        )

    wide['sum_of_singles'] = wide[single_sets].sum(axis=1)
    wide['is_adipose'] = _adipose_flags(wide).to_numpy()
    wide.to_csv(OUTPUT_DIR / f'Tab_alphagenome_{tag}{gene}_allele_set_effects.csv', index=False)
    print(f'{len(wide)} tracks shared across all {len(ALLELE_SETS)} allele sets for '
          f'{gene}; {int(wide["is_adipose"].sum())} are adipose-related.')

    # --- Panel A: sum of single-variant shifts vs. the joint haplotype shift -----
    if HAPLO_SET_NAME and HAPLO_SET_NAME in wide.columns:
        fig, (ax_add, ax_dist) = plt.subplots(1, 2, figsize=(13, 5.5))

        ax_add.scatter(wide['sum_of_singles'], wide[HAPLO_SET_NAME], s=14,
                       facecolor='none', edgecolor='dimgrey', alpha=0.45, linewidth=0.6,
                       label='All RNA tracks')
        adipose = wide[wide['is_adipose']]
        ax_add.scatter(adipose['sum_of_singles'], adipose[HAPLO_SET_NAME], s=26,
                       color='tab:orange', zorder=3, label='Adipose-related')

        limits = np.array([
            min(wide['sum_of_singles'].min(), wide[HAPLO_SET_NAME].min()),
            max(wide['sum_of_singles'].max(), wide[HAPLO_SET_NAME].max()),
        ])
        ax_add.plot(limits, limits, ls='dashed', lw=0.9, color='grey', label='y = x (additive)')

        # Slope through the origin: >1 means the four alleles together move the prediction
        # more than their individual effects add up to, <1 means they partly cancel.
        x, y = wide['sum_of_singles'].to_numpy(), wide[HAPLO_SET_NAME].to_numpy()
        slope = float(x @ y / (x @ x)) if np.any(x) else np.nan
        correlation = float(np.corrcoef(x, y)[0, 1])
        print(f'Additivity: slope through the origin = {slope:.2f}, r = '
              f'{correlation:.2f}. The four alleles together reach {slope:.0%} of the '
              f'sum of their individual effects.')
        ax_add.set_xlabel('Sum of single-variant mean differences')
        ax_add.set_ylabel(f'{HAPLO_SET_NAME} mean difference')
        ax_add.set_title(f'Additivity — {gene} [{region_label}]\n'
                         f'slope through origin = {slope:.2f}, r = {correlation:.2f}',
                         fontsize=10)
        ax_add.legend(fontsize=7, loc='best')

        # --- Panel B: distribution of per-track shifts, one box per allele set ---
        order = single_sets + [HAPLO_SET_NAME]
        ax_dist.boxplot([wide[name].to_numpy() for name in order], showfliers=False,
                        medianprops={'color': 'darkorange'})
        ax_dist.set_xticks(range(1, len(order) + 1))
        ax_dist.set_xticklabels(order)
        for position, name in enumerate(order, start=1):
            jitter = np.random.default_rng(0).normal(0, 0.045, len(adipose))
            ax_dist.scatter(position + jitter, adipose[name], s=12,
                            color='tab:orange', alpha=0.7, zorder=3)
        ax_dist.axhline(0, ls='dashed', lw=0.8, color='grey')
        ax_dist.set_ylabel('Mean difference (Perturbed - Baseline)')
        ax_dist.set_title(f'Per-track shift by allele set — {gene}\n'
                          '(orange: adipose-related tracks)', fontsize=10)
        ax_dist.tick_params(axis='x', rotation=30)

        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'SuppFig_alphagenome_{tag}{gene}_effect_additivity.png',
                    dpi=200, bbox_inches='tight')
        plt.show()
    else:
        print('No multi-variant allele set — additivity panel skipped.')

    # --- Adipose enrichment among the most-affected tracks ----------------------
    # Hypergeometric test, matching the manuscript's Borzoi analysis: how surprising is the
    # number of adipose tracks among the top N, given how many exist in the track universe?
    # Both classifications are reported. The keyword row is the exact procedure used for
    # Borzoi (fold enrichment 18.7, p = 1.9e-9 there) and is what to compare against the
    # manuscript; the classified row is the more complete list from 10d. They differ mainly
    # in the background rate, which is why the two p-values are not interchangeable.
    def _keyword_flags(summary):
        return summary['biosample_name'].str.lower().apply(
            lambda label: any(keyword in label for keyword in keywords)
        )


    methods = {'classified_10d': _adipose_flags}
    if 'is_adipose_track' in globals():
        methods['keyword_borzoi'] = _keyword_flags

    rows = []
    for method, flagger in methods.items():
        for name in ALLELE_SETS:
            summary = summaries[name][gene]  # already ranked by mean |difference|
            is_adipose = flagger(summary)
            n_total, n_adipose = len(summary), int(is_adipose.sum())
            top_n = min(ENRICH_TOP_N, n_total)
            n_hits = int(is_adipose.iloc[:top_n].sum())
            expected = top_n * n_adipose / n_total
            rows.append({
                'method': method,
                'allele_set': name,
                'n_edits': len(ALLELE_SETS[name]),
                'tracks_total': n_total,
                'adipose_total': n_adipose,
                'top_n': top_n,
                'adipose_in_top_n': n_hits,
                'expected': round(expected, 2),
                'fold_enrichment': round(n_hits / expected, 2) if expected else np.nan,
                'pvalue': stats.hypergeom.sf(n_hits - 1, n_total, n_adipose, top_n),
            })

    enrichment = pd.DataFrame(rows)
    enrichment.to_csv(OUTPUT_DIR / f'Tab_alphagenome_{tag}{gene}_adipose_enrichment.csv', index=False)
    display(enrichment)

    return wide, enrichment

    print('\nThe track universe here is the set of tracks scored for this gene under the '
          f'current HAPLO_STRAND_MODE, not the 1,543 RNA tracks of the Borzoi run (which '
          'included both strands and mouse tracks). Fold enrichments are therefore not '
          'directly comparable to the published value; see 10g.')


wide_comparison, enrichment = compare_allele_sets(effect_summary)

### 10f. Supplementary figure: predicted *MAP3K1* expression, all tracks ranked

The panels above are all about *change*. This one is about the baseline: where does AlphaGenome
predict *MAP3K1* is expressed at all, across every RNA-seq track in the model, and do the
adipose contexts sit at the top of that ranking?

It is the AlphaGenome counterpart of the manuscript's argument that adipose tissues carry some of
the highest baseline *MAP3K1* levels, which is what makes them susceptible to a variant effect in
the first place — the depletion wedge in the scatter panel of Figure 6 is the same observation
seen sideways. Nothing is predicted here: the baseline values come from the reference-sequence
prediction already computed in 10b, and because that baseline is shared by every allele set the
figure does not depend on which set is chosen.

Two panels. The upper one ranks every track by mean predicted coverage over the gene body, with
adipose-related tracks (classified in 10d) drawn in colour on top of the full distribution, so
their position in the ranking is visible against the whole track panel rather than in isolation.
The lower one names the top `N_TOP_LABELLED` tracks as a horizontal bar chart, coloured by the
same classification.

Alongside the figure, a two-sided Mann–Whitney U test asks whether adipose tracks are ranked
differently from the rest, reported with the rank-biserial correlation as an effect size. Read it
as a description of the model's predictions, not as evidence about adipose biology: the track
panel is a convenience sample of whatever experiments exist, tissues are represented unequally,
and a low p-value here reflects the composition of that panel as much as anything else. The
figure's real job is to show whether the tracks that respond most to the variants are also the
ones with the expression to respond with.

In [ ]:
# The manuscript quotes a single number from this cell, the adipose median baseline rank,
# in the Limitations rather than showing the panel. The number is printed below.
# @title 10f. Supplementary figure: ranked baseline expression per track { display-mode: "form" }
# @markdown Gene whose predicted baseline expression is ranked.
RANK_GENE = "MAP3K1"  # @param {type:"string"}
# @markdown Tracks named in the lower bar panel.
N_TOP_LABELLED = 30  # @param {type:"slider", min:10, max:60, step:5}
# @markdown Also mark, in the upper panel, the tracks most affected by this allele set.
# @markdown Leave blank to mark none.
MARK_TOP_AFFECTED_SET = "haplotype_4"  # @param {type:"string"}
N_MARK_AFFECTED = 20  # @param {type:"slider", min:0, max:40, step:5}

from scipy import stats

if '_short' not in globals():  # normally defined in 10c
    def _short(label: str, width: int = 38) -> str:
        return label if len(label) <= width else label[: width - 3] + '...'

def plot_baseline_ranking(summaries, gene: str = None, tag: str = '',
                          region_label: str = 'full gene body'):
    """Ranks every track by predicted baseline expression and highlights adipose ones.

    Reads the reference-sequence baseline already computed in 10b, so it costs nothing
    and is reused unchanged by the appendix.
    """
    gene = gene or RANK_GENE
    _rank_set = next(iter(summaries))  # baseline is shared, so any allele set will do
    baseline = (
        summaries[_rank_set][gene]
        .loc[:, ['track_name', 'biosample_name', 'strand', 'mean_baseline']]
        .copy()
    )
    baseline['is_adipose'] = _adipose_flags(baseline).to_numpy()
    baseline = baseline.sort_values('mean_baseline', ascending=False).reset_index(drop=True)
    baseline['rank'] = np.arange(1, len(baseline) + 1)
    baseline.to_csv(OUTPUT_DIR / f'baseline_expression_ranked_{tag}{gene}.csv', index=False)

    adipose = baseline[baseline['is_adipose']]
    other = baseline[~baseline['is_adipose']]

    # Rank test, two-sided. The rank-biserial correlation is the effect size that goes with U:
    # +1 means every adipose track outranks every other track, 0 means no separation.
    if len(adipose) and len(other):
        u_statistic, p_value = stats.mannwhitneyu(
            adipose['mean_baseline'], other['mean_baseline'], alternative='two-sided'
        )
        rank_biserial = 2 * u_statistic / (len(adipose) * len(other)) - 1
    else:
        u_statistic = p_value = rank_biserial = np.nan

    fig, (ax_rank, ax_bar) = plt.subplots(
        2, 1, figsize=(11, 5.5 + 0.28 * min(N_TOP_LABELLED, len(baseline))),
        height_ratios=[5.5, 0.28 * min(N_TOP_LABELLED, len(baseline)) + 1],
    )

    # --- Panel a: every track ranked, adipose drawn on top ----------------------
    ax_rank.scatter(baseline['rank'], baseline['mean_baseline'], s=9,
                    facecolor='none', edgecolor='dimgrey', alpha=0.4, linewidth=0.5,
                    label=f'All RNA tracks (n={len(baseline)})')
    ax_rank.scatter(adipose['rank'], adipose['mean_baseline'], s=30,
                    color='tab:orange', zorder=3,
                    label=f'Adipose-related (n={len(adipose)})')

    # A rug of adipose ranks along the top makes their distribution readable even where the
    # curve is dense and points overplot.
    top = ax_rank.get_ylim()[1]
    ax_rank.plot(adipose['rank'], np.full(len(adipose), top), marker='|', ls='none',
                 color='tab:orange', markersize=7, clip_on=False)

    marked = pd.DataFrame()
    if MARK_TOP_AFFECTED_SET.strip() and N_MARK_AFFECTED:
        set_name = MARK_TOP_AFFECTED_SET.strip()
        if set_name in summaries and gene in summaries[set_name]:
            # effect_summary is already ordered by mean |difference|, so the head is the
            # most-affected tracks; mark where they fall in the baseline ranking.
            affected = set(
                summaries[set_name][gene]['track_name'].head(N_MARK_AFFECTED)
            )
            marked = baseline[baseline['track_name'].isin(affected)]
            ax_rank.scatter(marked['rank'], marked['mean_baseline'], s=90,
                            facecolor='none', edgecolor='tab:blue', linewidth=1.1, zorder=2,
                            label=f'Top {N_MARK_AFFECTED} most affected by {set_name}')
        else:
            print(f'{MARK_TOP_AFFECTED_SET}: not an allele set with {gene} — not marked.')

    ax_rank.set_xlabel('Track rank (1 = highest predicted expression)')
    ax_rank.set_ylabel(f'Mean predicted {gene} coverage')
    ax_rank.set_title(
        f'Predicted baseline {gene} expression across all RNA-seq tracks\n'
        f'adipose median rank {adipose["rank"].median():.0f} of {len(baseline)}; '
        f'Mann-Whitney p = {p_value:.2e}, rank-biserial r = {rank_biserial:.2f}',
        fontsize=10,
    )
    ax_rank.legend(fontsize=7.5, loc='upper right')

    # --- Panel b: the top tracks, named ----------------------------------------
    n_top = min(N_TOP_LABELLED, len(baseline))
    top_tracks = baseline.head(n_top).iloc[::-1]
    colours = ['tab:orange' if flag else 'lightgrey' for flag in top_tracks['is_adipose']]
    ax_bar.barh(range(n_top), top_tracks['mean_baseline'], color=colours,
                edgecolor='dimgrey', linewidth=0.4)
    ax_bar.set_yticks(range(n_top))
    ax_bar.set_yticklabels(
        [_short(f'{row.biosample_name} ({row.strand})', 46)
         for row in top_tracks.itertuples()],
        fontsize=7.5,
    )
    ax_bar.set_ylim(-0.7, n_top - 0.3)
    ax_bar.set_xlabel(f'Mean predicted {gene} coverage')
    ax_bar.set_title(f'Top {n_top} tracks (orange: adipose-related)', fontsize=10)

    fig.suptitle(f'Supplementary figure: baseline {gene} expression by track [{region_label}]', fontsize=12,
                 y=1.01)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'Diag_alphagenome_{tag}{gene}_baseline_ranking.png',
                dpi=200, bbox_inches='tight')
    plt.show()

    print(f'Quoted in the Limitations: adipose median baseline rank '
          f'{adipose["rank"].median():.0f} of {len(baseline)}.')
    print(f'{len(adipose)} adipose track(s) of {len(baseline)}; '
          f'{int((adipose["rank"] <= 0.1 * len(baseline)).sum())} in the top decile of the '
          f'ranking, {int((adipose["rank"] <= 0.25 * len(baseline)).sum())} in the top quartile.')
    if len(marked):
        overlap = int(marked['is_adipose'].sum())
        print(f'Of the {len(marked)} most-affected tracks marked, {overlap} are adipose-related '
              f'and their median baseline rank is {marked["rank"].median():.0f}.')


plot_baseline_ranking(effect_summary)

### 10h. Alternative effect measures, matched to the Borzoi analysis

Mean absolute difference ranks tracks well but reads badly in a legend: the units are scaled
coverage, the sign is discarded, and the value depends on how much of the gene body sits in the
window. Cell 10b therefore computes four measures per track from the same predictions, and this
cell draws them.

| Measure | Question | Units |
|---|---|---|
| `mean_difference` | Which way did it move, per bin? | scaled coverage |
| `exon_mean_difference` | The same, over exonic bins only | scaled coverage |
| `exon_integrated_difference` | How much predicted signal was gained or lost in total? | coverage x bp |
| `exon_log2_fold_change` | By what **fraction** did predicted expression change? | log2 ratio |

The last usually reads best: dimensionless, so a highly expressed track and a quiet one can be
said to change by the same percentage, which the absolute measures cannot express.
`percent_change` carries the same number in a form that goes straight into prose. Exonic bins are
those overlapping an Ensembl-annotated exon of the gene, unioned across transcripts; introns are
mostly baseline noise in these predictions and dilute both means and sums.

**Figure 8** puts adipose tracks against everything else under each measure, with a two-sided
Mann-Whitney U and rank-biserial per panel. **Figure 9** plots each measure against baseline
expression of the gene. The empty wedge at low baseline in the absolute panels is a ceiling
effect — a track with little predicted expression has little room to move — and it should
largely disappear in the relative panel. Adipose tracks leading there too is a stronger statement
than the original figure makes, because it survives dividing the ceiling out.

These are deliberately the same four measures, the same panel layout and the same tests as
Figures 4 and 6 of the Borzoi notebook, so the two models can be compared directly. What is
comparable is the *pattern* — direction, which contexts lead, whether the ceiling explains it —
and not the values: the two models emit coverage on different scales, over different track
panels, from different windows. The same functions are reused by Appendix A on the cropped
region, so all three sets of panels are produced by one implementation.

In [ ]:
# The manuscript quotes these measures as numbers rather than showing the panels, so the
# outputs are Diag_ prefixed and stay out of the figure map.
# @title 10h. Figures 8 and 9: effect measures, adipose vs the rest { display-mode: "form" }
# @markdown Gene the measures are drawn for.
MEASURE_GENE = "MAP3K1"  # @param {type:"string"}
# @markdown Allele set drawn. The haplotype is the analogue of the Borzoi analysis.
MEASURE_SET = "haplotype_4"  # @param {type:"string"}

MEASURES = {
    'mean_difference': 'Mean difference\n(signed, per bin)',
    'exon_mean_difference': 'Exonic mean difference\n(signed, per bin)',
    'exon_integrated_difference': 'Exonic integrated difference\n(coverage x bp)',
    'exon_log2_fold_change': 'Exonic log2 fold change\n(relative)',
}
BASELINE_FOR = {
    measure: ('exon_mean_baseline' if measure.startswith('exon') else 'mean_baseline')
    for measure in MEASURES
}


def measure_table(summaries, gene: str = None, allele_set: str = None):
    """Per-track measures for one allele set and gene, with the adipose flag attached."""
    gene = gene or MEASURE_GENE
    allele_set = allele_set if allele_set in summaries else MEASURE_SET
    if allele_set not in summaries:
        allele_set = next(iter(summaries))
    table = summaries[allele_set][gene].copy()
    table['is_adipose'] = _adipose_flags(table).to_numpy()
    return table, allele_set, gene


def plot_measure_boxes(summaries, gene: str = None, allele_set: str = None,
                       tag: str = '', region_label: str = 'full gene body'):
    """One vertical box per measure, adipose tracks against everything else.

    This is the direct test of the claim: if the effect really concentrates in adipose
    contexts, the orange distribution sits away from the grey one under every measure, not
    only the one that ends up in the figure.
    """
    table, allele_set, gene = measure_table(summaries, gene, allele_set)
    adipose = table[table['is_adipose']]
    other = table[~table['is_adipose']]
    rng = np.random.default_rng(0)

    fig, axes = plt.subplots(1, len(MEASURES), figsize=(4.1 * len(MEASURES), 5.4))
    rows = []
    for ax, (column, label) in zip(axes, MEASURES.items()):
        groups = [other[column].to_numpy(), adipose[column].to_numpy()]
        ax.boxplot(groups, showfliers=False, widths=0.55, medianprops={'color': 'black'})
        for position, (values, colour) in enumerate(
            zip(groups, ['dimgrey', 'tab:orange']), start=1
        ):
            ax.scatter(position + rng.normal(0, 0.06, len(values)), values, s=10,
                       color=colour, alpha=0.35 if position == 1 else 0.9,
                       zorder=3, edgecolor='none')
        if len(groups[1]) and len(groups[0]):
            statistic, p_value = stats.mannwhitneyu(groups[1], groups[0],
                                                    alternative='two-sided')
            rank_biserial = 2 * statistic / (len(groups[0]) * len(groups[1])) - 1
        else:
            p_value = rank_biserial = np.nan
        ax.set_xticks([1, 2])
        ax.set_xticklabels([f'Other\n(n={len(other)})', f'Adipose\n(n={len(adipose)})'],
                           fontsize=8)
        ax.axhline(0, ls='dashed', lw=0.8, color='grey')
        ax.set_title(f'{label}\np = {p_value:.2e}, r = {rank_biserial:.2f}', fontsize=9)
        rows.append({'measure': column, 'allele_set': allele_set, 'gene': gene,
                     'region': region_label,
                     'adipose_median': float(np.median(groups[1])) if len(groups[1]) else np.nan,
                     'other_median': float(np.median(groups[0])) if len(groups[0]) else np.nan,
                     'mannwhitney_p': p_value, 'rank_biserial': rank_biserial})

    fig.suptitle(f'Figure 8. AlphaGenome: effect of {allele_set} on predicted {gene} '
                 f'expression, by measure [{region_label}]', fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'Diag_alphagenome_{tag}{gene}_{allele_set}_measures.png',
                dpi=200, bbox_inches='tight')
    plt.show()

    summary = pd.DataFrame(rows)
    summary.to_csv(OUTPUT_DIR / f'measure_summary_{tag}{allele_set}_{gene}.csv', index=False)
    display(summary)
    return summary


def plot_measures_vs_baseline(summaries, gene: str = None, allele_set: str = None,
                              tag: str = '', region_label: str = 'full gene body'):
    """Each measure against baseline expression of the gene, adipose highlighted.

    The wedge at low baseline is a ceiling effect under the absolute measures — a track
    that barely expresses the gene has no room to move. Under the relative measure it
    should largely vanish, and adipose tracks leading anyway is the stronger claim.
    """
    table, allele_set, gene = measure_table(summaries, gene, allele_set)
    adipose = table[table['is_adipose']]

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    for ax, (column, label) in zip(axes.ravel(), MEASURES.items()):
        baseline_column = BASELINE_FOR[column]
        ax.scatter(table[baseline_column], table[column], s=14, facecolor='none',
                   edgecolor='dimgrey', alpha=0.45, linewidth=0.6, label='All RNA tracks')
        ax.scatter(adipose[baseline_column], adipose[column], s=28, color='tab:orange',
                   zorder=3, label='Adipose-related')
        ax.axhline(0, ls='dashed', lw=0.8, color='grey')
        correlation = stats.spearmanr(table[baseline_column], table[column]).statistic
        ax.set_xlabel(f'{baseline_column.replace("_", " ")}')
        ax.set_ylabel(label.replace('\n', ' '))
        ax.set_title(f'{label.splitlines()[0]} — Spearman r = {correlation:.2f}', fontsize=9)
    axes[0, 0].legend(fontsize=8, loc='best')

    fig.suptitle(f'Figure 9. AlphaGenome: each measure vs. baseline {gene} expression '
                 f'({allele_set}) [{region_label}]', fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'Diag_alphagenome_{tag}{gene}_{allele_set}_measures_vs_baseline.png',
                dpi=200, bbox_inches='tight')
    plt.show()


measure_summary = plot_measure_boxes(effect_summary)
plot_measures_vs_baseline(effect_summary)

### 10g. Differences from the original Borzoi analysis

Section 10 is modelled on the Borzoi notebook behind manuscript Figure 3b,c, but the two are not
a like-for-like reproduction, and several of the differences change what the panels show rather
than merely how they look. They are listed here so the figure legend can state them, with the
settings that close each gap where closing it is possible.

| | Borzoi run | Section 10 as configured | Setting |
|---|---|---|---|
| **Input window** | 524,288 bp starting 100 bp upstream of rs459193, so the variants sit at the extreme left edge | 1 Mb centred on the *ANKRD55*–*MAP3K1* span; variants off-centre but interior | `HAPLO_INTERVAL` (section 4) |
| **Region quantified** | Gene start bin to the end of the output window — the 5' 55,744 bp of *MAP3K1* only (bins 4402–6144) | Whole gene body; the crop is repeated in Appendix A | `APPENDIX_GENE_MAX_BP` |
| **Resolution** | 32 bp bins | 1 bp, binned to 32 bp before differencing | `HAPLO_BIN_SIZE` |
| **Tracks** | All RNA tracks, both strands, human **and** mouse (1,543 total; 955 after dropping antisense) | Human only, one track per ontology term; sense + unstranded (396) | `HAPLO_STRAND_MODE` |
| **Adipose definition** | `fat`, `adipose`, `adipocyte` in the description; 33 of 1,543 RNA tracks | Ontology descent plus broadened patterns (10d) | 10d, reported both ways in 10e |
| **Ranking / panels** | Mean absolute difference over the region; top 20 boxed, top 15 coloured | Same | `N_TOP_BOX`, `N_TOP_SCATTER` |
| **Model** | Borzoi replicate fold 0, one of four | AlphaGenome, single hosted model | — |

Four of these deserve more than a table row.

**Position in the window is not a cosmetic difference.** Borzoi's variants sat 100 bp from the
start of a 524 kb input, which is about as far from the centre as a variant can be; these models
attend unevenly across the receptive field and their effect estimates are known to depend on
where in the window a variant falls. Predictions here place the variants elsewhere in a different
window entirely, so a discrepancy in effect size between the two analyses is expected and is not
by itself evidence about the variants.

**The region quantified also differs, and the missing part is not arbitrary.** Borzoi's window
cut *MAP3K1* off partway through, keeping the 5' end — the promoter-proximal portion, which is
where sequence models concentrate their signal. Including the whole gene body dilutes the mean
with distal sequence the model largely ignores — which is an argument for reporting both, not for
preferring the crop. Section 10 uses the full gene body because the window allows it; Appendix A
repeats everything on the same 5' 55,744 bp and quantifies how much the choice matters.

**Strand filtering changes the published figure's appearance.** Borzoi kept every RNA track, so
each biosample contributes a plus- and a minus-strand track, and the antisense ones carry almost
no signal for the gene — that is the dense cloud near the origin in the published scatter panel,
and it is also why several tissue names appear twice in the box panel. Filtering to the gene's own
strand removes it. Set `HAPLO_STRAND_MODE = "both"` to reproduce the published look; keep `"gene"`
for a cleaner null.

**Adipose coverage is the same in both models; the track counts are not.** Both cover the
same six adipose biosample types — adipose tissue (UBERON:0001013), subcutaneous adipose
(UBERON:0002190), omental fat pad (UBERON:0010414), mesenteric fat pad (UBERON:0015143),
mesenchymal stem cell of adipose (CL:0002570) and subcutaneous preadipocyte (CL:0002583).
Borzoi has 33 adipose RNA tracks because it carries one track per experiment (15 ENCODE
donor samples x 2 strands, plus 3 unstranded GTEx); AlphaGenome has 14 because it supplies
one track per ontology term per strand, plus 2 unstranded GTEx. The difference is
granularity, not tissue coverage, and it is why the AlphaGenome enrichment test is
underpowered: with 8 adipose tracks in a top-20 cut, at least 3 hits are needed for p < 0.05.

**Unstranded tracks must be kept.** AlphaGenome annotates unstranded assays as `.`, and GTEx
RNA-seq is unstranded. Filtering to the gene's strand alone therefore discards all 125
unstranded tracks, including `gtex Adipose_Subcutaneous` and `gtex Adipose_Visceral_Omentum` —
the two most directly comparable to this study's GTEx evidence — and mismatches the Borzoi
section, which keeps them. `HAPLO_STRAND_MODE = "gene_and_unstranded"` is the correct
setting; earlier runs of this notebook used `"gene"` and scored 271 tracks with 6 adipose
instead of 396 with 8.

**The enrichment numbers are not comparable across models.** The published fold enrichment of 18.7
is against a universe of 1,543 tracks with 33 adipose, of which some were mouse. AlphaGenome's
panel is human-only, and under strand filtering the universe shrinks again — so 10e reports both
the keyword and the classified counts, and the fold enrichment should be read within a model, not
across the two.

**rs173964's alternate allele is A>G.** Supplementary Table 6 settles it — LDlink gives the
variant as `(A/G)` in Europeans, with the G allele in phase with rs256903 A — and Supplementary
Table 8 uses `A>G`. Both sections derive their alleles from `CANONICAL_VARIANTS`, and cell 10a
raises rather than warns if they ever diverge. The supplementary captions of Figures 8-10 state
`A>C` and need correcting; any Borzoi output produced against `A>C` has to be regenerated.


## Appendix A. The same analyses on the cropped gene region

Section 10 quantifies the **full gene body**, which is the analysis proper: the whole of
*MAP3K1* sits inside the 1 Mb window, so there is no reason to look at part of it. The earlier
Borzoi run did not have that option — its output window cut *MAP3K1* off after 55,744 bp — and
the manuscript's Figure 3b,c therefore describe the 5' portion of the gene only.

This appendix repeats the section 10 analyses on that cropped region so the two are comparable,
and so the effect of the crop can be seen rather than assumed. It costs no additional API calls:
cell 10b reduces **both** region sets out of the same predictions, since cropping is a slicing
choice made after the model has run.

Everything here is the functions from 10c, 10e and 10f called on the cropped tables —
`draw_allele_set_figures`, `compare_allele_sets`, `plot_baseline_ranking` — with a `tag` that
keeps the output filenames apart. Nothing is reimplemented, so any change to the analysis
propagates to both.

A2 then puts the two side by side. The values will differ; the question that matters is whether
the *ranking* does, because every conclusion drawn from these panels is a statement about which
tracks respond most. A high Spearman correlation and a large top-20 overlap mean the crop is
immaterial to the conclusions and the published figure stands; a low one means the 5' restriction
was doing real work, and the full-gene numbers in section 10 are the ones to trust — the crop is
an artefact of Borzoi's window, not a biological choice.

In [ ]:
# @title A1. Rerun the section 10 analyses on the cropped gene region { display-mode: "form" }
# @markdown Gene the appendix figures are drawn for.
APPENDIX_GENE = "MAP3K1"  # @param {type:"string"}
# @markdown Also redraw the per-allele-set Figure 6 panels for the cropped region. Off by
# @markdown default — the comparison table and the two summary figures are the point here.
APPENDIX_DRAW_FIGURE6 = False  # @param {type:"boolean"}

APPENDIX_REGION = 'cropped_5prime'

if APPENDIX_REGION not in effect_summary_by_region:
    raise ValueError('No cropped region set — set APPENDIX_GENE_MAX_BP in 10b and re-run it.')

cropped_summary = effect_summary_by_region[APPENDIX_REGION]
cropped_diff = effect_diff_by_region[APPENDIX_REGION]
region_label = (f"5' {APPENDIX_GENE_MAX_BP:,} bp"
                f" ({REGION_SETS[APPENDIX_REGION][APPENDIX_GENE]['region']})")
print(f'Appendix region: {region_label}\n')

# The same three functions defined in 10c, 10e and 10f, called on the cropped tables.
if APPENDIX_DRAW_FIGURE6:
    draw_allele_set_figures(cropped_summary, cropped_diff, genes=[APPENDIX_GENE],
                            tag='cropped_', region_label=region_label)

cropped_wide, cropped_enrichment = compare_allele_sets(
    cropped_summary, gene=APPENDIX_GENE, tag='cropped_', region_label=region_label
)
plot_baseline_ranking(cropped_summary, gene=APPENDIX_GENE, tag='cropped_',
                      region_label=region_label)
cropped_measure_summary = plot_measure_boxes(
    cropped_summary, gene=APPENDIX_GENE, tag='cropped_', region_label=region_label
)
plot_measures_vs_baseline(cropped_summary, gene=APPENDIX_GENE, tag='cropped_',
                          region_label=region_label)

In [ ]:
# The manuscript quotes two numbers from this cell, the top-20 overlap and the Pearson r,
# in the Limitations rather than showing the panel. Both are printed below.
# @title A2. Full gene body vs. cropped region, side by side { display-mode: "form" }
# @markdown Allele set the comparison is drawn for.
COMPARE_SET = "haplotype_4"  # @param {type:"string"}
# @markdown Tracks highlighted in the scatter (top by mean absolute difference, full gene).
N_LABEL = 10  # @param {type:"slider", min:0, max:25, step:5}

set_name = COMPARE_SET if COMPARE_SET in ALLELE_SETS else next(iter(ALLELE_SETS))
full = effect_summary[set_name][APPENDIX_GENE]
crop = cropped_summary[set_name][APPENDIX_GENE]

merged = full.merge(crop, on='track_name', suffixes=('_full', '_crop'))
merged.to_csv(OUTPUT_DIR / f'appendix_full_vs_cropped_{APPENDIX_GENE}.csv', index=False)

fig, (ax_diff, ax_base) = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, column, label in (
    (ax_diff, 'mean_difference', 'Mean difference (Perturbed - Baseline)'),
    (ax_base, 'mean_baseline', 'Mean baseline signal'),
):
    x, y = merged[f'{column}_full'].to_numpy(), merged[f'{column}_crop'].to_numpy()
    ax.scatter(x, y, s=14, facecolor='none', edgecolor='dimgrey', alpha=0.45, linewidth=0.6)
    adipose = merged[_adipose_flags(
        merged.rename(columns={'biosample_name_full': 'biosample_name'})
    ).to_numpy()]
    ax.scatter(adipose[f'{column}_full'], adipose[f'{column}_crop'], s=26,
               color='tab:orange', zorder=3, label='Adipose-related')
    limits = np.array([min(x.min(), y.min()), max(x.max(), y.max())])
    ax.plot(limits, limits, ls='dashed', lw=0.9, color='grey', label='y = x')
    correlation = float(np.corrcoef(x, y)[0, 1])
    ax.set_xlabel(f'{label} — full gene body')
    ax.set_ylabel(f'{label} — cropped')
    ax.set_title(f'{label}\nPearson r = {correlation:.3f}', fontsize=10)
    ax.legend(fontsize=7.5, loc='best')

fig.suptitle(
    f'{APPENDIX_GENE}, {set_name}: full gene body vs. {region_label}', fontsize=12, y=1.02
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'Diag_alphagenome_{APPENDIX_GENE}_full_vs_cropped.png',
            dpi=200, bbox_inches='tight')
plt.show()

# Does cropping change *which* tracks come out on top? Rank agreement is the thing that
# matters for every conclusion drawn from these panels, more than the values themselves.
top_full = list(full['track_name'].head(20))
top_crop = list(crop['track_name'].head(20))
overlap = len(set(top_full) & set(top_crop))
rank_correlation = stats.spearmanr(
    merged['mean_abs_difference_full'], merged['mean_abs_difference_crop']
).statistic
print(f'Top-20 tracks shared between the two regions: {overlap}/20.')
print(f'Spearman correlation of mean |difference| across all tracks: {rank_correlation:.3f}.')
pearson_full_vs_crop = float(np.corrcoef(
    merged['mean_abs_difference_full'], merged['mean_abs_difference_crop'])[0, 1])
print(f'Quoted in the Limitations: {overlap} of the 20 most affected tracks shared, '
      f'Pearson r = {pearson_full_vs_crop:.2f}.')

enrichment_comparison = pd.concat([
    enrichment.assign(region='full_gene'),
    cropped_enrichment.assign(region=APPENDIX_REGION),
], ignore_index=True)
enrichment_comparison.to_csv(
    OUTPUT_DIR / f'appendix_enrichment_comparison_{APPENDIX_GENE}.csv', index=False
)
display(
    enrichment_comparison[enrichment_comparison['allele_set'] == set_name]
    [['region', 'method', 'tracks_total', 'adipose_total', 'adipose_in_top_n',
      'fold_enrichment', 'pvalue']]
)

## 11. Outputs

Everything written by this notebook lands in `alphagenome_outputs/`. The last cell zips that
folder and downloads it (or copies it to Drive) — worth running before the Colab runtime
disconnects, since its disk is not persistent.

| File | Section | Contents |
|---|---|---|
| `Diag_locus_overview.png` | 4 | Variant positions across *ANKRD55* / *MAP3K1* |
| `SuppFig_locus_expression_landscape.png` | 5 | Predicted RNA-seq, adipose tracks, full locus |
| `Diag_ref_alt_profiles_<rsid>.png` | 6 | REF vs ALT RNA-seq and ATAC for the focal variant |
| `ST8_all_variant_scores.csv` | 7 | All tidy scores (ATAC + ChIP-TF, all tracks) |
| `ST8a_P300_binding_differences.csv` | 7.1 | EP300 tracks, all tissues |
| `ST8b_ATAC_adipose_differences.csv` | 7.2 | ATAC, adipose ontology subset |
| `ST8c_KLF_SP_binding_differences.csv` | 7.3 | KLF/SP tracks, all tissues |
| `Diag_ism_logos_<rsid>.png` | 8 | REF/ALT sequence logos per tissue |
| `allelic_background_scores.csv`, `background_summary.csv` | 9 | Background comparison, per gene and tissue |
| `Fig_<gene>_effect_vs_mutational_background.png` | 9 | One figure per gene in `COMPARISON_GENES` |
| `allele_set_track_effects.csv` | 10 | Per-track baseline and difference, per allele set and gene |
| `SuppFig_alphagenome_<allele_set>_<gene>_ranked_tracks.png` | 10 | Ranked track shifts and shift vs. baseline expression |
| `adipose_track_classification.csv` | 10d | Every track, with its adipose classification and evidence |
| `allele_set_comparison_<gene>.csv` | 10e | Per-track shift of every allele set, side by side |
| `SuppFig_alphagenome_<gene>_effect_additivity.png` | 10e | Additivity check and per-set shift distributions |
| `adipose_enrichment_<gene>.csv` | 10e | Hypergeometric enrichment of adipose tracks |
| `baseline_expression_ranked_<gene>.csv` | 10f | Every track ranked by predicted baseline expression |
| `figS_baseline_expression_<gene>.png` | 10f | Supplementary: ranked baseline expression, adipose highlighted |
| `fig8_measures_<set>_<gene>.png`, `measure_summary_*.csv` | 10h | Each measure, adipose vs the rest |
| `fig9_measures_vs_baseline_<set>_<gene>.png` | 10h | Each measure vs baseline expression |
| `*cropped_*` counterparts of the above | Appendix A | Same analyses on the 5' cropped gene region |
| `appendix_full_vs_cropped_<gene>.csv`, `figA_full_vs_cropped_<gene>.png` | Appendix A | Full gene body vs cropped, per track |
| `appendix_enrichment_comparison_<gene>.csv` | Appendix A | Adipose enrichment under both regions |

**Interpreting the scores.** `raw_score` is the aggregated ALT − REF difference and its scale
depends on the scorer, so it is only comparable within an output type. `quantile_score` places
that value in the distribution of common variants and is comparable everywhere; values near ±1
are extreme. All predictions are model outputs, not measurements — they generate hypotheses for
experimental follow-up rather than confirming a mechanism.

In [ ]:
# @title Output manifest: every file, its scope and where it belongs
# Written last so the map of outputs to manuscript items stays in step with the run that
# produced them, rather than being maintained by hand.
manifest = write_manifest()
if manifest is not None:
    import pandas as pd

    with pd.option_context('display.width', 200, 'display.max_colwidth', 60,
                           'display.max_rows', 80):
        display(manifest.sort_values(['scope', 'kind', 'file']))
    print('\nby scope:')
    print(manifest.groupby(['scope', 'kind']).size().to_string())


In [ ]:
# @title Save the results { display-mode: "form" }
# @markdown Zips `alphagenome_outputs/` beside itself. In Colab it also downloads the archive,
# @markdown since the runtime disk is wiped on disconnect; tick the box to copy it to Google
# @markdown Drive as well. Locally the outputs are already on disk under `results/`.
SAVE_TO_DRIVE = False  # @param {type:"boolean"}
import shutil
import sys

# Section C loads Borzoi again, so anything Section B still holds is freed first.
release('locus_output', 'rna', 'focal_output', 'bg_scores', 'raw_scores')
report_memory('Section B outputs saved')

archive = shutil.make_archive(str(Path(OUTPUT_DIR).parent / 'alphagenome_outputs'), 'zip',
                              OUTPUT_DIR)
print('Files in', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path.name:45s} {path.stat().st_size / 1024:8.1f} KB')

if SAVE_TO_DRIVE and 'google.colab' in sys.modules:
    from google.colab import drive

    drive.mount('/content/drive')
    shutil.copy(archive, '/content/drive/MyDrive/')
    print(f'\nCopied to Google Drive: {Path(archive).name}')
elif SAVE_TO_DRIVE:
    print('\nSAVE_TO_DRIVE only applies in Colab; the archive is on the local disk already.')

try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print(f'\nArchive: {archive}')

# Section B's settings, appended to the same manifest Section A started.
if 'note' in globals():
    note(alphagenome_scope_main=MAIN_SCOPE_NAME, alphagenome_scope_supp=SUPP_SCOPE_NAME,
         alphagenome_variants=';'.join(sorted(CANONICAL_VARIANTS)))
    write_provenance()


---

# Section C — Cross-model comparison and window diagnostics

Section C answers three questions that neither section can answer alone.

**C1. Does it matter where in the input window the variants sit?** Borzoi's published run put
them ~100 bp from the left edge of a 524 kb input. The sweep re-runs the prediction with the
window slid along the genome and measures how much the effect on *MAP3K1*, and the ranking of
tracks by that effect, changes. It also states a constraint that is easy to miss: with a
196,608 bp output crop and ~305 kb between rs459193 and the *MAP3K1* TSS, the variants and the
promoter cannot both be in the predicted span — the edge placement is forced.

**C2. Does it matter how much of the gene is quantified?** The same predictions, reduced over
the 5′ 55,744 bp versus every *MAP3K1* bin available in the window. This is the Borzoi
counterpart of AlphaGenome's Appendix A, so the two models' cropping sensitivity can be compared.

**C3. The same sweep for AlphaGenome,** over adipose ontology terms only to keep the cost down.
AlphaGenome's 1 Mb context holds the whole gene *and* the variants, so here window position can
be varied without the trade-off Borzoi is stuck with — which is the cleanest available test of
whether edge placement explains the disagreement.

**C4. Is adipose enrichment masked by baseline expression?** Absolute effect measures scale with
how much a track expresses the gene, so the most-affected tracks tend to be the
highest-expressing ones. C4 regresses each measure on baseline expression and re-runs the
hypergeometric enrichment on the residuals, in both models. If adipose enrichment appears in the
residuals for AlphaGenome, the masking explanation is supported; if it does not, it should be
dropped.

**C5** copies the chosen panels into `figures_for_manuscript/` under the names used in the draft.

Everything here needs Sections A and B to have run in this session.

In [ ]:
# @title C1. Borzoi: does window position change the answer? { display-mode: "form" }
# @markdown Distances, in bp, from the input window's left edge to the leftmost variant. The
# @markdown published run used 100. Each value costs two forward passes.
SWEEP_VARIANT_OFFSETS = "100,5000,15000,30000,45000"  # @param {type:"string"}
# @markdown Width of the 5' crop compared against the full available gene span in C2.
SWEEP_CROP_BP = 55744  # @param {type:"integer"}
# @markdown Top tracks used for the adipose enrichment reported per offset.
SWEEP_TOP_N = 20  # @param {type:"slider", min:5, max:50, step:5}

import gc

import numpy as np
import pandas as pd
import torch
from scipy import stats

if 'BORZOI' not in globals():
    raise RuntimeError('Section A has not been run (or its snapshot cell was skipped).')

SWEEP_DIR = BORZOI['output_dir'].parent / 'cross_model'
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

B = BORZOI
GENE_START, GENE_END = B['gene_coordinates'][1], B['gene_coordinates'][2]
offsets = [int(value) for value in SWEEP_VARIANT_OFFSETS.split(',') if value.strip()]

# --- The geometric constraint, stated before anything is predicted ----------
# The variants sit ~305 kb from the MAP3K1 TSS, and only the central 196,608 bp of the input
# is predicted. Both cannot be inside the output at once, so the edge placement of the
# variants in the published run was forced by the model's crop, not chosen.
separation = GENE_START - B['leftmost']
print(f'rs459193 to {B["gene"]} TSS: {separation:,} bp; '
      f'predicted span: {B["output_length"]:,} bp.')
if separation > B['output_length']:
    print('=> The variants and the TSS CANNOT both lie in the predicted output. Any window '
          'that shows the gene must place the variants outside the output, near its left '
          'edge — this is a property of the model, not a choice in the analysis.\n')
else:
    print('=> Both can lie in the output; the edge placement was a choice.\n')

# And the trade-off is one-for-one: sliding the window right by n bp moves the variants n bp
# further from its start and simultaneously moves the output n bp away from the gene, so every
# base gained in variant margin is a base of MAP3K1 lost. The published 100 bp offset is
# therefore the placement that maximises gene coverage, not an arbitrary choice.
max_coverage = min(GENE_END, B['leftmost'] - 100 + B['crop'] + B['output_length']) - GENE_START
print(f'Sliding the window inwards costs gene coverage one-for-one; at the published 100 bp '
      f'offset,\n{max_coverage:,} bp of {B["gene"]} is inside the output, which is the '
      'maximum available.\n')


def sweep_sequences(window_start: int):
    """Reference and four-variant sequences for one window start.

    Deliberately re-implemented here from the snapshot rather than calling Section A's
    `build_sequences`: that function closes over the module-level `VARIANTS`, which
    Section B rebinds to AlphaGenome variant objects.
    """
    window_end = window_start + B['context_length']
    reference = B['fetch_reference']('chr5', window_start, window_end)
    bases = list(reference)
    for rsid, (_, position, ref, alt) in B['variants'].items():
        index = position - 1 - window_start
        if reference[index] != ref:
            raise ValueError(f'{rsid}: reference {ref} expected, found {reference[index]}')
        bases[index] = alt
    return reference, ''.join(bases)


def track_table(baseline, perturbed, columns):
    """Per-track measures over a set of output bins."""
    base = baseline[:, columns]
    pert = perturbed[:, columns]
    difference = pert - base
    table = pd.DataFrame({
        'track_index': rna_subset,
        'description': B['targets'].loc[rna_subset, 'description'].to_numpy(),
        'mean_baseline': base.mean(axis=1),
        'mean_difference': difference.mean(axis=1),
        'mean_abs_difference': np.abs(difference).mean(axis=1),
        'log2_fold_change': np.log2((pert.sum(axis=1) + 1e-3) / (base.sum(axis=1) + 1e-3)),
    })
    table['is_adipose'] = table['description'].str.contains(
        B['adipose_pattern'], case=False
    )
    return table


def adipose_enrichment(table, column='mean_abs_difference', top_n=SWEEP_TOP_N):
    order = table[column].abs().sort_values(ascending=False).index
    flags = table.loc[order, 'is_adipose'].to_numpy()
    n_total, n_adipose = len(table), int(table['is_adipose'].sum())
    top = min(top_n, n_total)
    hits = int(flags[:top].sum())
    expected = top * n_adipose / n_total
    return {
        'adipose_in_top_n': hits,
        'fold_enrichment': round(hits / expected, 2) if expected else np.nan,
        'pvalue': stats.hypergeom.sf(hits - 1, n_total, n_adipose, top),
    }


# Sense and unstranded RNA tracks, as in the corrected Figure 3b,c.
rna_subset = (B['rna_targets'].index[B['rna_targets']['strand'] != '-'].to_numpy()
              if B['drop_antisense'] else B['rna_targets'].index.to_numpy())

from borzoi_pytorch import Borzoi

# The device Section A settled on, which may be MPS rather than CUDA or the CPU.
device = globals().get('DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
sweep_model = Borzoi.from_pretrained(f'johahi/borzoi-replicate-{B["fold"]}').to(device).eval()


@torch.no_grad()
def sweep_predict(sequence: str) -> np.ndarray:
    tensor = B['one_hot'](sequence).to(device)
    output = sweep_model(tensor).squeeze(0).float().cpu().numpy()
    del tensor
    if device == 'cuda':
        torch.cuda.empty_cache()
    return output[rna_subset]


sweep_full, sweep_crop, rows = {}, {}, []
for offset in offsets:
    window_start = B['leftmost'] - offset
    output_start = window_start + B['crop']
    output_end = output_start + B['output_length']
    overlap_start, overlap_end = max(GENE_START, output_start), min(GENE_END, output_end)
    coverage = max(0, overlap_end - overlap_start)
    print(f'offset {offset:>6,} bp: output {output_start:,}-{output_end:,}, '
          f'{B["gene"]} coverage {coverage:,} bp', flush=True)
    if coverage < 2 * B['bin_size']:
        print('  gene not in the output window — skipped.')
        rows.append({'offset_bp': offset, 'window_start': window_start,
                     'coverage_bp': coverage, 'variants_in_output': False})
        continue

    reference, perturbed_sequence = sweep_sequences(window_start)
    baseline_prediction = sweep_predict(reference)
    perturbed_prediction = sweep_predict(perturbed_sequence)

    first_bin = int((overlap_start - output_start) // B['bin_size'])
    last_bin = int(np.ceil((overlap_end - output_start) / B['bin_size']))
    full_columns = np.arange(first_bin, last_bin)
    crop_columns = full_columns[: max(1, SWEEP_CROP_BP // B['bin_size'])]

    sweep_full[offset] = track_table(baseline_prediction, perturbed_prediction, full_columns)
    sweep_crop[offset] = track_table(baseline_prediction, perturbed_prediction, crop_columns)
    del baseline_prediction, perturbed_prediction
    gc.collect()

    table = sweep_full[offset]
    adipose = table[table['is_adipose']]
    other = table[~table['is_adipose']]
    statistic, p_value = stats.mannwhitneyu(other['log2_fold_change'],
                                            adipose['log2_fold_change'],
                                            alternative='two-sided')
    rows.append({
        'offset_bp': offset,
        'window_start': window_start,
        'coverage_bp': coverage,
        'coverage_fraction': coverage / (GENE_END - GENE_START),
        'variants_in_output': bool(output_start <= B['leftmost'] <= output_end),
        'variant_to_output_start_bp': B['leftmost'] - output_start,
        'median_log2fc_all': float(table['log2_fold_change'].median()),
        'median_log2fc_adipose': float(adipose['log2_fold_change'].median()),
        'median_log2fc_other': float(other['log2_fold_change'].median()),
        'adipose_vs_other_p': p_value,
        **adipose_enrichment(table),
    })

del sweep_model
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

sweep_summary = pd.DataFrame(rows)
# Rank stability: does moving the window reshuffle which tracks respond most?
reference_offset = offsets[0]
for offset in offsets:
    if offset in sweep_full and reference_offset in sweep_full:
        merged = sweep_full[reference_offset].merge(
            sweep_full[offset], on='track_index', suffixes=('_ref', '_this')
        )
        correlation = stats.spearmanr(merged['mean_abs_difference_ref'],
                                      merged['mean_abs_difference_this']).statistic
        sign_agreement = float(
            (np.sign(merged['log2_fold_change_ref'])
             == np.sign(merged['log2_fold_change_this'])).mean()
        )
    else:
        correlation = sign_agreement = np.nan
    sweep_summary.loc[sweep_summary['offset_bp'] == offset, 'spearman_vs_reference'] = correlation
    sweep_summary.loc[sweep_summary['offset_bp'] == offset, 'sign_agreement'] = sign_agreement

sweep_summary.to_csv(SWEEP_DIR / 'Tab_borzoi_window_sweep.csv', index=False)
display(sweep_summary)

In [ ]:
# @title Figure C1: effect of window position (Borzoi)
import matplotlib.pyplot as plt

valid = sweep_summary.dropna(subset=['median_log2fc_all'])

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# a) How much of the gene the output window actually contains at each offset. Sliding the
# window to move the variants inwards slides the output away from the gene, so these two
# desiderata trade off directly.
ax = axes[0, 0]
ax.bar(valid['offset_bp'].astype(str), valid['coverage_bp'] / 1000,
       color='tab:grey', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Variant offset from input start (bp)')
ax.set_ylabel(f'{BORZOI["gene"]} bp inside the output (kb)')
ax.set_title('a) Gene coverage vs. window position', fontsize=10)

# b) Signed effect, adipose against the rest, at each offset.
ax = axes[0, 1]
ax.plot(valid['offset_bp'], valid['median_log2fc_other'], 'o-', color='dimgrey',
        label='Other tracks')
ax.plot(valid['offset_bp'], valid['median_log2fc_adipose'], 'o-', color='tab:orange',
        label='Adipose-related')
ax.axhline(0, ls='dashed', lw=0.8, color='grey')
ax.set_xlabel('Variant offset from input start (bp)')
ax.set_ylabel('Median log2 fold change')
ax.set_title('b) Direction and size of the effect', fontsize=10)
ax.legend(fontsize=8)

# c) Rank stability against the published placement. Low values mean the identity of the
# most-affected tracks depends on where the variants sit, which would undercut any
# tissue-level claim drawn from a single window.
ax = axes[1, 0]
ax.plot(valid['offset_bp'], valid['spearman_vs_reference'], 'o-', color='tab:blue',
        label='Spearman of mean |difference|')
ax.plot(valid['offset_bp'], valid['sign_agreement'], 'o-', color='tab:green',
        label='Fraction of tracks with the same sign')
ax.set_ylim(-1.05, 1.05)
ax.axhline(0, ls='dashed', lw=0.8, color='grey')
ax.set_xlabel('Variant offset from input start (bp)')
ax.set_ylabel(f'Agreement with offset {sweep_summary["offset_bp"].iloc[0]:,} bp')
ax.set_title('c) Does the track ranking survive moving the window?', fontsize=10)
ax.legend(fontsize=8)

# d) Adipose enrichment at each offset.
ax = axes[1, 1]
ax.bar(valid['offset_bp'].astype(str), valid['fold_enrichment'],
       color='tab:orange', edgecolor='black', linewidth=0.5)
for position, (fold, p_value) in enumerate(zip(valid['fold_enrichment'], valid['pvalue'])):
    if np.isfinite(fold):
        ax.text(position, fold, f'p={p_value:.1e}', ha='center', va='bottom', fontsize=7)
ax.axhline(1, ls='dashed', lw=0.8, color='grey')
ax.set_xlabel('Variant offset from input start (bp)')
ax.set_ylabel(f'Adipose fold enrichment (top {SWEEP_TOP_N})')
ax.set_title('d) Adipose enrichment vs. window position', fontsize=10)

fig.suptitle('Figure C1. Borzoi: sensitivity to where the variants sit in the input window',
             fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(SWEEP_DIR / 'SuppFig_borzoi_window_position.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
# The manuscript quotes this sweep as numbers in the Limitations rather than showing it
# as a panel: the ranking of affected tracks changes with the window in both models.
# @title C3. AlphaGenome: the same sweep, without the crop trade-off { display-mode: "form" }
# @markdown Shifts applied to the locus interval, in bp. Positive slides the window right, which
# @markdown moves the variants towards its left edge. Each shift costs two model calls.
AG_SWEEP_SHIFTS = "-200000,-100000,0,100000,200000"  # @param {type:"string"}
# @markdown Ontology terms used. The adipose set from section 2 keeps this affordable; the
# @markdown sweep is about window geometry, not about ranking every track in the model.
AG_SWEEP_ALL_TRACKS = False  # @param {type:"boolean"}

ag_shifts = [int(value) for value in AG_SWEEP_SHIFTS.split(',') if value.strip()]
ag_terms = (sorted(set(dna_model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS).rna_seq['ontology_curie'].dropna()))
    if AG_SWEEP_ALL_TRACKS else list(ONTOLOGY_TERMS))
ag_gene = APPENDIX_GENE if 'APPENDIX_GENE' in globals() else 'MAP3K1'
ag_region = REGION_SETS['full_gene'][ag_gene]['region']
ag_variants = ALLELE_SETS[HAPLO_SET_NAME]

print(f'{len(ag_shifts)} shifts x 2 calls, {len(ag_terms)} ontology terms, '
      f'quantifying {ag_gene} over {ag_region.width:,} bp.\n')

ag_rows, ag_tables = [], {}
for shift in ag_shifts:
    interval = genome.Interval(HAPLO_INTERVAL.chromosome,
                               HAPLO_INTERVAL.start + shift,
                               HAPLO_INTERVAL.end + shift)
    inside = all(interval.contains(v.reference_interval) for v in ag_variants)
    covers_gene = interval.contains(ag_region)
    if not (inside and covers_gene):
        print(f'shift {shift:+,}: variants inside={inside}, gene covered={covers_gene} '
              '— skipped.')
        continue

    print(f'shift {shift:+,}: {interval} ...', flush=True)
    reference = reference_sequence(interval)
    perturbed_sequence = edited_sequence(ag_variants, reference, interval)

    baseline_output = dna_model.predict_sequence(
        sequence=reference, requested_outputs=[dna_client.OutputType.RNA_SEQ],
        ontology_terms=ag_terms, interval=interval,
    )
    base_raw, metadata = _gene_slice(baseline_output.rna_seq, ag_gene,
                                     REGION_SETS['full_gene'])
    base_values = _binned(base_raw.astype(np.float32))
    del baseline_output

    perturbed_output = dna_model.predict_sequence(
        sequence=perturbed_sequence, requested_outputs=[dna_client.OutputType.RNA_SEQ],
        ontology_terms=ag_terms, interval=interval,
    )
    pert_values = _binned(
        _gene_slice(perturbed_output.rna_seq, ag_gene,
                    REGION_SETS['full_gene'])[0].astype(np.float32)
    )
    del perturbed_output

    difference = pert_values - base_values
    table = pd.DataFrame({
        'track_name': list(metadata['name']),
        'biosample_name': list(metadata.get('biosample_name', metadata['name'])),
        'mean_baseline': base_values.mean(axis=0),
        'mean_difference': difference.mean(axis=0),
        'mean_abs_difference': np.abs(difference).mean(axis=0),
        'log2_fold_change': np.log2(
            (pert_values.sum(axis=0) + 1e-3) / (base_values.sum(axis=0) + 1e-3)
        ),
    })
    table['is_adipose'] = _adipose_flags(table).to_numpy()
    ag_tables[shift] = table

    # Where the variants sit inside this window, which is the quantity being varied.
    variant_centre = int(np.mean([v.position for v in ag_variants]))
    ag_rows.append({
        'shift_bp': shift,
        'interval_start': interval.start,
        'variant_offset_in_window_bp': variant_centre - interval.start,
        'variant_fraction_of_window': (variant_centre - interval.start) / interval.width,
        'median_log2fc_all': float(table['log2_fold_change'].median()),
        'median_log2fc_adipose': float(table.loc[table['is_adipose'],
                                                 'log2_fold_change'].median()),
        'median_mean_difference': float(table['mean_difference'].median()),
        'fraction_positive': float((table['mean_difference'] > 0).mean()),
    })
    del base_values, pert_values, difference

ag_sweep = pd.DataFrame(ag_rows)
ag_sweep.to_csv(SWEEP_DIR / 'Diag_alphagenome_window_position.csv', index=False)
display(ag_sweep)

if len(ag_sweep) > 1:
    reference_shift = ag_sweep['shift_bp'].iloc[0]
    for shift in ag_sweep['shift_bp']:
        merged = ag_tables[reference_shift].merge(
            ag_tables[shift], on='track_name', suffixes=('_ref', '_this')
        )
        ag_sweep.loc[ag_sweep['shift_bp'] == shift, 'spearman_vs_first'] = stats.spearmanr(
            merged['mean_abs_difference_ref'], merged['mean_abs_difference_this']
        ).statistic

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    axes[0].plot(ag_sweep['variant_fraction_of_window'], ag_sweep['median_log2fc_all'],
                 'o-', color='dimgrey', label='All tracks')
    axes[0].plot(ag_sweep['variant_fraction_of_window'], ag_sweep['median_log2fc_adipose'],
                 'o-', color='tab:orange', label='Adipose-related')
    axes[0].axhline(0, ls='dashed', lw=0.8, color='grey')
    axes[0].set_ylabel('Median log2 fold change')
    axes[0].set_title('a) Effect vs. variant position in window', fontsize=10)
    axes[0].legend(fontsize=8)

    axes[1].plot(ag_sweep['variant_fraction_of_window'], ag_sweep['fraction_positive'],
                 'o-', color='tab:green')
    axes[1].axhline(0.5, ls='dashed', lw=0.8, color='grey')
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('Fraction of tracks with increased expression')
    axes[1].set_title('b) Direction stability', fontsize=10)

    axes[2].plot(ag_sweep['variant_fraction_of_window'], ag_sweep['spearman_vs_first'],
                 'o-', color='tab:blue')
    axes[2].set_ylim(-1.05, 1.05)
    axes[2].set_ylabel('Spearman vs. first shift')
    axes[2].set_title('c) Rank stability', fontsize=10)

    for ax in axes:
        ax.set_xlabel('Variant position as a fraction of the input window')
    fig.suptitle(f'Figure C3. AlphaGenome: {ag_gene} effect vs. window geometry',
                 fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(SWEEP_DIR / 'Diag_alphagenome_window_position.png', dpi=250,
                bbox_inches='tight')
    plt.show()

print('AlphaGenome holds the whole gene and the variants in one window, so unlike Borzoi its '
      'window position can be varied\nwithout also changing how much of the gene is measured. '
      'If the direction flips here too, neither model\'s sign is\nabout the alleles; if it '
      'holds, window placement is not what separates the two models.')

In [ ]:
# @title C4. Is adipose enrichment masked by baseline expression? { display-mode: "form" }
# @markdown Measure compared across the two models. Exonic log2 fold change is the closest to
# @markdown dimensionless and therefore the fairest cross-model comparison.
CROSS_MEASURE = "exon_log2_fold_change"  # @param ["exon_log2_fold_change", "exon_mean_difference", "mean_difference"]
CROSS_TOP_N = 20  # @param {type:"slider", min:5, max:50, step:5}

cross_gene = MEASURE_GENE if 'MEASURE_GENE' in globals() else 'MAP3K1'
cross_set = MEASURE_SET if MEASURE_SET in effect_summary else next(iter(effect_summary))

alphagenome_table = effect_summary[cross_set][cross_gene].copy()
alphagenome_table['is_adipose'] = _adipose_flags(alphagenome_table).to_numpy()
borzoi_table = BORZOI['metrics'].copy()

baseline_column = ('exon_mean_baseline' if CROSS_MEASURE.startswith('exon')
                   else 'mean_baseline')

MODELS = {'Borzoi': borzoi_table, 'AlphaGenome': alphagenome_table}


def residualise(table, measure, baseline):
    """Effect with its dependence on baseline expression removed.

    Fitted on ranks with a quadratic, not on raw values with a line: the relationship is
    monotone but not linear, and ranks stop a few very highly expressed tracks from setting
    the fit. The residual answers "large for a track expressing this much", which is the
    question the raw ranking silently fails to ask.
    """
    x = stats.rankdata(table[baseline]) / len(table)
    y = table[measure].to_numpy()
    coefficients = np.polyfit(x, y, 2)
    return y - np.polyval(coefficients, x), x


def enrichment_of(values, flags, top_n=CROSS_TOP_N):
    order = np.argsort(-np.abs(values))
    ordered = np.asarray(flags)[order]
    n_total, n_adipose = len(values), int(np.sum(flags))
    top = min(top_n, n_total)
    hits = int(ordered[:top].sum())
    expected = top * n_adipose / n_total
    return hits, (round(hits / expected, 2) if expected else np.nan), \
        stats.hypergeom.sf(hits - 1, n_total, n_adipose, top)


rows = []
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for row_index, (model, table) in enumerate(MODELS.items()):
    residuals, ranks = residualise(table, CROSS_MEASURE, baseline_column)
    flags = table['is_adipose'].to_numpy()
    adipose_mask = flags.astype(bool)

    raw_hits, raw_fold, raw_p = enrichment_of(table[CROSS_MEASURE].to_numpy(), flags)
    res_hits, res_fold, res_p = enrichment_of(residuals, flags)
    correlation = stats.spearmanr(table[baseline_column], table[CROSS_MEASURE]).statistic
    _, adipose_p = stats.mannwhitneyu(residuals[adipose_mask], residuals[~adipose_mask],
                                      alternative='two-sided')
    rows.append({
        'model': model, 'measure': CROSS_MEASURE, 'tracks': len(table),
        'adipose_tracks': int(flags.sum()),
        'median_effect_adipose': float(np.median(table.loc[adipose_mask, CROSS_MEASURE])),
        'median_effect_other': float(np.median(table.loc[~adipose_mask, CROSS_MEASURE])),
        'spearman_effect_vs_baseline': correlation,
        'raw_adipose_in_top_n': raw_hits, 'raw_fold': raw_fold, 'raw_p': raw_p,
        'residual_adipose_in_top_n': res_hits, 'residual_fold': res_fold,
        'residual_p': res_p, 'residual_adipose_vs_other_p': adipose_p,
    })

    ax = axes[row_index, 0]
    ax.scatter(ranks, table[CROSS_MEASURE], s=12, facecolor='none', edgecolor='dimgrey',
               alpha=0.4, linewidth=0.5, label='All tracks')
    ax.scatter(ranks[adipose_mask], table.loc[adipose_mask, CROSS_MEASURE], s=26,
               color='tab:orange', zorder=3, label='Adipose-related')
    fit_x = np.linspace(0, 1, 100)
    ax.plot(fit_x, np.polyval(np.polyfit(ranks, table[CROSS_MEASURE].to_numpy(), 2), fit_x),
            color='tab:blue', lw=1.2, label='Baseline trend')
    ax.axhline(0, ls='dashed', lw=0.8, color='grey')
    ax.set_ylabel(CROSS_MEASURE.replace('_', ' '))
    ax.set_title(f'{model}: raw effect (Spearman vs baseline r = {correlation:.2f}; '
                 f'adipose in top {CROSS_TOP_N}: {raw_hits}, p = {raw_p:.1e})', fontsize=9)
    if row_index == 0:
        ax.legend(fontsize=8)

    ax = axes[row_index, 1]
    ax.scatter(ranks, residuals, s=12, facecolor='none', edgecolor='dimgrey', alpha=0.4,
               linewidth=0.5)
    ax.scatter(ranks[adipose_mask], residuals[adipose_mask], s=26, color='tab:orange',
               zorder=3)
    ax.axhline(0, ls='dashed', lw=0.8, color='grey')
    ax.set_ylabel('Residual effect')
    ax.set_title(f'{model}: baseline removed (adipose in top {CROSS_TOP_N}: {res_hits}, '
                 f'p = {res_p:.1e})', fontsize=9)

for ax in axes.ravel():
    ax.set_xlabel(f'Baseline {cross_gene} expression (rank, 0-1)')

fig.suptitle('Figure C4. Adipose enrichment before and after removing the baseline-expression '
             'ceiling', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(SWEEP_DIR / 'SuppFig_adipose_effect_vs_baseline.png', dpi=250,
            bbox_inches='tight')
plt.show()

cross_summary = pd.DataFrame(rows)
cross_summary.to_csv(SWEEP_DIR / 'Tab_cross_model_summary.csv', index=False)
display(cross_summary)

print('How to read this:')
print('  - Raw enrichment high in Borzoi but not AlphaGenome, AND residual enrichment present '
      'in both\n    => the masking explanation holds: AlphaGenome ranks by expression level, '
      'not by susceptibility.')
print('  - Residual enrichment absent in AlphaGenome => masking does NOT explain it, and the '
      'adipose claim\n    rests on Borzoi plus the eQTL and caQTL data rather than on both '
      'models. Report that.')
print('  - Note the medians above: the two models can disagree on sign while agreeing on '
      'which contexts respond.')

In [ ]:
# @title C5. Collect the manuscript panels under their final names { display-mode: "form" }
# Copies every panel the manuscript uses into one folder, under the name the figure map
# and the manuscript legends refer to. Anything not listed here is a diagnostic and stays
# in its section folder. Ten supplementary figures, down from the earlier sixteen: the
# panels that carried a single number now have that number quoted in the text instead.
#
# The three source folders are resolved explicitly. OUTPUT_DIR is rebound from
# 'borzoi_figures' to 'alphagenome_outputs' in Section B, so referring to it here would
# have sent every Borzoi lookup to the AlphaGenome folder and reported nine files as
# missing. Section A's own path is recovered from the snapshot instead.
import shutil
from pathlib import Path

BORZOI_DIR = Path(BORZOI['output_dir'])            # Section A wrote here
ALPHA_DIR = Path(globals().get('RESULTS_ROOT', Path.cwd())) / 'alphagenome_outputs'
CROSS_DIR = Path(globals().get('SWEEP_DIR', BORZOI_DIR.parent / 'cross_model'))

for _label, _folder in (('Borzoi', BORZOI_DIR), ('AlphaGenome', ALPHA_DIR),
                        ('cross-model', CROSS_DIR)):
    print(f'{_label:12s} -> {_folder.resolve()}'
          f'{"" if _folder.exists() else "   [MISSING, run that section first]"}')
print()

MANUSCRIPT_DIR = Path(globals().get('RESULTS_ROOT', Path.cwd())) / 'manuscript_panels'
MANUSCRIPT_DIR.mkdir(exist_ok=True)

PANELS = {
    # --- main figure panels ------------------------------------------------
    'Fig3b_variant_prioritisation_5variants':
        ALPHA_DIR / MAIN_SCOPE_NAME / f'Fig_variant_prioritisation__{MAIN_SCOPE_NAME}.png',
    'Fig4ab_borzoi_MAP3K1_tracks_and_baseline':
        BORZOI_DIR / 'Fig_borzoi_MAP3K1_tracks_and_baseline.png',
    'Fig4c_borzoi_adipose_vs_other_measures':
        BORZOI_DIR / 'Fig_borzoi_MAP3K1_adipose_vs_other_measures.png',
    'Fig4d_MAP3K1_effect_vs_mutational_background':
        ALPHA_DIR / 'Fig_MAP3K1_effect_vs_mutational_background.png',
    # --- supplementary figures S8 to S17 -----------------------------------
    'SuppFig08_borzoi_channel_differences':
        BORZOI_DIR / 'SuppFig_borzoi_channel_differences.png',
    'SuppFig09_borzoi_profiles_variant_centred':
        BORZOI_DIR / 'SuppFig_borzoi_profiles_variant_centred.png',
    'SuppFig10_borzoi_profiles_upstream_window':
        BORZOI_DIR / 'SuppFig_borzoi_profiles_upstream_window.png',
    'SuppFig11_locus_expression_landscape':
        ALPHA_DIR / 'SuppFig_locus_expression_landscape.png',
    'SuppFig12_SETD9_effect_vs_mutational_background':
        ALPHA_DIR / 'Fig_SETD9_effect_vs_mutational_background.png',
    'SuppFig13_borzoi_window_position':
        CROSS_DIR / 'SuppFig_borzoi_window_position.png',
    'SuppFig14_adipose_effect_vs_baseline':
        CROSS_DIR / 'SuppFig_adipose_effect_vs_baseline.png',
    'SuppFig15_alphagenome_haplotype_ranked_tracks':
        ALPHA_DIR / 'SuppFig_alphagenome_haplotype_4_MAP3K1_ranked_tracks.png',
    'SuppFig16_variant_effect_additivity':
        ALPHA_DIR / 'SuppFig_alphagenome_MAP3K1_effect_additivity.png',
    'SuppFig17_ism_vs_mutational_background':
        ALPHA_DIR / 'SuppFig_ism_vs_mutational_background.png',
    # The same prioritisation panel over every credible-set variant. Same scores as
    # Figure 3b, more rows.
    'SuppFig_variant_prioritisation_25variants':
        ALPHA_DIR / SUPP_SCOPE_NAME / f'SuppFig_variant_prioritisation__{SUPP_SCOPE_NAME}.png',
}

# Supplementary tables, same contract.
TABLES = {
    # ST8 at both scopes: the five the main text reports, and every credible-set variant.
    'ST08a_EP300_binding_differences_5variants':
        ALPHA_DIR / MAIN_SCOPE_NAME / f'Tab_ST8a_EP300_binding_differences__{MAIN_SCOPE_NAME}.csv',
    'ST08b_ATAC_adipose_differences_5variants':
        ALPHA_DIR / MAIN_SCOPE_NAME / f'Tab_ST8b_ATAC_adipose_differences__{MAIN_SCOPE_NAME}.csv',
    'ST08c_KLF_SP_binding_differences_5variants':
        ALPHA_DIR / MAIN_SCOPE_NAME / f'Tab_ST8c_KLF_SP_binding_differences__{MAIN_SCOPE_NAME}.csv',
    'ST08a_EP300_binding_differences_25variants':
        ALPHA_DIR / SUPP_SCOPE_NAME / f'Tab_ST8a_EP300_binding_differences__{SUPP_SCOPE_NAME}.csv',
    'ST08b_ATAC_adipose_differences_25variants':
        ALPHA_DIR / SUPP_SCOPE_NAME / f'Tab_ST8b_ATAC_adipose_differences__{SUPP_SCOPE_NAME}.csv',
    'ST08c_KLF_SP_binding_differences_25variants':
        ALPHA_DIR / SUPP_SCOPE_NAME / f'Tab_ST8c_KLF_SP_binding_differences__{SUPP_SCOPE_NAME}.csv',
    'ST10_borzoi_effect_measures': BORZOI_DIR / 'Tab_borzoi_MAP3K1_effect_measures.csv',
    'ST10_borzoi_measure_summary': BORZOI_DIR / 'Tab_borzoi_MAP3K1_measure_summary.csv',
    'ST11_borzoi_adipose_enrichment': BORZOI_DIR / 'Tab_borzoi_adipose_enrichment.csv',
    'ST12_borzoi_measure_enrichment': BORZOI_DIR / 'Tab_borzoi_MAP3K1_measure_enrichment.csv',
    'ST13_borzoi_window_sweep': CROSS_DIR / 'Tab_borzoi_window_sweep.csv',
    'ST14_alphagenome_allele_set_effects':
        ALPHA_DIR / 'Tab_alphagenome_MAP3K1_allele_set_effects.csv',
    'ST15_alphagenome_adipose_enrichment':
        ALPHA_DIR / 'Tab_alphagenome_MAP3K1_adipose_enrichment.csv',
    'ST16_mutational_background_summary':
        ALPHA_DIR / 'Tab_mutational_background_summary.csv',
    'ST16_allelic_background_scores':
        ALPHA_DIR / 'Tab_allelic_background_scores.csv',
    # Supporting data for the supplement: what selected each variant, and the block-level
    # effects that neither model reports variant by variant.
    'STnew_variant_evidence':
        Path(globals().get('RESULTS_ROOT', Path.cwd())) / 'variant_evidence.csv',
    'STnew_borzoi_allele_set_effects':
        BORZOI_DIR / 'Tab_borzoi_MAP3K1_allele_set_effects.csv',
    'STnew_alphagenome_all_variant_scores':
        ALPHA_DIR / SUPP_SCOPE_NAME / f'Tab_ST8_all_variant_scores__{SUPP_SCOPE_NAME}.csv',
}

# ST8d is the existing ChromBPNet variant effect sheet. It is not produced here and is
# carried over from the previous supplementary workbook.
collected, missing = [], []
for group, mapping in (('figure', PANELS), ('table', TABLES)):
    for name, source_path in mapping.items():
        source_path = Path(source_path)
        if source_path.exists():
            shutil.copy(source_path, MANUSCRIPT_DIR / f'{name}{source_path.suffix}')
            collected.append(name)
        else:
            missing.append((group, name, str(source_path)))

print(f'Collected {len(collected)} items into {MANUSCRIPT_DIR.resolve()}')
for name in collected:
    print(f'  {name}')
if missing:
    print(f'\nNot found ({len(missing)}). Run the section that produces each one:')
    for group, name, source_path in missing:
        print(f'  [{group}] {name}  <-  {source_path}')
else:
    print('\nEvery panel and table the manuscript refers to is present.')

# One CSV mapping each collected file to the manuscript item it feeds, written beside the
# files themselves so the folder documents itself.
import pandas as pd

SCOPE_OF = {MAIN_SCOPE_NAME: 'main text (5 variants)',
            SUPP_SCOPE_NAME: 'supplementary (25 variants)'}
rows = []
for group, mapping in (('figure', PANELS), ('table', TABLES)):
    for name, source_path in mapping.items():
        source_path = Path(source_path)
        scope = next((label for key, label in SCOPE_OF.items() if key in str(source_path)),
                     'both / not scope-specific')
        rows.append({'manuscript_item': name, 'kind': group, 'scope': scope,
                     'collected_file': f'{name}{source_path.suffix}',
                     'produced_by': source_path.parent.name,
                     'present': source_path.exists()})
figure_map = pd.DataFrame(rows).sort_values(['kind', 'manuscript_item'])
figure_map.to_csv(MANUSCRIPT_DIR / 'figure_and_table_map.csv', index=False)
print(f'\nMap written to {(MANUSCRIPT_DIR / "figure_and_table_map.csv").resolve()}')
with pd.option_context('display.width', 200, 'display.max_colwidth', 46,
                       'display.max_rows', 80):
    display(figure_map)


---

# Methods

Both sections' methods, as written for the manuscript. Numbers that depend on settings are the
defaults in this notebook.

## Methods — Section C (window and cropping diagnostics)

To assess how far the predictions depend on the geometry of the prediction window rather than on
the substituted alleles, the Borzoi analysis was repeated with the input window slid along the
genome so that the leftmost variant lay at a range of distances from the window's start, holding
the alleles and the model fixed. For each placement, the portion of *MAP3K1* falling inside the
predicted span was recorded together with per-track effect measures, and placements were compared
by the Spearman correlation of per-track mean absolute differences, the proportion of tracks
whose effect retained its sign, and the hypergeometric enrichment of adipose-related tracks among
the most affected. Because the separation between the variants and the *MAP3K1* transcription
start site exceeds the model's predicted span, no window contains both, and the variants
necessarily lie outside the predicted region in any window that covers the gene; this constraint
is a property of the architecture rather than a choice of analysis. The same predictions were
additionally reduced over the 5' portion of the gene and over its full extent within the window,
to separate the effect of cropping from that of window placement. The equivalent sweep was
performed with AlphaGenome, whose context accommodates both the variants and the entire gene, so
that window placement could be varied independently of the region quantified.

Dependence of effect size on baseline expression was removed by fitting a quadratic to each
measure against the rank-transformed baseline expression of the gene and taking residuals; the
enrichment analysis was then repeated on residuals. This distinguishes tracks that respond
strongly for their expression level from tracks that respond strongly because they express the
gene highly, the latter being the expected behaviour of any absolute measure.


## Methods — Section A (Borzoi)

Predictions were generated with Borzoi (`borzoi-pytorch`, replicate fold 0 unless configured
otherwise) on GRCh38/hg38. The model takes 524,288 bp of input sequence and predicts the central
196,608 bp at 32 bp resolution across 7,611 human tracks, of which 1,543 are RNA-seq.

### Variant selection and orientation

Variants were taken from the signed-LD fine mapping rather than from a fixed list. A variant
was selected if it was a member of a SharePro effect group, a member of a final CARMA
credible set in either fine-mapped trait, or a member of a credible set in at least 5 of 20
CARMA seeds in either trait; the four fine-mapped variants and rs3843467 were carried
regardless. Selected variants were assigned to one of two blocks by linkage disequilibrium
with its anchor, rs256903 for the fine-mapped 3-prime signal and rs3843467 for the
independent one, requiring |r| >= 0.5; two selected variants correlated with neither anchor
were excluded and are reported in the notebook output.

Coordinates and alleles were read from the LD matrix's own variant index, which records the
GRCh37 position and the A1 and A2 alleles that every sign in the matrix and every harmonised
beta refer to. GRCh38 positions were obtained by applying a single offset, calculated from
five variants whose GRCh38 coordinates had been checked by hand against dbSNP and required to
be identical for all five (+704,173 bp). The reference base implied by that mapping was then
checked against the GRCh38 sequence retrieved for each prediction window, and a mismatch
raises rather than warns.

The risk-raising allele was determined from the sign of the harmonised effect estimate, which
refers to A1, propagated to weakly associated variants through the sign of their correlation
with their block anchor; the two routes were required to agree wherever a variant's own |Z|
exceeded 3. The resulting orientation reproduces all seven risk alleles recorded in the
LDlink Correlated_Alleles column of Supplementary Table 6.

Alternate alleles for the four linked variants (rs459193 A>G, rs256903 C>A, rs173964 A>G, rs256904 A>T) were taken from a single shared definition used by both the Borzoi and the AlphaGenome analyses, with the alternate alleles chosen as those in phase on the risk haplotype (LDlink, European 1000 Genomes samples; Supplementary Table 6). A configuration-time assertion fails if the two analyses disagree on any allele.

The reference sequence for each input window was retrieved from the UCSC REST API, and the
alternate alleles of rs459193, rs256903, rs173964 and rs256904 — four variants in near-perfect
linkage disequilibrium — were substituted into it simultaneously to give the haplotype sequence;
rs3843467 was excluded on the grounds of distance and lower linkage disequilibrium. Each
reference allele was verified against the retrieved base before substitution and each alternate
allele against the Ensembl variation record for the corresponding rsID. Baseline and perturbed
predictions were made from the same retrieved sequence and differ only at the four substituted
bases.

Two input framings were used, since no single window comfortably contains both the variants and
*MAP3K1*: one centred on rs459193, and one beginning 100 bp upstream of it, which brings the 5'
55,744 bp of *MAP3K1* into the predicted span. Track-level quantification of *MAP3K1* used the
second framing, over output bins 4,402 to 6,144, covering the promoter and the first exons and
introns. Each track was summarised by its mean predicted coverage and its mean
perturbed-minus-baseline difference over that region, and tracks were ranked by mean absolute
difference.

RNA-seq tracks are stranded and supplied in pairs. Since *MAP3K1* is transcribed from the plus
strand, minus-strand tracks carry negligible signal for it, and their inclusion contributes a
concentration of near-zero values that reflects transcript orientation rather than a null
distribution; they were therefore excluded, retaining sense and unstranded tracks. Enrichment of
adipose-related tracks among the twenty most affected was assessed with a hypergeometric test,
adipose tracks being those whose description matches fat, adipose or adipocyte, and is reported
for both track universes, as excluding antisense tracks alters the background rate and hence the
fold enrichment.

Variant effects on *MAP3K1* were quantified under four measures computed from the same
predictions: the signed mean difference per bin; the signed difference summed over the region and
multiplied by bin width, giving an integrated change in coverage x bp; each of these restricted to
exonic bins; and the log2 ratio of total perturbed to total baseline signal over exonic bins, with
a pseudocount of 1e-3, giving a dimensionless relative change. Exonic bins were those overlapping
any exon of *MAP3K1* annotated in Ensembl, unioned across transcripts; where the annotation could
not be retrieved, exonic bins were defined empirically as those whose baseline signal, pooled
across the fifty most-expressing tracks, exceeded the 90th percentile of the region. Adipose-related
tracks were compared against the remainder under each measure by a two-sided Mann-Whitney U test
with the rank-biserial correlation as effect size, and the hypergeometric enrichment of adipose
tracks among the twenty most affected was computed under each. Each measure was additionally
plotted against baseline predicted expression of the gene, and its Spearman correlation with that
baseline reported, since a measure correlating strongly with baseline expression is in part
reporting how much a track expresses the gene rather than how far the alleles moved it. The same
four measures, panel layout and tests were applied to the AlphaGenome predictions of the same
locus; the pattern of effects is comparable between the models, the absolute values are not. The measures are strongly correlated
with one another and the track panel represents tissues unequally, so these tests are reported as
descriptions of the predictions rather than as independent evidence.

Predicted differences are reported in the model's scaled coverage units and are not calibrated
against a background variant distribution. They are interpreted through the consistency of their
direction across independent tracks and their relationship to baseline expression rather than
through absolute magnitude. The models were trained on reference genomes and their ability to
recover the direction of variant effects is known to be limited, so a directional disagreement
with measured eQTL data is expected and is not evidence about the locus.

## Methods — Section B (AlphaGenome)

*Written as a methods section. Numbers that depend on the configuration cells — window sizes,
ontology terms, aggregation choices — are stated as configured above; update them if you change
the settings.*

## Model and sequence context

Regulatory predictions were generated with AlphaGenome (`alphagenome` Python client, model
accessed through the hosted API) on the GRCh38/hg38 assembly. Unless stated otherwise all
predictions used the model's maximum input context of 1,048,576 bp (1 Mb). For variant-centred
analyses the input interval was centred on the variant position; for the locus-level reference
prediction the interval was centred on the midpoint of the *ANKRD55*–*MAP3K1* span, which is
approximately 800 kb wide and therefore fits within a single input window together with all
five variants. Transcript models were taken from GENCODE v46, restricted to protein-coding
genes with transcript support level 1; the longest transcript per gene was used for locus-wide
figures and the full supported transcript set for zoomed views.

## Variants and tissue context

Five variants at the chromosome 5 insulin-resistance locus were analysed (rs459193, rs256903,
rs173964, rs256904, rs3843467), specified as hg38 coordinates with reference and alternate
alleles. Predictions were restricted to eight adipose and mesenchymal ontology terms spanning
bulk adipose tissue, depot-specific fat, mature adipocytes, preadipocytes and adipose-derived
mesenchymal stem cells. Ontology terms with no track of a given output type in the model were
reported and excluded rather than imputed; the tracks contributing to each term are enumerated
in the notebook output.

## Reference predictions across the locus

Baseline RNA-seq coverage was predicted for the unmodified reference sequence across the full
locus interval for all adipose ontology terms. RNA-seq tracks are stranded, so each term
contributes at least one track per strand and genes are visible only in the track matching
their transcriptional orientation.

## Single-variant reference versus alternate predictions

For one focal variant, paired predictions were generated with the reference and alternate
alleles substituted into an otherwise identical input sequence, and RNA-seq and ATAC tracks
were overlaid over a 32 kb window centred on the variant. Because both predictions derive from
the same interval and differ only at the substituted base, the difference between them isolates
the allelic effect.

## Variant effect scoring

Variant effects were quantified with the recommended scorer configurations. Chromatin
accessibility and transcription factor binding were scored with `CenterMaskScorer`
(width 501 bp, `DIFF_LOG2_SUM` aggregation) applied to the ATAC and ChIP-TF outputs
respectively; this aggregates the log2 alternate-minus-reference difference over a 501 bp
window centred on the variant. Transcript abundance was scored with the gene-centric
`GeneMaskLFCScorer` on the RNA-seq output, which returns one value per gene in the input
window. Center-mask scorers return every track in the model rather than only the requested
ontology terms, which permits analyses in biosamples outside the adipose panel.

Two quantities are reported per track. The raw score is the aggregated
alternate-minus-reference difference and is comparable only within an output type. The quantile
score expresses that value as its rank within the distribution of scores obtained for a
background set of common variants, and is therefore comparable across scorers, tracks and
output types, with values approaching ±1 indicating effects at the extreme of that background
distribution.

### Derived tables

Three subsets were extracted. (i) *EP300 binding*: all ChIP-TF tracks for EP300, across all
biosamples, since EP300 is profiled in cell lines rather than in adipose tissue; predicted loss
of EP300 binding is treated as evidence of enhancer disruption. (ii) *Chromatin accessibility
in adipose*: ATAC tracks restricted to the eight adipose and mesenchymal ontology terms.
(iii) *KLF and SP family binding*: ChIP-TF tracks for factors matching `SP` followed by a single
digit or `KLF` followed by digits, across all biosamples. Tables are sorted by quantile score
where available and by raw score otherwise.

## In silico mutagenesis

Every possible single-nucleotide substitution within a window centred on each variant was scored
against the same 1 Mb input context, using the gene-level RNA-seq scorer for *MAP3K1*. The full
context length was retained because shorter windows exclude *MAP3K1* for the upstream variants,
leaving the gene-centric scorer without a target. Mutagenesis was performed twice per variant:
once on the reference sequence, and once on a background carrying the alternate allele, so that
motif structure could be compared between allelic contexts. A motif present on the reference
background and absent on the alternate background indicates disruption by the variant.

Scoring returns all tracks of the requested output type for all genes in the window, so the
scoring pass is independent of tissue; tissue specificity enters only at the extraction step,
where rows are restricted to the focal gene and columns to a single ontology term. Where an
ontology term is represented by several tracks, those tracks are collapsed to one value per
mutated base. **This choice is consequential.** The mean is unbiased; the maximum is
systematically greater than zero even for tracks containing only noise. Because sequence-logo
matrices are mean-centred at each position and the background base is drawn at minus the mean of
its three alternatives, a constant bias in the underlying scores translates into a uniform
vertical shift of every letter, which can render an entire panel one-sided for reasons unrelated
to sequence content. Results are therefore reported under both aggregations, and features are
interpreted only where they persist across both. Reference and alternate logos for a given
tissue are drawn on a shared symmetric y-axis, since independently autoscaled panels can
manufacture apparent asymmetry between allelic backgrounds.

## Comparison against background variants

Predicted gene-level effects were placed against three null distributions. First, an allelic
background consisting of the remaining possible alternate alleles at the same position, which
holds position fixed and isolates the identity of the base change. Second, a local sequence
background comprising all single-nucleotide substitutions within the mutagenesis window,
obtained without additional scoring from the reference-background in silico mutagenesis results.
Third, the model's own quantile score, which references a genome-wide set of common variants.
The first two are local and therefore more conservative than the third. For each variant an
empirical percentile was computed as the proportion of background scores falling below the
observed score, using the local background where available.

Comparisons were made for two genes, *MAP3K1* and *SETD9*, in bulk adipose tissue and in
adipose-derived mesenchymal stem cells. Both genes lie within the variant-centred window and
were obtained from the same scoring pass at no additional cost. *SETD9* functions as a
within-window comparator: concordant effects on both genes are more consistent with a broad
regional influence on predicted expression, whereas an effect confined to *MAP3K1* suggests
greater specificity. Gene-level scores are not comparable across track sets, so panels for
different tissues are drawn on independent axes.

## Sequence-level allele substitution: single variants and the haplotype

To evaluate alternate alleles individually and jointly under one procedure, the reference
sequence of the locus window was retrieved and alternate alleles were substituted directly into
it, and predictions were generated from the edited sequences rather than through the
single-variant substitution interface. Five allele sets were defined: rs459193, rs256903,
rs173964 and rs256904 each written into the sequence alone, and the four written in
simultaneously as the risk haplotype; rs3843467 was excluded from the haplotype on the grounds
of distance and lower linkage disequilibrium. Reference allele identity was verified against the
retrieved sequence at every substituted position before prediction, and each edited sequence was
confirmed to differ from the reference at exactly the expected number of positions.

Predictions for the unmodified sequence and for each edited sequence were generated under
identical settings, so that any pair differs only at the substituted bases. RNA-seq output was
requested for all tracks in the model, batched over ontology terms to bound memory; within each
batch the unmodified-sequence prediction was computed once and reused as the common baseline for
every allele set, so that all sets are compared against identical baseline values.

For each gene, predicted coverage was restricted to the gene body as annotated in GENCODE v46,
clipped to the prediction window where necessary, and to tracks either matching the transcriptional
orientation of the gene or produced by unstranded assays, since stranded tracks of the
opposite orientation carry negligible signal for the gene whereas unstranded tracks, which
include all GTEx samples, carry signal irrespective of orientation. Each track was summarised by its mean baseline coverage and its mean
perturbed-minus-baseline difference across positions in the gene body, and tracks were ranked by
mean absolute difference. Differences are reported in raw predicted-coverage units and are not
calibrated against a background variant distribution; they are interpreted through the
consistency of their direction across independent tracks and their relationship to baseline
expression, the latter being bounded from below by the baseline itself.

Because all allele sets share a baseline and a track set, the four single-variant differences
were summed per track and regressed through the origin on the difference measured with all four
alleles present, giving a direct test of additivity in the model's predictions. Enrichment of
adipose-related contexts among the most affected tracks was assessed by a hypergeometric test on
the top twenty tracks ranked by mean absolute difference, following the procedure applied to the
Borzoi predictions. Adipose tracks were identified by ontology descent rather than by keyword:
the descendant closures of adipose tissue, fat pad, fat cell, preadipocyte and mesenchymal stem
cell of adipose tissue were retrieved from the EMBL-EBI Ontology Lookup Service and intersected
with the ontology terms present in the model metadata, supplemented by pattern matching on
biosample names and ontology labels for terms naming a depot or a cell line rather than the
tissue (omentum, epiploic tissue, SGBS, 3T3-L1, among others). Patterns that are adipose only in
context, such as subcutaneous, visceral and mesenteric, were reported separately and excluded by
default, since they also match non-adipose biosamples; the full per-track classification and the
evidence admitting each track are reported. Fold enrichments are not comparable between the two models, whose
RNA-seq track panels differ.

## Limitations

All values reported here are model predictions rather than measurements, and establish neither
that a variant is causal nor that it acts on the gene whose predicted expression changes.
Effect sizes are small in absolute terms and their interpretation depends on the aggregation and
scaling choices documented above. Genes and tissues were selected a priori from the locus and
disease context rather than by scanning; where several variants, tissues and genes are examined
together, the resulting rankings should be treated as hypothesis-generating and warranting
experimental follow-up.